# Google Play Phase 2 Cadence Test — Run B Follow-up Collection

This notebook completes the second controlled cadence test by running a follow-up collection from the database saved after the Run B first collection.

The follow-up keeps the controlled setup unchanged:

- the same 10 Google Play apps
- the same target of 1,200 newest reviews per app
- the same English and United States settings
- the same Phase 2 SQLite database
- the same database continuation logic
- the same duplicate-prevention logic
- the same run-level and app-level summary structure

The analysis will distinguish between:

- review IDs that are new to the database
- reviews actually posted between the two Run B collections
- older reviews that appear in the returned review window later
- reviews with missing or unusable timestamps

For timestamp validation, each app is compared with its own collection timestamp from the Run B first collection. The exact interval between collections will be recorded rather than assuming a fixed 12-hour gap.

The final results will be used to evaluate which apps consistently benefit from more frequent collection and which apps mainly return duplicates.

In [1]:
!pip -q install google-play-scraper

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.2/50.2 kB 2.4 MB/s eta 0:00:00


## 1. Import packages and set the fixed follow-up configuration

This follow-up collection continues from the checkpoint created after the Run B first collection.

The source, app list, review target, sorting method, language, country, database schema, and duplicate identity remain unchanged.

In [2]:
import os
import re
import json
import time
import shutil
import sqlite3
import zipfile
import hashlib
import warnings
from pathlib import Path
from datetime import datetime, timezone

import pandas as pd

from google.colab import files
from google_play_scraper import reviews, Sort

warnings.filterwarnings("ignore")

BASE_DIR = Path("/content")
DATABASE_DIR = BASE_DIR / "database"
OUTPUT_DIR = BASE_DIR / "outputs"

DATABASE_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

DB_PATH = DATABASE_DIR / "google_play_reviews.sqlite"

SOURCE = "google_play"
LANGUAGE = "en"
COUNTRY = "us"
SORT_METHOD = Sort.NEWEST

TARGET_REVIEWS_PER_APP = 1200
REQUEST_SLEEP_SECONDS = 2

RUN_LABEL = "phase2_cadence_runB_followup_collection"
FREQUENCY_LABEL = "runB_followup_collection"

RUN_ID = (
    f"{RUN_LABEL}_"
    f"{datetime.now(timezone.utc).strftime('%Y%m%d_%H%M%S')}"
)

print("Run ID:", RUN_ID)
print("Run label:", RUN_LABEL)
print("Frequency label:", FREQUENCY_LABEL)
print("Source:", SOURCE)
print("Language / country:", LANGUAGE, "/", COUNTRY)
print("Target reviews per app:", TARGET_REVIEWS_PER_APP)
print("Database path:", DB_PATH)
print("Output folder:", OUTPUT_DIR)

Run ID: phase2_cadence_runB_followup_collection_20260714_163017
Run label: phase2_cadence_runB_followup_collection
Frequency label: runB_followup_collection
Source: google_play
Language / country: en / us
Target reviews per app: 1200
Database path: /content/database/google_play_reviews.sqlite
Output folder: /content/outputs


## 2. Load and validate the Run B first-collection checkpoint

The follow-up must continue from the exact database saved after the Run B first collection.

Before the database is used, this section verifies:

- the checkpoint manifest and file hashes
- the database SHA-256 value
- the six required Phase 2 tables
- the exact same 10-app configuration
- 30,848 matching raw and cleaned review rows
- five completed prior Phase 2 runs
- the completed Run B first-collection record
- 10 app-level collection timestamps from the first collection
- the absence of duplicate review identities and orphan records

The checkpoint database will only be copied into the working directory after all required checks pass.

In [3]:
UPLOAD_DIR = Path("/content/uploaded_runB_checkpoint")
EXTRACT_DIR = UPLOAD_DIR / "extracted"
DATABASE_EXTRACT_DIR = UPLOAD_DIR / "database_extracted"

if UPLOAD_DIR.exists():
    shutil.rmtree(UPLOAD_DIR)

UPLOAD_DIR.mkdir(parents=True, exist_ok=True)
EXTRACT_DIR.mkdir(parents=True, exist_ok=True)
DATABASE_EXTRACT_DIR.mkdir(parents=True, exist_ok=True)


def calculate_sha256(file_path):
    sha256 = hashlib.sha256()

    with open(file_path, "rb") as file:
        for block in iter(
            lambda: file.read(1024 * 1024),
            b"",
        ):
            sha256.update(block)

    return sha256.hexdigest()


print("Upload this checkpoint file:")
print(
    "phase2_cadence_runB_first_collection_"
    "checkpoint_20260714_020137_utc.zip"
)

uploaded = files.upload()

if len(uploaded) != 1:
    raise ValueError(
        "Please upload exactly one Run B first-collection "
        "checkpoint zip file."
    )

uploaded_name = next(iter(uploaded))
saved_upload_path = UPLOAD_DIR / uploaded_name

saved_upload_path.write_bytes(
    uploaded[uploaded_name]
)

if not zipfile.is_zipfile(saved_upload_path):
    raise ValueError(
        "The uploaded checkpoint is not a valid zip file."
    )

with zipfile.ZipFile(saved_upload_path, "r") as zf:
    zf.extractall(EXTRACT_DIR)

print("Checkpoint zip extracted successfully.")

required_checkpoint_files = [
    "checkpoint_manifest.csv",
    (
        "phase2_cadence_runB_first_collection_"
        "metadata.json"
    ),
    (
        "phase2_cadence_runB_first_collection_"
        "completed_run_record.csv"
    ),
    (
        "phase2_cadence_runB_first_collection_"
        "app_summary.csv"
    ),
    (
        "phase2_cadence_runB_first_collection_"
        "timestamp_summary.csv"
    ),
    (
        "phase2_cadence_runB_first_collection_"
        "timestamp_audit.csv"
    ),
    (
        "google_play_reviews_after_"
        "runB_first_collection.sqlite.zip"
    ),
]

missing_checkpoint_files = [
    file_name
    for file_name in required_checkpoint_files
    if not (EXTRACT_DIR / file_name).exists()
]

if missing_checkpoint_files:
    raise FileNotFoundError(
        "The checkpoint is missing required files: "
        f"{missing_checkpoint_files}"
    )

# Validate the checkpoint manifest.
manifest_path = (
    EXTRACT_DIR / "checkpoint_manifest.csv"
)

manifest_df = pd.read_csv(manifest_path)

required_manifest_columns = {
    "file_name",
    "size_bytes",
    "sha256",
}

if not required_manifest_columns.issubset(
    manifest_df.columns
):
    raise ValueError(
        "The checkpoint manifest does not contain "
        "the required columns."
    )

manifest_failures = []

for _, row in manifest_df.iterrows():
    file_name = str(row["file_name"])
    file_path = EXTRACT_DIR / file_name

    if not file_path.exists():
        manifest_failures.append(
            f"{file_name}: file is missing"
        )
        continue

    expected_size = int(row["size_bytes"])
    actual_size = int(file_path.stat().st_size)

    expected_hash = str(row["sha256"]).strip()
    actual_hash = calculate_sha256(file_path)

    if actual_size != expected_size:
        manifest_failures.append(
            f"{file_name}: size mismatch"
        )

    if actual_hash != expected_hash:
        manifest_failures.append(
            f"{file_name}: SHA-256 mismatch"
        )

if manifest_failures:
    raise ValueError(
        "Checkpoint manifest validation failed: "
        f"{manifest_failures}"
    )

print("Checkpoint manifest validation passed.")

# Read and validate checkpoint metadata.
metadata_path = (
    EXTRACT_DIR
    / (
        "phase2_cadence_runB_first_collection_"
        "metadata.json"
    )
)

with open(
    metadata_path,
    "r",
    encoding="utf-8",
) as file:
    checkpoint_metadata = json.load(file)

expected_metadata = {
    "run_label": (
        "phase2_cadence_runB_first_collection"
    ),
    "frequency_label": "runB_first_collection",
    "source": "google_play",
    "language": "en",
    "country": "us",
    "target_reviews_per_app": 1200,
    "app_count": 10,
    "records_fetched_total": 12000,
    "new_records_inserted_total": 8638,
    "duplicates_skipped_total": 3362,
    "review_rows_after": 30848,
}

metadata_failures = []

for field_name, expected_value in (
    expected_metadata.items()
):
    actual_value = checkpoint_metadata.get(
        field_name
    )

    if actual_value != expected_value:
        metadata_failures.append(
            {
                "field": field_name,
                "expected": expected_value,
                "actual": actual_value,
            }
        )

if metadata_failures:
    raise ValueError(
        "Checkpoint metadata validation failed: "
        f"{metadata_failures}"
    )

PREVIOUS_RUN_ID = str(
    checkpoint_metadata["run_id"]
)

PREVIOUS_RUN_LABEL = str(
    checkpoint_metadata["run_label"]
)

PREVIOUS_RUN_STARTED_AT = str(
    checkpoint_metadata["run_started_at"]
)

PREVIOUS_RUN_FINISHED_AT = str(
    checkpoint_metadata["run_finished_at"]
)

# Extract and validate the checkpoint database.
database_zip_path = (
    EXTRACT_DIR
    / (
        "google_play_reviews_after_"
        "runB_first_collection.sqlite.zip"
    )
)

if not zipfile.is_zipfile(database_zip_path):
    raise ValueError(
        "The checkpoint database archive "
        "is not a valid zip file."
    )

with zipfile.ZipFile(
    database_zip_path,
    "r",
) as zf:
    zf.extractall(DATABASE_EXTRACT_DIR)

database_candidates = [
    path
    for path in DATABASE_EXTRACT_DIR.rglob("*.sqlite")
    if path.is_file()
]

if len(database_candidates) != 1:
    raise FileNotFoundError(
        "Expected exactly one SQLite database "
        f"but found {len(database_candidates)}."
    )

source_db_path = database_candidates[0]

database_sha256 = calculate_sha256(
    source_db_path
)

expected_database_sha256 = str(
    checkpoint_metadata["database_sha256"]
)

if database_sha256 != expected_database_sha256:
    raise ValueError(
        "The extracted database SHA-256 does not "
        "match the checkpoint metadata."
    )

required_tables = [
    "phase2_reviews_raw",
    "phase2_reviews_cleaned",
    "phase2_apps",
    "phase2_ingestion_runs",
    "phase2_app_run_summary",
    "phase2_quality_flags",
]

expected_apps = [
    ("YouTube", "com.google.android.youtube"),
    ("TikTok", "com.zhiliaoapp.musically"),
    ("Spotify", "com.spotify.music"),
    ("Instagram", "com.instagram.android"),
    ("Uber", "com.ubercab"),
    ("DoorDash", "com.dd.doordash"),
    ("Duolingo", "com.duolingo"),
    (
        "Google Maps",
        "com.google.android.apps.maps",
    ),
    ("Netflix", "com.netflix.mediaclient"),
    ("Reddit", "com.reddit.frontpage"),
]

with sqlite3.connect(source_db_path) as test_conn:
    test_conn.execute(
        "PRAGMA foreign_keys = ON"
    )

    existing_tables = pd.read_sql_query(
        """
        SELECT name
        FROM sqlite_master
        WHERE type = 'table'
        """,
        test_conn,
    )["name"].tolist()

    missing_tables = [
        table_name
        for table_name in required_tables
        if table_name not in existing_tables
    ]

    if missing_tables:
        raise ValueError(
            "The checkpoint database is missing "
            f"required tables: {missing_tables}"
        )

    app_config_df = pd.read_sql_query(
        """
        SELECT
            app_name,
            app_id,
            source,
            language,
            country,
            title_from_store,
            score_from_store,
            ratings_from_store,
            installs_from_store
        FROM phase2_apps
        ORDER BY rowid
        """,
        test_conn,
    )

    actual_apps = list(
        app_config_df[
            ["app_name", "app_id"]
        ].itertuples(
            index=False,
            name=None,
        )
    )

    if actual_apps != expected_apps:
        raise ValueError(
            "The checkpoint app list or app order "
            "does not match the fixed configuration."
        )

    if not app_config_df["source"].eq(
        SOURCE
    ).all():
        raise ValueError(
            "The checkpoint source does not "
            "match Google Play."
        )

    if not app_config_df["language"].eq(
        LANGUAGE
    ).all():
        raise ValueError(
            "The checkpoint language does not "
            "match English."
        )

    if not app_config_df["country"].eq(
        COUNTRY
    ).all():
        raise ValueError(
            "The checkpoint country does not "
            "match the United States."
        )

    raw_rows_before = int(
        pd.read_sql_query(
            """
            SELECT COUNT(*) AS n
            FROM phase2_reviews_raw
            """,
            test_conn,
        )["n"].iloc[0]
    )

    cleaned_rows_before = int(
        pd.read_sql_query(
            """
            SELECT COUNT(*) AS n
            FROM phase2_reviews_cleaned
            """,
            test_conn,
        )["n"].iloc[0]
    )

    completed_run_count = int(
        pd.read_sql_query(
            """
            SELECT COUNT(*) AS n
            FROM phase2_ingestion_runs
            WHERE status = 'completed'
            """,
            test_conn,
        )["n"].iloc[0]
    )

    app_summary_row_count = int(
        pd.read_sql_query(
            """
            SELECT COUNT(*) AS n
            FROM phase2_app_run_summary
            """,
            test_conn,
        )["n"].iloc[0]
    )

    latest_run_df = pd.read_sql_query(
        """
        SELECT
            run_id,
            run_label,
            frequency_label,
            target_reviews_per_app,
            app_count,
            run_started_at,
            run_finished_at,
            status,
            records_fetched_total,
            new_records_inserted_total,
            duplicates_skipped_total,
            review_rows_after
        FROM phase2_ingestion_runs
        ORDER BY run_started_at DESC
        LIMIT 1
        """,
        test_conn,
    )

    duplicate_identity_groups = int(
        pd.read_sql_query(
            """
            SELECT COUNT(*) AS n
            FROM (
                SELECT
                    source,
                    app_id,
                    review_id,
                    COUNT(*) AS row_count
                FROM phase2_reviews_raw
                GROUP BY
                    source,
                    app_id,
                    review_id
                HAVING COUNT(*) > 1
            )
            """,
            test_conn,
        )["n"].iloc[0]
    )

    raw_without_cleaned = int(
        pd.read_sql_query(
            """
            SELECT COUNT(*) AS n
            FROM phase2_reviews_raw AS r
            LEFT JOIN phase2_reviews_cleaned AS c
                ON r.review_key = c.review_key
            WHERE c.review_key IS NULL
            """,
            test_conn,
        )["n"].iloc[0]
    )

    cleaned_without_raw = int(
        pd.read_sql_query(
            """
            SELECT COUNT(*) AS n
            FROM phase2_reviews_cleaned AS c
            LEFT JOIN phase2_reviews_raw AS r
                ON c.review_key = r.review_key
            WHERE r.review_key IS NULL
            """,
            test_conn,
        )["n"].iloc[0]
    )

    orphan_quality_flags = int(
        pd.read_sql_query(
            """
            SELECT COUNT(*) AS n
            FROM phase2_quality_flags AS q
            LEFT JOIN phase2_reviews_raw AS r
                ON q.review_key = r.review_key
            WHERE r.review_key IS NULL
            """,
            test_conn,
        )["n"].iloc[0]
    )

    previous_app_fetch_df = pd.read_sql_query(
        """
        SELECT
            a.app_name,
            a.app_id,
            COUNT(r.review_key) AS inserted_rows,
            COUNT(
                DISTINCT r.fetched_at
            ) AS distinct_fetch_timestamps,
            MIN(r.fetched_at) AS previous_fetched_at,
            MAX(r.fetched_at) AS previous_fetched_at_max
        FROM phase2_apps AS a
        LEFT JOIN phase2_reviews_raw AS r
            ON a.app_id = r.app_id
            AND r.run_id = ?
        GROUP BY
            a.app_name,
            a.app_id
        ORDER BY a.rowid
        """,
        test_conn,
        params=(PREVIOUS_RUN_ID,),
    )

database_checks = {
    "raw_rows_equal_30848": (
        raw_rows_before == 30848
    ),
    "cleaned_rows_equal_30848": (
        cleaned_rows_before == 30848
    ),
    "raw_and_cleaned_counts_match": (
        raw_rows_before == cleaned_rows_before
    ),
    "five_completed_prior_runs": (
        completed_run_count == 5
    ),
    "fifty_app_summary_rows": (
        app_summary_row_count == 50
    ),
    "latest_run_matches_metadata": (
        len(latest_run_df) == 1
        and latest_run_df["run_id"].iloc[0]
        == PREVIOUS_RUN_ID
    ),
    "latest_run_completed": (
        len(latest_run_df) == 1
        and latest_run_df["status"].iloc[0]
        == "completed"
    ),
    "latest_run_is_first_collection": (
        len(latest_run_df) == 1
        and latest_run_df["run_label"].iloc[0]
        == (
            "phase2_cadence_runB_"
            "first_collection"
        )
    ),
    "latest_run_used_1200_target": (
        len(latest_run_df) == 1
        and int(
            latest_run_df[
                "target_reviews_per_app"
            ].iloc[0]
        )
        == 1200
    ),
    "latest_run_used_10_apps": (
        len(latest_run_df) == 1
        and int(
            latest_run_df["app_count"].iloc[0]
        )
        == 10
    ),
    "ten_previous_app_fetch_boundaries": (
        len(previous_app_fetch_df) == 10
    ),
    "all_apps_have_inserted_rows": (
        previous_app_fetch_df[
            "inserted_rows"
        ].gt(0).all()
    ),
    "one_fetch_timestamp_per_app": (
        previous_app_fetch_df[
            "distinct_fetch_timestamps"
        ].eq(1).all()
    ),
    "no_duplicate_review_identities": (
        duplicate_identity_groups == 0
    ),
    "no_raw_rows_without_cleaned": (
        raw_without_cleaned == 0
    ),
    "no_cleaned_rows_without_raw": (
        cleaned_without_raw == 0
    ),
    "no_orphan_quality_flags": (
        orphan_quality_flags == 0
    ),
}

failed_database_checks = [
    check_name
    for check_name, passed
    in database_checks.items()
    if not passed
]

if failed_database_checks:
    raise ValueError(
        "Checkpoint database validation failed: "
        f"{failed_database_checks}"
    )

# Copy the validated checkpoint database into
# the working database directory.
if DB_PATH.exists():
    DB_PATH.unlink()

shutil.copy2(
    source_db_path,
    DB_PATH,
)

previous_app_fetch_df[
    "previous_fetched_at_parsed"
] = pd.to_datetime(
    previous_app_fetch_df[
        "previous_fetched_at"
    ],
    utc=True,
    errors="raise",
)

validation_time = pd.Timestamp.now(
    tz="UTC"
)

previous_app_fetch_df[
    "hours_since_previous_collection"
] = (
    validation_time
    - previous_app_fetch_df[
        "previous_fetched_at_parsed"
    ]
).dt.total_seconds() / 3600

checkpoint_validation_df = pd.DataFrame(
    [
        {
            "validation_check": check_name,
            "passed": passed,
        }
        for check_name, passed
        in database_checks.items()
    ]
)

print(
    "Run B first-collection checkpoint "
    "validated and loaded successfully."
)
print("-" * 78)
print("Database path:", DB_PATH)
print(
    "Database size:",
    f"{DB_PATH.stat().st_size / (1024 ** 2):.2f} MB",
)
print("Raw review rows:", f"{raw_rows_before:,}")
print(
    "Cleaned review rows:",
    f"{cleaned_rows_before:,}",
)
print(
    "Completed prior runs:",
    completed_run_count,
)
print(
    "Previous run:",
    PREVIOUS_RUN_LABEL,
)
print(
    "Previous run finished at:",
    PREVIOUS_RUN_FINISHED_AT,
)
print(
    "Current validation time:",
    validation_time.isoformat(),
)

print("\nPrevious app-level collection boundaries:")
display(
    previous_app_fetch_df[
        [
            "app_name",
            "app_id",
            "inserted_rows",
            "previous_fetched_at",
            "hours_since_previous_collection",
        ]
    ]
)

print("\nCheckpoint validation checks:")
display(checkpoint_validation_df)

print("\nLatest completed run:")
display(latest_run_df)

Upload this checkpoint file:
phase2_cadence_runB_first_collection_checkpoint_20260714_020137_utc.zip


Saving phase2_cadence_runB_first_collection_checkpoint_20260714_020137_utc.zip to phase2_cadence_runB_first_collection_checkpoint_20260714_020137_utc.zip
Checkpoint zip extracted successfully.
Checkpoint manifest validation passed.
Run B first-collection checkpoint validated and loaded successfully.
------------------------------------------------------------------------------
Database path: /content/database/google_play_reviews.sqlite
Database size: 74.09 MB
Raw review rows: 30,848
Cleaned review rows: 30,848
Completed prior runs: 5
Previous run: phase2_cadence_runB_first_collection
Previous run finished at: 2026-07-14T01:58:44.989455+00:00
Current validation time: 2026-07-14T16:31:23.540146+00:00

Previous app-level collection boundaries:


,app_name,app_id,inserted_rows,previous_fetched_at,hours_since_previous_collection
0,YouTube,com.google.android.youtube,1193,2026-07-14T01:56:37.779729+00:00,14.579378
1,TikTok,com.zhiliaoapp.musically,1188,2026-07-14T01:56:42.049396+00:00,14.578192
2,Spotify,com.spotify.music,1200,2026-07-14T01:56:45.466435+00:00,14.577243
3,Instagram,com.instagram.android,1199,2026-07-14T01:56:48.271551+00:00,14.576463
4,Uber,com.ubercab,1199,2026-07-14T01:56:51.086121+00:00,14.575682
5,DoorDash,com.dd.doordash,547,2026-07-14T01:56:54.049406+00:00,14.574859
6,Duolingo,com.duolingo,5,2026-07-14T01:56:57.030308+00:00,14.574031
7,Google Maps,com.google.android.apps.maps,866,2026-07-14T01:56:59.944051+00:00,14.573221
8,Netflix,com.netflix.mediaclient,742,2026-07-14T01:57:02.795277+00:00,14.572429
9,Reddit,com.reddit.frontpage,499,2026-07-14T01:57:05.791189+00:00,14.571597



Checkpoint validation checks:


,validation_check,passed
0,raw_rows_equal_30848,True
1,cleaned_rows_equal_30848,True
2,raw_and_cleaned_counts_match,True
3,five_completed_prior_runs,True
4,fifty_app_summary_rows,True
5,latest_run_matches_metadata,True
6,latest_run_completed,True
7,latest_run_is_first_collection,True
8,latest_run_used_1200_target,True
9,latest_run_used_10_apps,True



Latest completed run:


,run_id,run_label,frequency_label,target_reviews_per_app,app_count,run_started_at,run_finished_at,status,records_fetched_total,new_records_inserted_total,duplicates_skipped_total,review_rows_after
0,phase2_cadence_runB_first_collection_20260714_...,phase2_cadence_runB_first_collection,runB_first_collection,1200,10,2026-07-14T01:54:19.819646+00:00,2026-07-14T01:58:44.989455+00:00,completed,12000,8638,3362,30848


## 3. Open the validated database and record the follow-up pre-run snapshot

Before the follow-up collection begins, this section records the current database state and the exact previous collection boundary for each app.

The snapshot confirms:

- 30,848 existing raw and cleaned review rows
- five completed prior Phase 2 runs
- the completed Run B first collection as the latest run
- the same fixed 10-app configuration
- one previous collection timestamp for each app
- the current database size and app-level review counts

Each app's previous `fetched_at` value will be used later to determine whether newly inserted reviews were actually posted between the two Run B collections.

In [4]:
# Close an older connection only if this cell is rerun.
if "conn" in globals():
    try:
        conn.close()
    except Exception:
        pass

conn = sqlite3.connect(DB_PATH)
conn.execute("PRAGMA foreign_keys = ON")
cur = conn.cursor()

# Read the fixed app configuration from the working database.
app_config_df = pd.read_sql_query(
    """
    SELECT
        app_name,
        app_id,
        source,
        language,
        country,
        title_from_store,
        score_from_store,
        ratings_from_store,
        installs_from_store
    FROM phase2_apps
    ORDER BY rowid
    """,
    conn,
)

expected_apps = [
    ("YouTube", "com.google.android.youtube"),
    ("TikTok", "com.zhiliaoapp.musically"),
    ("Spotify", "com.spotify.music"),
    ("Instagram", "com.instagram.android"),
    ("Uber", "com.ubercab"),
    ("DoorDash", "com.dd.doordash"),
    ("Duolingo", "com.duolingo"),
    ("Google Maps", "com.google.android.apps.maps"),
    ("Netflix", "com.netflix.mediaclient"),
    ("Reddit", "com.reddit.frontpage"),
]

actual_apps = list(
    app_config_df[
        ["app_name", "app_id"]
    ].itertuples(
        index=False,
        name=None,
    )
)

if actual_apps != expected_apps:
    raise ValueError(
        "The working database app configuration "
        "does not match the fixed 10-app setup."
    )

if not app_config_df["source"].eq(SOURCE).all():
    raise ValueError(
        "The working database source does not match Google Play."
    )

if not app_config_df["language"].eq(LANGUAGE).all():
    raise ValueError(
        "The working database language does not match English."
    )

if not app_config_df["country"].eq(COUNTRY).all():
    raise ValueError(
        "The working database country does not match the United States."
    )

if TARGET_REVIEWS_PER_APP != 1200:
    raise ValueError(
        "The controlled target must remain 1,200 reviews per app."
    )

# Record the pre-run database state.
db_size_before_mb = float(
    DB_PATH.stat().st_size / (1024 ** 2)
)

raw_rows_before = int(
    pd.read_sql_query(
        """
        SELECT COUNT(*) AS n
        FROM phase2_reviews_raw
        """,
        conn,
    )["n"].iloc[0]
)

cleaned_rows_before = int(
    pd.read_sql_query(
        """
        SELECT COUNT(*) AS n
        FROM phase2_reviews_cleaned
        """,
        conn,
    )["n"].iloc[0]
)

quality_flag_rows_before = int(
    pd.read_sql_query(
        """
        SELECT COUNT(*) AS n
        FROM phase2_quality_flags
        """,
        conn,
    )["n"].iloc[0]
)

app_summary_rows_before = int(
    pd.read_sql_query(
        """
        SELECT COUNT(*) AS n
        FROM phase2_app_run_summary
        """,
        conn,
    )["n"].iloc[0]
)

if raw_rows_before != 30848:
    raise ValueError(
        f"Expected 30,848 raw review rows, "
        f"but found {raw_rows_before:,}."
    )

if cleaned_rows_before != 30848:
    raise ValueError(
        f"Expected 30,848 cleaned review rows, "
        f"but found {cleaned_rows_before:,}."
    )

if raw_rows_before != cleaned_rows_before:
    raise ValueError(
        "Raw and cleaned review counts do not match."
    )

prior_runs_df = pd.read_sql_query(
    """
    SELECT
        run_id,
        run_label,
        frequency_label,
        target_reviews_per_app,
        app_count,
        run_started_at,
        run_finished_at,
        runtime_seconds,
        status,
        records_fetched_total,
        new_records_inserted_total,
        duplicates_skipped_total,
        review_rows_before,
        review_rows_after,
        review_rows_growth,
        db_size_before_mb,
        db_size_after_mb,
        db_size_growth_mb,
        errors_total
    FROM phase2_ingestion_runs
    ORDER BY run_started_at
    """,
    conn,
)

if len(prior_runs_df) != 5:
    raise ValueError(
        f"Expected exactly 5 prior runs, "
        f"but found {len(prior_runs_df)}."
    )

if not prior_runs_df["status"].eq("completed").all():
    raise ValueError(
        "At least one prior Phase 2 run is not completed."
    )

latest_prior_run = prior_runs_df.iloc[-1]

if latest_prior_run["run_id"] != PREVIOUS_RUN_ID:
    raise ValueError(
        "The latest completed run does not match "
        "the Run B first-collection checkpoint."
    )

if (
    latest_prior_run["run_label"]
    != "phase2_cadence_runB_first_collection"
):
    raise ValueError(
        "The latest prior run is not the expected "
        "Run B first collection."
    )

# Read one app-specific previous fetch boundary for each app.
previous_app_boundary_df = pd.read_sql_query(
    """
    SELECT
        a.app_name,
        a.app_id,
        COUNT(r.review_key) AS inserted_rows,
        COUNT(
            DISTINCT r.fetched_at
        ) AS distinct_fetch_timestamps,
        MIN(r.fetched_at) AS previous_fetched_at,
        MAX(r.fetched_at) AS previous_fetched_at_max
    FROM phase2_apps AS a
    LEFT JOIN phase2_reviews_raw AS r
        ON a.app_id = r.app_id
        AND r.run_id = ?
    GROUP BY
        a.app_name,
        a.app_id
    ORDER BY a.rowid
    """,
    conn,
    params=(PREVIOUS_RUN_ID,),
)

if len(previous_app_boundary_df) != 10:
    raise ValueError(
        "Expected 10 app-specific previous collection boundaries."
    )

if not previous_app_boundary_df[
    "inserted_rows"
].gt(0).all():
    raise ValueError(
        "At least one app has no inserted records "
        "in the Run B first collection."
    )

if not previous_app_boundary_df[
    "distinct_fetch_timestamps"
].eq(1).all():
    raise ValueError(
        "At least one app does not have exactly one "
        "previous fetch timestamp."
    )

if not (
    previous_app_boundary_df["previous_fetched_at"]
    == previous_app_boundary_df["previous_fetched_at_max"]
).all():
    raise ValueError(
        "The minimum and maximum previous fetch timestamps "
        "do not match for at least one app."
    )

previous_app_boundary_df[
    "previous_fetched_at_parsed"
] = pd.to_datetime(
    previous_app_boundary_df["previous_fetched_at"],
    utc=True,
    errors="raise",
)

snapshot_at_ts = pd.Timestamp.now(tz="UTC")
snapshot_at = snapshot_at_ts.isoformat()

previous_app_boundary_df[
    "hours_since_previous_collection_at_snapshot"
] = (
    snapshot_at_ts
    - previous_app_boundary_df[
        "previous_fetched_at_parsed"
    ]
).dt.total_seconds() / 3600

previous_fetched_at_by_app = dict(
    zip(
        previous_app_boundary_df["app_id"],
        previous_app_boundary_df[
            "previous_fetched_at_parsed"
        ],
    )
)

pre_run_app_snapshot_df = pd.read_sql_query(
    """
    SELECT
        a.app_name,
        a.app_id,
        COUNT(r.review_key) AS database_review_rows,
        MIN(r.review_created_at) AS earliest_review_timestamp,
        MAX(r.review_created_at) AS latest_review_timestamp
    FROM phase2_apps AS a
    LEFT JOIN phase2_reviews_raw AS r
        ON a.app_id = r.app_id
    GROUP BY
        a.app_name,
        a.app_id
    ORDER BY a.rowid
    """,
    conn,
)

pre_run_snapshot_df = pd.DataFrame(
    [
        {
            "snapshot_at": snapshot_at,
            "database_path": str(DB_PATH),
            "database_size_mb": db_size_before_mb,
            "raw_review_rows": raw_rows_before,
            "cleaned_review_rows": cleaned_rows_before,
            "quality_flag_rows": quality_flag_rows_before,
            "app_run_summary_rows": app_summary_rows_before,
            "completed_prior_runs": len(prior_runs_df),
            "previous_run_id": PREVIOUS_RUN_ID,
            "previous_run_label": PREVIOUS_RUN_LABEL,
            "previous_run_started_at": PREVIOUS_RUN_STARTED_AT,
            "previous_run_finished_at": PREVIOUS_RUN_FINISHED_AT,
        }
    ]
)

print("Follow-up pre-run snapshot validated.")
print("-" * 76)
print("Fixed apps:", len(app_config_df))
print(
    "Target reviews per app:",
    TARGET_REVIEWS_PER_APP,
)
print(
    "Raw review rows before follow-up:",
    f"{raw_rows_before:,}",
)
print(
    "Cleaned review rows before follow-up:",
    f"{cleaned_rows_before:,}",
)
print(
    "Database size before follow-up:",
    f"{db_size_before_mb:.2f} MB",
)
print(
    "Completed prior runs:",
    len(prior_runs_df),
)
print(
    "Previous completed run:",
    PREVIOUS_RUN_LABEL,
)
print(
    "Snapshot recorded at:",
    snapshot_at,
)

print("\nPrevious app-specific collection boundaries:")
display(
    previous_app_boundary_df[
        [
            "app_name",
            "app_id",
            "inserted_rows",
            "previous_fetched_at",
            "hours_since_previous_collection_at_snapshot",
        ]
    ]
)

print("\nPre-run app-level database snapshot:")
display(pre_run_app_snapshot_df)

print("\nRun history through Run B first collection:")
display(prior_runs_df)

Follow-up pre-run snapshot validated.
----------------------------------------------------------------------------
Fixed apps: 10
Target reviews per app: 1200
Raw review rows before follow-up: 30,848
Cleaned review rows before follow-up: 30,848
Database size before follow-up: 74.09 MB
Completed prior runs: 5
Previous completed run: phase2_cadence_runB_first_collection
Snapshot recorded at: 2026-07-14T16:34:07.114307+00:00

Previous app-specific collection boundaries:


,app_name,app_id,inserted_rows,previous_fetched_at,hours_since_previous_collection_at_snapshot
0,YouTube,com.google.android.youtube,1193,2026-07-14T01:56:37.779729+00:00,14.624815
1,TikTok,com.zhiliaoapp.musically,1188,2026-07-14T01:56:42.049396+00:00,14.623629
2,Spotify,com.spotify.music,1200,2026-07-14T01:56:45.466435+00:00,14.622680
3,Instagram,com.instagram.android,1199,2026-07-14T01:56:48.271551+00:00,14.621901
4,Uber,com.ubercab,1199,2026-07-14T01:56:51.086121+00:00,14.621119
5,DoorDash,com.dd.doordash,547,2026-07-14T01:56:54.049406+00:00,14.620296
6,Duolingo,com.duolingo,5,2026-07-14T01:56:57.030308+00:00,14.619468
7,Google Maps,com.google.android.apps.maps,866,2026-07-14T01:56:59.944051+00:00,14.618658
8,Netflix,com.netflix.mediaclient,742,2026-07-14T01:57:02.795277+00:00,14.617866
9,Reddit,com.reddit.frontpage,499,2026-07-14T01:57:05.791189+00:00,14.617034



Pre-run app-level database snapshot:


,app_name,app_id,database_review_rows,earliest_review_timestamp,latest_review_timestamp
0,YouTube,com.google.android.youtube,4825,2026-07-06T09:09:19+00:00,2026-07-13T01:56:32+00:00
1,TikTok,com.zhiliaoapp.musically,3653,2026-07-05T03:11:35+00:00,2026-07-13T01:54:48+00:00
2,Spotify,com.spotify.music,3718,2026-07-05T12:54:10+00:00,2026-07-13T01:56:37+00:00
3,Instagram,com.instagram.android,4849,2026-07-06T13:35:02+00:00,2026-07-13T01:56:17+00:00
4,Uber,com.ubercab,3131,2026-07-04T03:52:16+00:00,2026-07-13T01:46:36+00:00
5,DoorDash,com.dd.doordash,1950,2026-06-28T23:28:16+00:00,2026-07-13T01:45:26+00:00
6,Duolingo,com.duolingo,2095,2026-07-06T04:53:03+00:00,2026-07-12T12:05:22+00:00
7,Google Maps,com.google.android.apps.maps,2484,2026-06-30T02:30:54+00:00,2026-07-13T01:48:56+00:00
8,Netflix,com.netflix.mediaclient,2213,2026-06-27T05:44:07+00:00,2026-07-13T01:26:29+00:00
9,Reddit,com.reddit.frontpage,1930,2026-06-27T04:42:08+00:00,2026-07-13T01:51:50+00:00



Run history through Run B first collection:


,run_id,run_label,frequency_label,target_reviews_per_app,app_count,run_started_at,run_finished_at,runtime_seconds,status,records_fetched_total,new_records_inserted_total,duplicates_skipped_total,review_rows_before,review_rows_after,review_rows_growth,db_size_before_mb,db_size_after_mb,db_size_growth_mb,errors_total
0,phase2_day1_controlled_scale_20260708_034445,phase2_day1_controlled_scale,once_daily_baseline,1200,10,2026-07-08T03:44:46.106977+00:00,2026-07-08T03:45:09.378094+00:00,23.271117,completed,12000,12000,0,0,12000,12000,1.832031,25.492188,23.660156,0
1,phase2_day2_daily_followup_20260708_041135,phase2_day2_daily_followup,daily_followup,1200,10,2026-07-08T04:16:09.351152+00:00,2026-07-08T04:17:25.150737+00:00,75.799585,completed,12000,156,11844,12000,12156,156,25.492188,30.187500,4.695312,0
2,phase2_day3_controlled_repeated_run_20260709_0...,phase2_day3_controlled_repeated_run,daily_followup_controlled_baseline,1200,10,2026-07-09T04:04:39.642719+00:00,2026-07-09T04:05:34.352603+00:00,29.269241,completed,12000,5659,6341,12156,17815,5659,30.187500,43.761719,13.574219,0
3,phase2_cadence_runA_twice_daily_test_20260709_...,phase2_cadence_runA_twice_daily_test,twice_daily_same_day_second_run,1200,10,2026-07-09T21:14:19.095514+00:00,2026-07-09T21:15:07.605058+00:00,28.142863,completed,12000,4395,7605,17815,22210,4395,43.761719,55.523438,11.761719,0
4,phase2_cadence_runB_first_collection_20260714_...,phase2_cadence_runB_first_collection,runB_first_collection,1200,10,2026-07-14T01:54:19.819646+00:00,2026-07-14T01:58:44.989455+00:00,10.093505,completed,12000,8638,3362,22210,30848,8638,55.523438,74.085938,18.562500,0


## 4. Define the normalization and database insertion functions

These functions preserve the same Phase 2 processing and duplicate-prevention logic used in the previous controlled collections.

Each valid review is identified by a deterministic key created from:

`source + app_id + review_id`

The database uses this key to prevent the same review from being inserted more than once.

For each newly inserted raw review, the notebook also creates:

- a matching cleaned review record
- applicable data-quality flags
- the original source response stored as JSON

The schema, normalization logic, and review identity are unchanged for the follow-up collection.

In [5]:
def utc_now_iso():
    return datetime.now(timezone.utc).isoformat()


def to_iso_utc(value):
    if value is None:
        return None

    try:
        if pd.isna(value):
            return None
    except (TypeError, ValueError):
        pass

    if isinstance(value, pd.Timestamp):
        dt = value.to_pydatetime()
    elif isinstance(value, datetime):
        dt = value
    else:
        try:
            dt = pd.to_datetime(value).to_pydatetime()
        except Exception:
            return str(value)

    if dt.tzinfo is None:
        dt = dt.replace(tzinfo=timezone.utc)

    return dt.astimezone(
        timezone.utc
    ).isoformat(
        timespec="seconds"
    )


def make_hash(text):
    return hashlib.sha256(
        str(text).encode("utf-8")
    ).hexdigest()


def make_review_key(
    source,
    app_id,
    review_id,
):
    return make_hash(
        f"{source}|{app_id}|{review_id}"
    )


def make_flag_id(
    run_id,
    review_key,
    flag_name,
):
    return make_hash(
        f"{run_id}|{review_key}|{flag_name}"
    )


def clean_content(text):
    if text is None:
        return None

    cleaned = str(text).strip()
    cleaned = re.sub(r"\s+", " ", cleaned)

    return cleaned


def json_ready(value):
    if isinstance(
        value,
        (datetime, pd.Timestamp),
    ):
        return to_iso_utc(value)

    return value


def raw_review_to_json(raw_review):
    cleaned = {}

    for key, value in raw_review.items():
        cleaned[key] = json_ready(value)

    return json.dumps(
        cleaned,
        ensure_ascii=False,
    )


def normalize_review(
    raw_review,
    app_id,
    app_name,
    fetched_at,
    run_id,
):
    review_id = raw_review.get("reviewId")

    review_key = None

    if review_id:
        review_key = make_review_key(
            SOURCE,
            app_id,
            review_id,
        )

    content_raw = raw_review.get("content")
    reply_content_raw = raw_review.get(
        "replyContent"
    )

    app_version = (
        raw_review.get(
            "reviewCreatedVersion"
        )
        or raw_review.get("appVersion")
    )

    return {
        "review_key": review_key,
        "source": SOURCE,
        "app_id": app_id,
        "app_name": app_name,
        "review_id": review_id,
        "user_name": raw_review.get(
            "userName"
        ),
        "user_image": raw_review.get(
            "userImage"
        ),
        "content_raw": content_raw,
        "score": raw_review.get("score"),
        "thumbs_up_count": raw_review.get(
            "thumbsUpCount"
        ),
        "review_created_at": to_iso_utc(
            raw_review.get("at")
        ),
        "reply_content_raw": (
            reply_content_raw
        ),
        "replied_at": to_iso_utc(
            raw_review.get("repliedAt")
        ),
        "app_version": app_version,
        "fetched_at": fetched_at,
        "run_id": run_id,
        "raw_json": raw_review_to_json(
            raw_review
        ),
    }


def make_cleaned_row(
    raw_row,
    cleaned_at,
):
    content_cleaned = clean_content(
        raw_row.get("content_raw")
    )

    return {
        "review_key": raw_row.get(
            "review_key"
        ),
        "source": raw_row.get("source"),
        "app_id": raw_row.get("app_id"),
        "content_cleaned": content_cleaned,
        "content_length": (
            len(content_cleaned)
            if content_cleaned is not None
            else None
        ),
        "has_developer_reply": (
            1
            if raw_row.get(
                "reply_content_raw"
            )
            not in [None, ""]
            else 0
        ),
        "score": raw_row.get("score"),
        "review_created_at": raw_row.get(
            "review_created_at"
        ),
        "app_version": raw_row.get(
            "app_version"
        ),
        "cleaned_at": cleaned_at,
        "run_id": raw_row.get("run_id"),
    }


def generate_quality_flags(
    raw_row,
    run_id,
):
    flags = []
    created_at = utc_now_iso()

    review_key = raw_row.get(
        "review_key"
    )
    app_id = raw_row.get("app_id")

    if review_key is None:
        return flags

    def add_flag(
        flag_name,
        severity,
        flag_value,
    ):
        flags.append(
            {
                "flag_id": make_flag_id(
                    run_id,
                    review_key,
                    flag_name,
                ),
                "review_key": review_key,
                "run_id": run_id,
                "app_id": app_id,
                "flag_name": flag_name,
                "flag_severity": severity,
                "flag_value": flag_value,
                "created_at": created_at,
            }
        )

    if raw_row.get(
        "content_raw"
    ) is None:
        add_flag(
            "missing_content",
            "warning",
            "missing",
        )
    elif str(
        raw_row.get("content_raw")
    ).strip() == "":
        add_flag(
            "empty_content",
            "warning",
            "empty",
        )

    if raw_row.get("score") is None:
        add_flag(
            "missing_score",
            "warning",
            "missing",
        )
    elif raw_row.get("score") not in [
        1,
        2,
        3,
        4,
        5,
    ]:
        add_flag(
            "invalid_score",
            "warning",
            str(raw_row.get("score")),
        )

    if raw_row.get(
        "review_created_at"
    ) is None:
        add_flag(
            "missing_review_date",
            "warning",
            "missing",
        )

    if raw_row.get(
        "app_version"
    ) in [None, ""]:
        add_flag(
            "missing_app_version",
            "info",
            "missing",
        )

    if raw_row.get(
        "reply_content_raw"
    ) in [None, ""]:
        add_flag(
            "missing_developer_reply",
            "info",
            "missing",
        )

    return flags


def insert_raw_review(
    conn,
    raw_row,
):
    columns = [
        "review_key",
        "source",
        "app_id",
        "app_name",
        "review_id",
        "user_name",
        "user_image",
        "content_raw",
        "score",
        "thumbs_up_count",
        "review_created_at",
        "reply_content_raw",
        "replied_at",
        "app_version",
        "fetched_at",
        "run_id",
        "raw_json",
    ]

    sql = f"""
        INSERT OR IGNORE INTO phase2_reviews_raw
        ({",".join(columns)})
        VALUES ({",".join(["?"] * len(columns))})
    """

    values = [
        raw_row.get(column)
        for column in columns
    ]

    cursor = conn.cursor()
    cursor.execute(sql, values)

    return int(cursor.rowcount)


def insert_cleaned_review(
    conn,
    cleaned_row,
):
    columns = [
        "review_key",
        "source",
        "app_id",
        "content_cleaned",
        "content_length",
        "has_developer_reply",
        "score",
        "review_created_at",
        "app_version",
        "cleaned_at",
        "run_id",
    ]

    sql = f"""
        INSERT OR IGNORE INTO phase2_reviews_cleaned
        ({",".join(columns)})
        VALUES ({",".join(["?"] * len(columns))})
    """

    values = [
        cleaned_row.get(column)
        for column in columns
    ]

    cursor = conn.cursor()
    cursor.execute(sql, values)

    return int(cursor.rowcount)


def insert_quality_flag(
    conn,
    flag_row,
):
    columns = [
        "flag_id",
        "review_key",
        "run_id",
        "app_id",
        "flag_name",
        "flag_severity",
        "flag_value",
        "created_at",
    ]

    sql = f"""
        INSERT OR IGNORE INTO phase2_quality_flags
        ({",".join(columns)})
        VALUES ({",".join(["?"] * len(columns))})
    """

    values = [
        flag_row.get(column)
        for column in columns
    ]

    cursor = conn.cursor()
    cursor.execute(sql, values)

    return int(cursor.rowcount)


print("Helper functions loaded successfully.")
print(
    "Duplicate identity: "
    "source + app_id + review_id"
)

Helper functions loaded successfully.
Duplicate identity: source + app_id + review_id


## 5. Create the Run B follow-up collection record

This step creates the database record for the follow-up collection in the second controlled cadence test.

The follow-up continues directly from the completed Run B first collection and keeps the same:

- Phase 2 SQLite database
- fixed 10-app configuration
- 1,200-review target per app
- Google Play source settings
- duplicate-prevention logic
- run-level and app-level tracking structure

The exact app-specific interval between the first and follow-up collections will be calculated from the stored collection timestamps.

In [6]:
existing_run_id_count = int(
    pd.read_sql_query(
        """
        SELECT COUNT(*) AS n
        FROM phase2_ingestion_runs
        WHERE run_id = ?
        """,
        conn,
        params=(RUN_ID,),
    )["n"].iloc[0]
)

if existing_run_id_count != 0:
    raise ValueError(
        "This Run ID already exists in the database. "
        "Do not insert the same run twice."
    )

existing_current_run_app_rows = int(
    pd.read_sql_query(
        """
        SELECT COUNT(*) AS n
        FROM phase2_app_run_summary
        WHERE run_id = ?
        """,
        conn,
        params=(RUN_ID,),
    )["n"].iloc[0]
)

existing_current_run_reviews = int(
    pd.read_sql_query(
        """
        SELECT COUNT(*) AS n
        FROM phase2_reviews_raw
        WHERE run_id = ?
        """,
        conn,
        params=(RUN_ID,),
    )["n"].iloc[0]
)

if (
    existing_current_run_app_rows != 0
    or existing_current_run_reviews != 0
):
    raise ValueError(
        "The current Run ID already contains collection results."
    )

run_started_at = utc_now_iso()

apps_included = ", ".join(
    app_config_df["app_name"].tolist()
)

cur.execute(
    """
    INSERT INTO phase2_ingestion_runs (
        run_id,
        run_label,
        phase,
        frequency_label,
        source,
        language,
        country,
        target_reviews_per_app,
        app_count,
        apps_included,
        run_started_at,
        status,
        records_fetched_total,
        new_records_inserted_total,
        duplicates_skipped_total,
        errors_total,
        quality_flag_total,
        quality_flags_inserted,
        db_size_before_mb,
        review_rows_before,
        notes
    )
    VALUES (
        ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?,
        ?, ?, ?, ?, ?, ?, ?, ?, ?, ?
    )
    """,
    (
        str(RUN_ID),
        str(RUN_LABEL),
        "phase2",
        str(FREQUENCY_LABEL),
        str(SOURCE),
        str(LANGUAGE),
        str(COUNTRY),
        int(TARGET_REVIEWS_PER_APP),
        int(len(app_config_df)),
        str(apps_included),
        str(run_started_at),
        "running",
        0,
        0,
        0,
        0,
        0,
        0,
        float(db_size_before_mb),
        int(raw_rows_before),
        (
            "Run B follow-up collection for the second "
            "controlled cadence test. Continues from the "
            "completed Run B first-collection database "
            "using the same 10 apps, the same 1,200-review "
            "target, and the same duplicate-prevention "
            "logic. App-specific previous fetched_at "
            "timestamps will be used to validate whether "
            "newly inserted reviews were actually posted "
            "between collections."
        ),
    ),
)

conn.commit()

created_run_df = pd.read_sql_query(
    """
    SELECT
        run_id,
        run_label,
        frequency_label,
        target_reviews_per_app,
        app_count,
        run_started_at,
        status,
        review_rows_before,
        db_size_before_mb,
        notes
    FROM phase2_ingestion_runs
    WHERE run_id = ?
    """,
    conn,
    params=(RUN_ID,),
)

if len(created_run_df) != 1:
    raise ValueError(
        "The Run B follow-up record "
        "was not created correctly."
    )

if created_run_df[
    "status"
].iloc[0] != "running":
    raise ValueError(
        "The follow-up run record does not "
        "have the expected running status."
    )

if int(
    created_run_df[
        "review_rows_before"
    ].iloc[0]
) != 30848:
    raise ValueError(
        "The follow-up run record contains "
        "an incorrect pre-run review count."
    )

print(
    "Run B follow-up collection record "
    "created successfully."
)
print("Run started at:", run_started_at)
print(
    "Review rows before:",
    f"{raw_rows_before:,}",
)
print(
    "Database size before:",
    f"{db_size_before_mb:.2f} MB",
)

display(created_run_df)

Run B follow-up collection record created successfully.
Run started at: 2026-07-14T16:34:49.444978+00:00
Review rows before: 30,848
Database size before: 74.09 MB


,run_id,run_label,frequency_label,target_reviews_per_app,app_count,run_started_at,status,review_rows_before,db_size_before_mb,notes
0,phase2_cadence_runB_followup_collection_202607...,phase2_cadence_runB_followup_collection,runB_followup_collection,1200,10,2026-07-14T16:34:49.444978+00:00,running,30848,74.085938,Run B follow-up collection for the second cont...


## 6. Run the Run B follow-up collection

This step collects the 1,200 newest reviews for each of the same 10 apps.

The source settings, app order, review target, database schema, and duplicate-prevention logic remain unchanged.

For each app, the notebook records:

- reviews returned by Google Play
- unique and repeated review IDs within the returned batch
- records newly inserted into the database
- records skipped because they already exist
- the previous and current app-specific collection timestamps
- the exact interval between collections
- review timestamp range
- data-quality checks
- runtime and collection errors

A review ID that is new to the database is not automatically treated as a newly posted review. That distinction will be evaluated after collection using the stored review timestamps and each app's exact collection boundaries.

In [7]:
# Confirm that this follow-up collection has not already started.
current_run_status_df = pd.read_sql_query(
    """
    SELECT status
    FROM phase2_ingestion_runs
    WHERE run_id = ?
    """,
    conn,
    params=(RUN_ID,),
)

if len(current_run_status_df) != 1:
    raise ValueError(
        "The Run B follow-up record was not found."
    )

if current_run_status_df["status"].iloc[0] != "running":
    raise ValueError(
        "This follow-up collection is not in running status."
    )

existing_app_summary_count = int(
    pd.read_sql_query(
        """
        SELECT COUNT(*) AS n
        FROM phase2_app_run_summary
        WHERE run_id = ?
        """,
        conn,
        params=(RUN_ID,),
    )["n"].iloc[0]
)

existing_run_review_count = int(
    pd.read_sql_query(
        """
        SELECT COUNT(*) AS n
        FROM phase2_reviews_raw
        WHERE run_id = ?
        """,
        conn,
        params=(RUN_ID,),
    )["n"].iloc[0]
)

existing_run_flag_count = int(
    pd.read_sql_query(
        """
        SELECT COUNT(*) AS n
        FROM phase2_quality_flags
        WHERE run_id = ?
        """,
        conn,
        params=(RUN_ID,),
    )["n"].iloc[0]
)

if any(
    value != 0
    for value in [
        existing_app_summary_count,
        existing_run_review_count,
        existing_run_flag_count,
    ]
):
    raise ValueError(
        "The Run B follow-up already contains results. "
        "Do not run this collection cell twice."
    )

collection_start_time = time.time()

app_summaries = []
quality_flags_created = []
new_review_rows = []
errors = []

print("Starting Run B follow-up collection...")
print("Run ID:", RUN_ID)

for i, app_row in app_config_df.iterrows():
    app_start_time = time.time()

    app_name = str(app_row["app_name"])
    app_id = str(app_row["app_id"])

    print("\n" + "=" * 90)
    print(
        f"[{i + 1}/{len(app_config_df)}] "
        f"Collecting {app_name} ({app_id})"
    )

    records_fetched = 0
    unique_reviews_in_batch = 0
    duplicate_reviews_in_batch = 0
    new_records_inserted = 0
    duplicates_skipped = 0
    quality_flag_count = 0
    quality_flags_inserted = 0
    error_message = ""

    missing_review_id_count = 0
    missing_content_count = 0
    empty_content_count = 0
    missing_score_count = 0
    invalid_score_count = 0
    missing_review_date_count = 0
    missing_app_version_count = 0
    missing_developer_reply_count = 0

    min_review_date = None
    max_review_date = None

    app_new_review_rows = []
    app_quality_flags_created = []

    previous_fetched_at_ts = (
        previous_fetched_at_by_app[app_id]
    )

    current_fetched_at = None
    current_fetched_at_ts = None
    interval_hours = None

    conn.execute("SAVEPOINT app_collection")

    try:
        current_fetched_at = utc_now_iso()

        current_fetched_at_ts = pd.to_datetime(
            current_fetched_at,
            utc=True,
            errors="raise",
        )

        interval_hours = float(
            (
                current_fetched_at_ts
                - previous_fetched_at_ts
            ).total_seconds()
            / 3600
        )

        fetched_reviews, continuation_token = reviews(
            app_id,
            lang=LANGUAGE,
            country=COUNTRY,
            sort=SORT_METHOD,
            count=TARGET_REVIEWS_PER_APP,
        )

        records_fetched = int(len(fetched_reviews))

        normalized_rows = [
            normalize_review(
                raw_review,
                app_id,
                app_name,
                current_fetched_at,
                RUN_ID,
            )
            for raw_review in fetched_reviews
        ]

        valid_review_keys = [
            row["review_key"]
            for row in normalized_rows
            if row.get("review_key") is not None
        ]

        unique_reviews_in_batch = int(
            len(set(valid_review_keys))
        )

        duplicate_reviews_in_batch = int(
            len(valid_review_keys)
            - unique_reviews_in_batch
        )

        review_dates = [
            row["review_created_at"]
            for row in normalized_rows
            if row.get("review_created_at") is not None
        ]

        if review_dates:
            min_review_date = min(review_dates)
            max_review_date = max(review_dates)

        for raw_row in normalized_rows:
            if raw_row.get("review_id") in [None, ""]:
                missing_review_id_count += 1
                continue

            if raw_row.get("content_raw") is None:
                missing_content_count += 1
            elif str(
                raw_row.get("content_raw")
            ).strip() == "":
                empty_content_count += 1

            if raw_row.get("score") is None:
                missing_score_count += 1
            elif raw_row.get("score") not in [
                1,
                2,
                3,
                4,
                5,
            ]:
                invalid_score_count += 1

            if raw_row.get("review_created_at") is None:
                missing_review_date_count += 1

            if raw_row.get("app_version") in [None, ""]:
                missing_app_version_count += 1

            if raw_row.get("reply_content_raw") in [
                None,
                "",
            ]:
                missing_developer_reply_count += 1

            inserted_raw = insert_raw_review(
                conn,
                raw_row,
            )

            if inserted_raw == 1:
                cleaned_row = make_cleaned_row(
                    raw_row,
                    utc_now_iso(),
                )

                inserted_cleaned = insert_cleaned_review(
                    conn,
                    cleaned_row,
                )

                if inserted_cleaned != 1:
                    raise ValueError(
                        "A newly inserted raw review did not "
                        "create a matching cleaned review."
                    )

                new_records_inserted += 1
                app_new_review_rows.append(raw_row)

            else:
                duplicates_skipped += 1

            flags = generate_quality_flags(
                raw_row,
                RUN_ID,
            )

            quality_flag_count += len(flags)

            for flag in flags:
                inserted_flag = insert_quality_flag(
                    conn,
                    flag,
                )

                quality_flags_inserted += inserted_flag

                if inserted_flag == 1:
                    app_quality_flags_created.append(
                        flag
                    )

        app_runtime_seconds = float(
            time.time() - app_start_time
        )

        app_summary = {
            "run_id": RUN_ID,
            "app_name": app_name,
            "app_id": app_id,
            "target_reviews": int(
                TARGET_REVIEWS_PER_APP
            ),
            "records_fetched": int(records_fetched),
            "unique_reviews_in_batch": int(
                unique_reviews_in_batch
            ),
            "duplicate_reviews_in_batch": int(
                duplicate_reviews_in_batch
            ),
            "new_records_inserted": int(
                new_records_inserted
            ),
            "duplicates_skipped": int(
                duplicates_skipped
            ),
            "runtime_seconds": app_runtime_seconds,
            "min_review_date": min_review_date,
            "max_review_date": max_review_date,
            "missing_review_id_count": int(
                missing_review_id_count
            ),
            "missing_content_count": int(
                missing_content_count
            ),
            "empty_content_count": int(
                empty_content_count
            ),
            "missing_score_count": int(
                missing_score_count
            ),
            "invalid_score_count": int(
                invalid_score_count
            ),
            "missing_review_date_count": int(
                missing_review_date_count
            ),
            "missing_app_version_count": int(
                missing_app_version_count
            ),
            "missing_developer_reply_count": int(
                missing_developer_reply_count
            ),
            "quality_flag_count": int(
                quality_flag_count
            ),
            "quality_flags_inserted": int(
                quality_flags_inserted
            ),
            "previous_fetched_at": (
                previous_fetched_at_ts.isoformat()
            ),
            "current_fetched_at": (
                current_fetched_at_ts.isoformat()
            ),
            "collection_interval_hours": (
                interval_hours
            ),
            "error_message": error_message,
        }

        cur.execute(
            """
            INSERT INTO phase2_app_run_summary (
                run_id,
                app_name,
                app_id,
                target_reviews,
                records_fetched,
                unique_reviews_in_batch,
                duplicate_reviews_in_batch,
                new_records_inserted,
                duplicates_skipped,
                runtime_seconds,
                min_review_date,
                max_review_date,
                missing_review_id_count,
                missing_content_count,
                empty_content_count,
                missing_score_count,
                invalid_score_count,
                missing_review_date_count,
                missing_app_version_count,
                missing_developer_reply_count,
                quality_flag_count,
                error_message
            )
            VALUES (
                ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?,
                ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?
            )
            """,
            (
                str(app_summary["run_id"]),
                str(app_summary["app_name"]),
                str(app_summary["app_id"]),
                int(app_summary["target_reviews"]),
                int(app_summary["records_fetched"]),
                int(
                    app_summary[
                        "unique_reviews_in_batch"
                    ]
                ),
                int(
                    app_summary[
                        "duplicate_reviews_in_batch"
                    ]
                ),
                int(
                    app_summary[
                        "new_records_inserted"
                    ]
                ),
                int(
                    app_summary[
                        "duplicates_skipped"
                    ]
                ),
                float(
                    app_summary["runtime_seconds"]
                ),
                app_summary["min_review_date"],
                app_summary["max_review_date"],
                int(
                    app_summary[
                        "missing_review_id_count"
                    ]
                ),
                int(
                    app_summary[
                        "missing_content_count"
                    ]
                ),
                int(
                    app_summary[
                        "empty_content_count"
                    ]
                ),
                int(
                    app_summary[
                        "missing_score_count"
                    ]
                ),
                int(
                    app_summary[
                        "invalid_score_count"
                    ]
                ),
                int(
                    app_summary[
                        "missing_review_date_count"
                    ]
                ),
                int(
                    app_summary[
                        "missing_app_version_count"
                    ]
                ),
                int(
                    app_summary[
                        "missing_developer_reply_count"
                    ]
                ),
                int(
                    app_summary[
                        "quality_flag_count"
                    ]
                ),
                str(app_summary["error_message"]),
            ),
        )

        conn.execute(
            "RELEASE SAVEPOINT app_collection"
        )
        conn.commit()

        app_summaries.append(app_summary)
        new_review_rows.extend(app_new_review_rows)
        quality_flags_created.extend(
            app_quality_flags_created
        )

    except Exception as e:
        conn.execute(
            "ROLLBACK TO SAVEPOINT app_collection"
        )
        conn.execute(
            "RELEASE SAVEPOINT app_collection"
        )
        conn.commit()

        error_message = repr(e)

        errors.append(
            {
                "app_name": app_name,
                "app_id": app_id,
                "error_message": error_message,
            }
        )

        app_runtime_seconds = float(
            time.time() - app_start_time
        )

        app_summary = {
            "run_id": RUN_ID,
            "app_name": app_name,
            "app_id": app_id,
            "target_reviews": int(
                TARGET_REVIEWS_PER_APP
            ),
            "records_fetched": int(records_fetched),
            "unique_reviews_in_batch": int(
                unique_reviews_in_batch
            ),
            "duplicate_reviews_in_batch": int(
                duplicate_reviews_in_batch
            ),
            "new_records_inserted": 0,
            "duplicates_skipped": 0,
            "runtime_seconds": app_runtime_seconds,
            "min_review_date": min_review_date,
            "max_review_date": max_review_date,
            "missing_review_id_count": int(
                missing_review_id_count
            ),
            "missing_content_count": int(
                missing_content_count
            ),
            "empty_content_count": int(
                empty_content_count
            ),
            "missing_score_count": int(
                missing_score_count
            ),
            "invalid_score_count": int(
                invalid_score_count
            ),
            "missing_review_date_count": int(
                missing_review_date_count
            ),
            "missing_app_version_count": int(
                missing_app_version_count
            ),
            "missing_developer_reply_count": int(
                missing_developer_reply_count
            ),
            "quality_flag_count": 0,
            "quality_flags_inserted": 0,
            "previous_fetched_at": (
                previous_fetched_at_ts.isoformat()
            ),
            "current_fetched_at": (
                current_fetched_at_ts.isoformat()
                if current_fetched_at_ts is not None
                else None
            ),
            "collection_interval_hours": (
                interval_hours
            ),
            "error_message": error_message,
        }

        app_summaries.append(app_summary)

        cur.execute(
            """
            INSERT INTO phase2_app_run_summary (
                run_id,
                app_name,
                app_id,
                target_reviews,
                records_fetched,
                unique_reviews_in_batch,
                duplicate_reviews_in_batch,
                new_records_inserted,
                duplicates_skipped,
                runtime_seconds,
                min_review_date,
                max_review_date,
                missing_review_id_count,
                missing_content_count,
                empty_content_count,
                missing_score_count,
                invalid_score_count,
                missing_review_date_count,
                missing_app_version_count,
                missing_developer_reply_count,
                quality_flag_count,
                error_message
            )
            VALUES (
                ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?,
                ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?
            )
            """,
            (
                str(app_summary["run_id"]),
                str(app_summary["app_name"]),
                str(app_summary["app_id"]),
                int(app_summary["target_reviews"]),
                int(app_summary["records_fetched"]),
                int(
                    app_summary[
                        "unique_reviews_in_batch"
                    ]
                ),
                int(
                    app_summary[
                        "duplicate_reviews_in_batch"
                    ]
                ),
                0,
                0,
                float(
                    app_summary["runtime_seconds"]
                ),
                app_summary["min_review_date"],
                app_summary["max_review_date"],
                int(
                    app_summary[
                        "missing_review_id_count"
                    ]
                ),
                int(
                    app_summary[
                        "missing_content_count"
                    ]
                ),
                int(
                    app_summary[
                        "empty_content_count"
                    ]
                ),
                int(
                    app_summary[
                        "missing_score_count"
                    ]
                ),
                int(
                    app_summary[
                        "invalid_score_count"
                    ]
                ),
                int(
                    app_summary[
                        "missing_review_date_count"
                    ]
                ),
                int(
                    app_summary[
                        "missing_app_version_count"
                    ]
                ),
                int(
                    app_summary[
                        "missing_developer_reply_count"
                    ]
                ),
                0,
                str(app_summary["error_message"]),
            ),
        )

        conn.commit()

        print("ERROR:", error_message)

    print(
        f"Fetched={app_summary['records_fetched']:,} | "
        f"New inserts="
        f"{app_summary['new_records_inserted']:,} | "
        f"Duplicates skipped="
        f"{app_summary['duplicates_skipped']:,} | "
        f"Interval="
        f"{app_summary['collection_interval_hours']:.2f}h | "
        f"Runtime="
        f"{app_summary['runtime_seconds']:.2f}s | "
        f"Quality flags="
        f"{app_summary['quality_flag_count']:,}"
    )

    time.sleep(REQUEST_SLEEP_SECONDS)

collection_runtime_seconds = float(
    time.time() - collection_start_time
)

print("\n" + "=" * 90)
print("Run B follow-up collection finished.")
print(
    "Wall-clock collection runtime:",
    f"{collection_runtime_seconds:.2f} seconds",
)
print("Apps processed:", len(app_summaries))
print("Apps with errors:", len(errors))

Starting Run B follow-up collection...
Run ID: phase2_cadence_runB_followup_collection_20260714_163017

[1/10] Collecting YouTube (com.google.android.youtube)
Fetched=1,200 | New inserts=994 | Duplicates skipped=206 | Interval=14.66h | Runtime=1.93s | Quality flags=1,218

[2/10] Collecting TikTok (com.zhiliaoapp.musically)
Fetched=1,200 | New inserts=406 | Duplicates skipped=794 | Interval=14.66h | Runtime=0.79s | Quality flags=616

[3/10] Collecting Spotify (com.spotify.music)
Fetched=1,200 | New inserts=456 | Duplicates skipped=744 | Interval=14.66h | Runtime=0.77s | Quality flags=1,274

[4/10] Collecting Instagram (com.instagram.android)
Fetched=1,200 | New inserts=1,196 | Duplicates skipped=4 | Interval=14.66h | Runtime=0.85s | Quality flags=1,589

[5/10] Collecting Uber (com.ubercab)
Fetched=1,200 | New inserts=309 | Duplicates skipped=891 | Interval=14.66h | Runtime=0.78s | Quality flags=1,348

[6/10] Collecting DoorDash (com.dd.doordash)
Fetched=1,200 | New inserts=25 | Duplicat

## 7. Finalize the follow-up collection and validate the updated database

After the follow-up collection, this section verifies that the run was written to the database correctly.

The validation confirms that:

- all 10 apps were processed
- exactly 12,000 reviews were returned
- new inserts and skipped duplicates reconcile to the fetched total
- every newly inserted raw review has one matching cleaned review
- database growth matches the number of newly inserted reviews
- no duplicate review identities were created
- no orphan raw, cleaned, or quality-flag records were created
- app-specific collection intervals were recorded correctly
- the run-level database record matches the app-level results

The follow-up run is marked as completed only after all required checks pass.

In [8]:
app_summary_df = pd.DataFrame(app_summaries)
error_df = pd.DataFrame(errors)

if len(app_summary_df) != 10:
    raise ValueError(
        f"Expected 10 app summaries, "
        f"but found {len(app_summary_df)}."
    )

if app_summary_df["app_id"].nunique() != 10:
    raise ValueError(
        "The follow-up results do not contain "
        "10 unique apps."
    )

expected_app_ids = app_config_df[
    "app_id"
].tolist()

actual_app_ids = app_summary_df[
    "app_id"
].tolist()

if actual_app_ids != expected_app_ids:
    raise ValueError(
        "The app order or app IDs changed "
        "during the follow-up collection."
    )

records_fetched_total = int(
    app_summary_df[
        "records_fetched"
    ].sum()
)

new_records_inserted_total = int(
    app_summary_df[
        "new_records_inserted"
    ].sum()
)

duplicates_skipped_total = int(
    app_summary_df[
        "duplicates_skipped"
    ].sum()
)

errors_total = int(len(error_df))

quality_flag_total = int(
    app_summary_df[
        "quality_flag_count"
    ].sum()
)

quality_flags_inserted_total = int(
    app_summary_df[
        "quality_flags_inserted"
    ].sum()
)

processing_runtime_seconds = float(
    app_summary_df[
        "runtime_seconds"
    ].sum()
)

missing_review_id_total = int(
    app_summary_df[
        "missing_review_id_count"
    ].sum()
)

interval_min_hours = float(
    app_summary_df[
        "collection_interval_hours"
    ].min()
)

interval_max_hours = float(
    app_summary_df[
        "collection_interval_hours"
    ].max()
)

interval_mean_hours = float(
    app_summary_df[
        "collection_interval_hours"
    ].mean()
)

if records_fetched_total != 12000:
    raise ValueError(
        f"Expected 12,000 fetched reviews, "
        f"but found {records_fetched_total:,}."
    )

if (
    new_records_inserted_total
    + duplicates_skipped_total
    != records_fetched_total
):
    raise ValueError(
        "New inserts plus skipped duplicates "
        "do not match the fetched total."
    )

if errors_total != 0:
    raise ValueError(
        f"The follow-up collection contains "
        f"{errors_total} app error(s)."
    )

if missing_review_id_total != 0:
    raise ValueError(
        f"The follow-up returned "
        f"{missing_review_id_total} review(s) "
        "without a review ID."
    )

if not app_summary_df[
    "records_fetched"
].eq(1200).all():
    raise ValueError(
        "At least one app did not return "
        "exactly 1,200 reviews."
    )

if app_summary_df[
    "previous_fetched_at"
].isna().any():
    raise ValueError(
        "At least one app is missing its "
        "previous collection timestamp."
    )

if app_summary_df[
    "current_fetched_at"
].isna().any():
    raise ValueError(
        "At least one app is missing its "
        "current collection timestamp."
    )

if app_summary_df[
    "collection_interval_hours"
].le(0).any():
    raise ValueError(
        "At least one app has a non-positive "
        "collection interval."
    )

app_summary_rows_in_db = int(
    pd.read_sql_query(
        """
        SELECT COUNT(*) AS n
        FROM phase2_app_run_summary
        WHERE run_id = ?
        """,
        conn,
        params=(RUN_ID,),
    )["n"].iloc[0]
)

database_app_summary_df = pd.read_sql_query(
    """
    SELECT
        app_name,
        app_id,
        records_fetched,
        new_records_inserted,
        duplicates_skipped,
        error_message
    FROM phase2_app_run_summary
    WHERE run_id = ?
    ORDER BY rowid
    """,
    conn,
    params=(RUN_ID,),
)

run_raw_rows = int(
    pd.read_sql_query(
        """
        SELECT COUNT(*) AS n
        FROM phase2_reviews_raw
        WHERE run_id = ?
        """,
        conn,
        params=(RUN_ID,),
    )["n"].iloc[0]
)

run_cleaned_rows = int(
    pd.read_sql_query(
        """
        SELECT COUNT(*) AS n
        FROM phase2_reviews_cleaned
        WHERE run_id = ?
        """,
        conn,
        params=(RUN_ID,),
    )["n"].iloc[0]
)

quality_flags_inserted_in_db = int(
    pd.read_sql_query(
        """
        SELECT COUNT(*) AS n
        FROM phase2_quality_flags
        WHERE run_id = ?
        """,
        conn,
        params=(RUN_ID,),
    )["n"].iloc[0]
)

raw_rows_after = int(
    pd.read_sql_query(
        """
        SELECT COUNT(*) AS n
        FROM phase2_reviews_raw
        """,
        conn,
    )["n"].iloc[0]
)

cleaned_rows_after = int(
    pd.read_sql_query(
        """
        SELECT COUNT(*) AS n
        FROM phase2_reviews_cleaned
        """,
        conn,
    )["n"].iloc[0]
)

review_rows_growth = int(
    raw_rows_after - raw_rows_before
)

db_size_after_mb = float(
    DB_PATH.stat().st_size / (1024 ** 2)
)

db_size_growth_mb = float(
    db_size_after_mb - db_size_before_mb
)

duplicate_identity_groups = int(
    pd.read_sql_query(
        """
        SELECT COUNT(*) AS n
        FROM (
            SELECT
                source,
                app_id,
                review_id,
                COUNT(*) AS row_count
            FROM phase2_reviews_raw
            GROUP BY
                source,
                app_id,
                review_id
            HAVING COUNT(*) > 1
        )
        """,
        conn,
    )["n"].iloc[0]
)

raw_without_cleaned = int(
    pd.read_sql_query(
        """
        SELECT COUNT(*) AS n
        FROM phase2_reviews_raw AS r
        LEFT JOIN phase2_reviews_cleaned AS c
            ON r.review_key = c.review_key
        WHERE c.review_key IS NULL
        """,
        conn,
    )["n"].iloc[0]
)

cleaned_without_raw = int(
    pd.read_sql_query(
        """
        SELECT COUNT(*) AS n
        FROM phase2_reviews_cleaned AS c
        LEFT JOIN phase2_reviews_raw AS r
            ON c.review_key = r.review_key
        WHERE r.review_key IS NULL
        """,
        conn,
    )["n"].iloc[0]
)

quality_flags_without_raw = int(
    pd.read_sql_query(
        """
        SELECT COUNT(*) AS n
        FROM phase2_quality_flags AS q
        LEFT JOIN phase2_reviews_raw AS r
            ON q.review_key = r.review_key
        WHERE r.review_key IS NULL
        """,
        conn,
    )["n"].iloc[0]
)

foreign_key_violations_df = pd.read_sql_query(
    "PRAGMA foreign_key_check",
    conn,
)

foreign_key_violation_count = int(
    len(foreign_key_violations_df)
)

database_records_fetched_total = int(
    database_app_summary_df[
        "records_fetched"
    ].sum()
)

database_new_insert_total = int(
    database_app_summary_df[
        "new_records_inserted"
    ].sum()
)

database_duplicates_total = int(
    database_app_summary_df[
        "duplicates_skipped"
    ].sum()
)

validation_checks = {
    "10_app_summary_rows": (
        app_summary_rows_in_db == 10
    ),
    "12000_reviews_fetched": (
        records_fetched_total == 12000
    ),
    "new_plus_duplicates_match_fetched": (
        new_records_inserted_total
        + duplicates_skipped_total
        == records_fetched_total
    ),
    "database_app_fetched_total_matches": (
        database_records_fetched_total
        == records_fetched_total
    ),
    "database_app_new_insert_total_matches": (
        database_new_insert_total
        == new_records_inserted_total
    ),
    "database_app_duplicate_total_matches": (
        database_duplicates_total
        == duplicates_skipped_total
    ),
    "run_raw_rows_match_new_inserts": (
        run_raw_rows
        == new_records_inserted_total
    ),
    "run_cleaned_rows_match_new_inserts": (
        run_cleaned_rows
        == new_records_inserted_total
    ),
    "database_growth_matches_new_inserts": (
        review_rows_growth
        == new_records_inserted_total
    ),
    "raw_and_cleaned_totals_match": (
        raw_rows_after
        == cleaned_rows_after
    ),
    "quality_flag_insert_count_matches": (
        quality_flags_inserted_in_db
        == quality_flags_inserted_total
    ),
    "no_duplicate_review_identities": (
        duplicate_identity_groups == 0
    ),
    "no_raw_rows_without_cleaned_rows": (
        raw_without_cleaned == 0
    ),
    "no_cleaned_rows_without_raw_rows": (
        cleaned_without_raw == 0
    ),
    "no_orphan_quality_flags": (
        quality_flags_without_raw == 0
    ),
    "no_foreign_key_violations": (
        foreign_key_violation_count == 0
    ),
    "no_missing_review_ids": (
        missing_review_id_total == 0
    ),
    "no_app_errors": (
        errors_total == 0
    ),
    "all_intervals_are_positive": (
        app_summary_df[
            "collection_interval_hours"
        ].gt(0).all()
    ),
}

failed_checks = [
    check_name
    for check_name, passed
    in validation_checks.items()
    if not passed
]

if failed_checks:
    raise ValueError(
        "Post-run validation failed: "
        f"{failed_checks}"
    )

run_finished_at = utc_now_iso()

cur.execute(
    """
    UPDATE phase2_ingestion_runs
    SET
        run_finished_at = ?,
        runtime_seconds = ?,
        status = ?,
        records_fetched_total = ?,
        new_records_inserted_total = ?,
        duplicates_skipped_total = ?,
        errors_total = ?,
        quality_flag_total = ?,
        quality_flags_inserted = ?,
        db_size_after_mb = ?,
        db_size_growth_mb = ?,
        review_rows_after = ?,
        review_rows_growth = ?
    WHERE run_id = ?
    """,
    (
        str(run_finished_at),
        float(processing_runtime_seconds),
        "completed",
        int(records_fetched_total),
        int(new_records_inserted_total),
        int(duplicates_skipped_total),
        int(errors_total),
        int(quality_flag_total),
        int(quality_flags_inserted_in_db),
        float(db_size_after_mb),
        float(db_size_growth_mb),
        int(raw_rows_after),
        int(review_rows_growth),
        str(RUN_ID),
    ),
)

conn.commit()

completed_run_df = pd.read_sql_query(
    """
    SELECT
        run_id,
        run_label,
        frequency_label,
        run_started_at,
        run_finished_at,
        runtime_seconds,
        status,
        records_fetched_total,
        new_records_inserted_total,
        duplicates_skipped_total,
        errors_total,
        quality_flag_total,
        quality_flags_inserted,
        review_rows_before,
        review_rows_after,
        review_rows_growth,
        db_size_before_mb,
        db_size_after_mb,
        db_size_growth_mb
    FROM phase2_ingestion_runs
    WHERE run_id = ?
    """,
    conn,
    params=(RUN_ID,),
)

if len(completed_run_df) != 1:
    raise ValueError(
        "The completed follow-up run record "
        "could not be verified."
    )

if completed_run_df[
    "status"
].iloc[0] != "completed":
    raise ValueError(
        "The follow-up run was not "
        "marked as completed."
    )

app_summary_df["new_insert_rate"] = (
    app_summary_df["new_records_inserted"]
    / app_summary_df["records_fetched"]
)

app_summary_df["duplicate_rate"] = (
    app_summary_df["duplicates_skipped"]
    / app_summary_df["records_fetched"]
)

app_summary_display_df = app_summary_df[
    [
        "app_name",
        "records_fetched",
        "new_records_inserted",
        "duplicates_skipped",
        "new_insert_rate",
        "duplicate_rate",
        "collection_interval_hours",
        "runtime_seconds",
        "min_review_date",
        "max_review_date",
        "error_message",
    ]
].copy()

app_summary_display_df[
    "new_insert_rate"
] = app_summary_display_df[
    "new_insert_rate"
].map(
    lambda value: f"{value:.2%}"
)

app_summary_display_df[
    "duplicate_rate"
] = app_summary_display_df[
    "duplicate_rate"
].map(
    lambda value: f"{value:.2%}"
)

app_summary_display_df[
    "collection_interval_hours"
] = app_summary_display_df[
    "collection_interval_hours"
].map(
    lambda value: f"{value:.2f}"
)

validation_df = pd.DataFrame(
    [
        {
            "validation_check": check_name,
            "passed": passed,
        }
        for check_name, passed
        in validation_checks.items()
    ]
)

print(
    "Run B follow-up collection "
    "completed and validated."
)
print("-" * 76)
print(
    "Records fetched:",
    f"{records_fetched_total:,}",
)
print(
    "New records inserted:",
    f"{new_records_inserted_total:,}",
)
print(
    "Duplicates skipped:",
    f"{duplicates_skipped_total:,}",
)
print(
    "New insert rate:",
    (
        f"{new_records_inserted_total / records_fetched_total:.2%}"
    ),
)
print(
    "Duplicate rate:",
    (
        f"{duplicates_skipped_total / records_fetched_total:.2%}"
    ),
)
print(
    "Processing runtime:",
    f"{processing_runtime_seconds:.2f} seconds",
)
print(
    "Wall-clock collection runtime:",
    f"{collection_runtime_seconds:.2f} seconds",
)
print(
    "Collection interval range:",
    (
        f"{interval_min_hours:.2f}–"
        f"{interval_max_hours:.2f} hours"
    ),
)
print(
    "Average collection interval:",
    f"{interval_mean_hours:.2f} hours",
)
print("Errors:", errors_total)
print(
    "Review rows before:",
    f"{raw_rows_before:,}",
)
print(
    "Review rows after:",
    f"{raw_rows_after:,}",
)
print(
    "Review row growth:",
    f"{review_rows_growth:,}",
)
print(
    "Database size before:",
    f"{db_size_before_mb:.2f} MB",
)
print(
    "Database size after:",
    f"{db_size_after_mb:.2f} MB",
)
print(
    "Database size growth:",
    f"{db_size_growth_mb:.2f} MB",
)
print(
    "Duplicate identity groups:",
    duplicate_identity_groups,
)
print(
    "Raw rows without cleaned rows:",
    raw_without_cleaned,
)
print(
    "Cleaned rows without raw rows:",
    cleaned_without_raw,
)
print(
    "Orphan quality flags:",
    quality_flags_without_raw,
)
print(
    "Foreign-key violations:",
    foreign_key_violation_count,
)

print("\nApp-level follow-up summary:")
display(app_summary_display_df)

print("\nDatabase validation checks:")
display(validation_df)

print("\nCompleted follow-up run record:")
display(completed_run_df)

Run B follow-up collection completed and validated.
----------------------------------------------------------------------------
Records fetched: 12,000
New records inserted: 3,753
Duplicates skipped: 8,247
New insert rate: 31.27%
Duplicate rate: 68.73%
Processing runtime: 9.46 seconds
Wall-clock collection runtime: 30.02 seconds
Collection interval range: 14.66–14.66 hours
Average collection interval: 14.66 hours
Errors: 0
Review rows before: 30,848
Review rows after: 34,601
Review row growth: 3,753
Database size before: 74.09 MB
Database size after: 84.68 MB
Database size growth: 10.59 MB
Duplicate identity groups: 0
Raw rows without cleaned rows: 0
Cleaned rows without raw rows: 0
Orphan quality flags: 0
Foreign-key violations: 0

App-level follow-up summary:


,app_name,records_fetched,new_records_inserted,duplicates_skipped,new_insert_rate,duplicate_rate,collection_interval_hours,runtime_seconds,min_review_date,max_review_date,error_message
0,YouTube,1200,994,206,82.83%,17.17%,14.66,1.929823,2026-07-12T19:33:26+00:00,2026-07-13T16:35:07+00:00,
1,TikTok,1200,406,794,33.83%,66.17%,14.66,0.788907,2026-07-11T16:43:34+00:00,2026-07-13T16:34:23+00:00,
2,Spotify,1200,456,744,38.00%,62.00%,14.66,0.770332,2026-07-12T02:40:14+00:00,2026-07-13T16:35:05+00:00,
3,Instagram,1200,1196,4,99.67%,0.33%,14.66,0.853803,2026-07-13T08:02:32+00:00,2026-07-13T16:36:13+00:00,
4,Uber,1200,309,891,25.75%,74.25%,14.66,0.783383,2026-07-10T16:56:33+00:00,2026-07-13T16:34:04+00:00,
5,DoorDash,1200,25,1175,2.08%,97.92%,14.66,0.927520,2026-07-03T21:12:32+00:00,2026-07-13T16:32:29+00:00,
6,Duolingo,1200,3,1197,0.25%,99.75%,14.66,0.972536,2026-07-06T18:18:00+00:00,2026-07-13T16:07:35+00:00,
7,Google Maps,1200,151,1049,12.58%,87.42%,14.66,0.842846,2026-07-08T05:13:39+00:00,2026-07-13T16:34:26+00:00,
8,Netflix,1200,152,1048,12.67%,87.33%,14.66,0.856286,2026-07-06T22:18:19+00:00,2026-07-13T16:31:09+00:00,
9,Reddit,1200,61,1139,5.08%,94.92%,14.66,0.739151,2026-07-03T16:12:54+00:00,2026-07-13T16:32:15+00:00,



Database validation checks:


,validation_check,passed
0,10_app_summary_rows,True
1,12000_reviews_fetched,True
2,new_plus_duplicates_match_fetched,True
3,database_app_fetched_total_matches,True
4,database_app_new_insert_total_matches,True
5,database_app_duplicate_total_matches,True
6,run_raw_rows_match_new_inserts,True
7,run_cleaned_rows_match_new_inserts,True
8,database_growth_matches_new_inserts,True
9,raw_and_cleaned_totals_match,True



Completed follow-up run record:


,run_id,run_label,frequency_label,run_started_at,run_finished_at,runtime_seconds,status,records_fetched_total,new_records_inserted_total,duplicates_skipped_total,errors_total,quality_flag_total,quality_flags_inserted,review_rows_before,review_rows_after,review_rows_growth,db_size_before_mb,db_size_after_mb,db_size_growth_mb
0,phase2_cadence_runB_followup_collection_202607...,phase2_cadence_runB_followup_collection,runB_followup_collection,2026-07-14T16:34:49.444978+00:00,2026-07-14T16:38:18.338912+00:00,9.464587,completed,12000,3753,8247,0,12600,12600,30848,34601,3753,74.085938,84.679688,10.59375


## 8. Validate whether newly inserted reviews were posted between the two Run B collections

A review ID can be new to the database even when the review itself was posted before the previous collection.

For each newly inserted review, this section compares the review timestamp with the exact app-specific collection boundaries:

- **Posted between collections:** posted after the app's Run B first-collection timestamp and no later than its follow-up collection timestamp
- **Older review surfaced later:** posted on or before the app's first-collection timestamp but first returned during the follow-up
- **Timestamp after fetch:** the review timestamp is later than the follow-up collection timestamp
- **Missing or unusable timestamp:** one or more required timestamps cannot be parsed

Using app-specific boundaries avoids treating the 10 sequential app requests as though they occurred at exactly the same time.

In [9]:
current_run_timing_df = pd.read_sql_query(
    """
    SELECT
        run_id,
        run_label,
        run_started_at,
        run_finished_at,
        status,
        new_records_inserted_total
    FROM phase2_ingestion_runs
    WHERE run_id = ?
    """,
    conn,
    params=(RUN_ID,),
)

if len(current_run_timing_df) != 1:
    raise ValueError(
        "The completed follow-up run record was not found."
    )

if current_run_timing_df["status"].iloc[0] != "completed":
    raise ValueError(
        "The follow-up collection must be completed "
        "before timestamp validation."
    )

expected_new_insert_total = int(
    current_run_timing_df[
        "new_records_inserted_total"
    ].iloc[0]
)

# Build the exact previous and current collection boundary for each app.
app_collection_boundary_df = app_summary_df[
    [
        "app_name",
        "app_id",
        "new_records_inserted",
        "previous_fetched_at",
        "current_fetched_at",
        "collection_interval_hours",
    ]
].copy()

if len(app_collection_boundary_df) != 10:
    raise ValueError(
        "Expected collection boundaries for 10 apps."
    )

if app_collection_boundary_df["app_id"].nunique() != 10:
    raise ValueError(
        "The collection-boundary table does not contain "
        "10 unique apps."
    )

app_collection_boundary_df[
    "previous_fetched_at_parsed"
] = pd.to_datetime(
    app_collection_boundary_df["previous_fetched_at"],
    utc=True,
    errors="coerce",
)

app_collection_boundary_df[
    "current_fetched_at_parsed"
] = pd.to_datetime(
    app_collection_boundary_df["current_fetched_at"],
    utc=True,
    errors="coerce",
)

if app_collection_boundary_df[
    "previous_fetched_at_parsed"
].isna().any():
    raise ValueError(
        "At least one previous collection timestamp "
        "could not be parsed."
    )

if app_collection_boundary_df[
    "current_fetched_at_parsed"
].isna().any():
    raise ValueError(
        "At least one current collection timestamp "
        "could not be parsed."
    )

app_collection_boundary_df[
    "recalculated_interval_hours"
] = (
    app_collection_boundary_df[
        "current_fetched_at_parsed"
    ]
    - app_collection_boundary_df[
        "previous_fetched_at_parsed"
    ]
).dt.total_seconds() / 3600

interval_difference = (
    app_collection_boundary_df[
        "recalculated_interval_hours"
    ]
    - app_collection_boundary_df[
        "collection_interval_hours"
    ]
).abs()

if interval_difference.gt(0.000001).any():
    raise ValueError(
        "At least one stored collection interval "
        "does not match its timestamps."
    )

# Read all records inserted during the follow-up collection.
new_review_timestamp_audit_df = pd.read_sql_query(
    """
    SELECT
        review_key,
        app_name,
        app_id,
        review_id,
        review_created_at,
        fetched_at,
        score,
        app_version,
        run_id
    FROM phase2_reviews_raw
    WHERE run_id = ?
    ORDER BY
        app_name,
        review_created_at,
        review_id
    """,
    conn,
    params=(RUN_ID,),
)

if len(new_review_timestamp_audit_df) != expected_new_insert_total:
    raise ValueError(
        "The timestamp-audit row count does not match "
        "the follow-up run's new insert count."
    )

new_review_timestamp_audit_df[
    "review_timestamp_parsed"
] = pd.to_datetime(
    new_review_timestamp_audit_df["review_created_at"],
    utc=True,
    errors="coerce",
)

new_review_timestamp_audit_df[
    "row_fetched_at_parsed"
] = pd.to_datetime(
    new_review_timestamp_audit_df["fetched_at"],
    utc=True,
    errors="coerce",
)

# Attach the app-specific previous and current boundaries.
new_review_timestamp_audit_df = (
    new_review_timestamp_audit_df.merge(
        app_collection_boundary_df[
            [
                "app_id",
                "previous_fetched_at_parsed",
                "current_fetched_at_parsed",
                "collection_interval_hours",
            ]
        ],
        on="app_id",
        how="left",
        validate="many_to_one",
    )
)

if new_review_timestamp_audit_df[
    "previous_fetched_at_parsed"
].isna().any():
    raise ValueError(
        "At least one audit row is missing its "
        "previous collection boundary."
    )

if new_review_timestamp_audit_df[
    "current_fetched_at_parsed"
].isna().any():
    raise ValueError(
        "At least one audit row is missing its "
        "current collection boundary."
    )

# Every newly inserted row for an app should contain the same
# fetched_at value used as that app's current collection boundary.
fetched_at_boundary_match = (
    new_review_timestamp_audit_df[
        "row_fetched_at_parsed"
    ]
    == new_review_timestamp_audit_df[
        "current_fetched_at_parsed"
    ]
)

if not fetched_at_boundary_match.all():
    raise ValueError(
        "At least one review fetched_at timestamp does not "
        "match its app's current collection boundary."
    )


def classify_followup_timestamp(row):
    review_ts = row["review_timestamp_parsed"]
    previous_ts = row["previous_fetched_at_parsed"]
    current_ts = row["current_fetched_at_parsed"]

    if (
        pd.isna(review_ts)
        or pd.isna(previous_ts)
        or pd.isna(current_ts)
    ):
        return "missing_or_unusable_timestamp"

    if review_ts <= previous_ts:
        return "older_review_surfaced_later"

    if review_ts <= current_ts:
        return "posted_between_collections"

    return "timestamp_after_fetch"


new_review_timestamp_audit_df[
    "timestamp_classification"
] = new_review_timestamp_audit_df.apply(
    classify_followup_timestamp,
    axis=1,
)

new_review_timestamp_audit_df[
    "posted_between_collections"
] = (
    new_review_timestamp_audit_df[
        "timestamp_classification"
    ]
    == "posted_between_collections"
).astype(int)

new_review_timestamp_audit_df[
    "older_review_surfaced_later"
] = (
    new_review_timestamp_audit_df[
        "timestamp_classification"
    ]
    == "older_review_surfaced_later"
).astype(int)

new_review_timestamp_audit_df[
    "timestamp_after_fetch"
] = (
    new_review_timestamp_audit_df[
        "timestamp_classification"
    ]
    == "timestamp_after_fetch"
).astype(int)

new_review_timestamp_audit_df[
    "missing_or_unusable_timestamp"
] = (
    new_review_timestamp_audit_df[
        "timestamp_classification"
    ]
    == "missing_or_unusable_timestamp"
).astype(int)


def timestamp_to_iso(value):
    if value is None or pd.isna(value):
        return None

    return pd.Timestamp(value).isoformat()


expected_new_inserts_by_app = (
    app_summary_df
    .set_index("app_id")["new_records_inserted"]
    .to_dict()
)

boundary_by_app = (
    app_collection_boundary_df
    .set_index("app_id")
    .to_dict("index")
)

timestamp_summary_rows = []

for _, app_row in app_config_df.iterrows():
    app_name = str(app_row["app_name"])
    app_id = str(app_row["app_id"])

    app_audit_df = new_review_timestamp_audit_df[
        new_review_timestamp_audit_df["app_id"] == app_id
    ].copy()

    valid_timestamp_df = app_audit_df[
        app_audit_df["review_timestamp_parsed"].notna()
    ]

    posted_between_df = app_audit_df[
        app_audit_df["timestamp_classification"]
        == "posted_between_collections"
    ]

    audited_new_inserts = int(len(app_audit_df))

    posted_between_count = int(
        app_audit_df[
            "posted_between_collections"
        ].sum()
    )

    older_surfaced_count = int(
        app_audit_df[
            "older_review_surfaced_later"
        ].sum()
    )

    timestamp_after_fetch_count = int(
        app_audit_df[
            "timestamp_after_fetch"
        ].sum()
    )

    missing_timestamp_count = int(
        app_audit_df[
            "missing_or_unusable_timestamp"
        ].sum()
    )

    boundary = boundary_by_app[app_id]

    timestamp_summary_rows.append(
        {
            "app_name": app_name,
            "app_id": app_id,
            "expected_new_inserts": int(
                expected_new_inserts_by_app.get(
                    app_id,
                    0,
                )
            ),
            "audited_new_inserts": audited_new_inserts,
            "previous_fetched_at": timestamp_to_iso(
                boundary[
                    "previous_fetched_at_parsed"
                ]
            ),
            "current_fetched_at": timestamp_to_iso(
                boundary[
                    "current_fetched_at_parsed"
                ]
            ),
            "collection_interval_hours": float(
                boundary[
                    "collection_interval_hours"
                ]
            ),
            "posted_between_collections_count": (
                posted_between_count
            ),
            "older_reviews_surfaced_later_count": (
                older_surfaced_count
            ),
            "timestamp_after_fetch_count": (
                timestamp_after_fetch_count
            ),
            "missing_or_unusable_timestamp_count": (
                missing_timestamp_count
            ),
            "posted_between_collections_rate": (
                posted_between_count / audited_new_inserts
                if audited_new_inserts > 0
                else 0.0
            ),
            "older_surfaced_rate": (
                older_surfaced_count / audited_new_inserts
                if audited_new_inserts > 0
                else 0.0
            ),
            "new_insert_timestamp_min": (
                timestamp_to_iso(
                    valid_timestamp_df[
                        "review_timestamp_parsed"
                    ].min()
                )
                if len(valid_timestamp_df) > 0
                else None
            ),
            "new_insert_timestamp_max": (
                timestamp_to_iso(
                    valid_timestamp_df[
                        "review_timestamp_parsed"
                    ].max()
                )
                if len(valid_timestamp_df) > 0
                else None
            ),
            "posted_between_timestamp_min": (
                timestamp_to_iso(
                    posted_between_df[
                        "review_timestamp_parsed"
                    ].min()
                )
                if len(posted_between_df) > 0
                else None
            ),
            "posted_between_timestamp_max": (
                timestamp_to_iso(
                    posted_between_df[
                        "review_timestamp_parsed"
                    ].max()
                )
                if len(posted_between_df) > 0
                else None
            ),
        }
    )

timestamp_summary_df = pd.DataFrame(
    timestamp_summary_rows
)

timestamp_summary_df[
    "classification_total"
] = (
    timestamp_summary_df[
        "posted_between_collections_count"
    ]
    + timestamp_summary_df[
        "older_reviews_surfaced_later_count"
    ]
    + timestamp_summary_df[
        "timestamp_after_fetch_count"
    ]
    + timestamp_summary_df[
        "missing_or_unusable_timestamp_count"
    ]
)

timestamp_validation_checks = {
    "audit_rows_match_followup_new_inserts": (
        len(new_review_timestamp_audit_df)
        == expected_new_insert_total
    ),
    "app_level_audit_counts_match_new_inserts": (
        timestamp_summary_df[
            "audited_new_inserts"
        ].equals(
            timestamp_summary_df[
                "expected_new_inserts"
            ]
        )
    ),
    "all_timestamp_classes_reconcile": (
        timestamp_summary_df[
            "classification_total"
        ].equals(
            timestamp_summary_df[
                "audited_new_inserts"
            ]
        )
    ),
    "all_10_apps_in_timestamp_summary": (
        len(timestamp_summary_df) == 10
    ),
    "all_rows_have_current_run_id": (
        new_review_timestamp_audit_df[
            "run_id"
        ].eq(RUN_ID).all()
    ),
    "all_rows_match_current_fetch_boundary": (
        fetched_at_boundary_match.all()
    ),
    "all_collection_intervals_positive": (
        timestamp_summary_df[
            "collection_interval_hours"
        ].gt(0).all()
    ),
}

failed_timestamp_checks = [
    check_name
    for check_name, passed
    in timestamp_validation_checks.items()
    if not passed
]

if failed_timestamp_checks:
    raise ValueError(
        "Follow-up timestamp validation failed: "
        f"{failed_timestamp_checks}"
    )

posted_between_collections_total = int(
    timestamp_summary_df[
        "posted_between_collections_count"
    ].sum()
)

older_surfaced_total = int(
    timestamp_summary_df[
        "older_reviews_surfaced_later_count"
    ].sum()
)

timestamp_after_fetch_total = int(
    timestamp_summary_df[
        "timestamp_after_fetch_count"
    ].sum()
)

missing_timestamp_total = int(
    timestamp_summary_df[
        "missing_or_unusable_timestamp_count"
    ].sum()
)

timestamp_validation_df = pd.DataFrame(
    [
        {
            "validation_check": check_name,
            "passed": passed,
        }
        for check_name, passed
        in timestamp_validation_checks.items()
    ]
)

timestamp_summary_display_df = (
    timestamp_summary_df.copy()
)

timestamp_summary_display_df[
    "posted_between_collections_rate"
] = timestamp_summary_display_df[
    "posted_between_collections_rate"
].map(
    lambda value: f"{value:.2%}"
)

timestamp_summary_display_df[
    "older_surfaced_rate"
] = timestamp_summary_display_df[
    "older_surfaced_rate"
].map(
    lambda value: f"{value:.2%}"
)

timestamp_summary_display_df[
    "collection_interval_hours"
] = timestamp_summary_display_df[
    "collection_interval_hours"
].map(
    lambda value: f"{value:.2f}"
)

print(
    "Run B follow-up timestamp validation passed."
)
print("-" * 82)
print(
    "New inserts audited:",
    f"{len(new_review_timestamp_audit_df):,}",
)
print(
    "Posted between collections:",
    f"{posted_between_collections_total:,}",
)
print(
    "Older reviews surfaced later:",
    f"{older_surfaced_total:,}",
)
print(
    "Timestamps after fetch:",
    f"{timestamp_after_fetch_total:,}",
)
print(
    "Missing or unusable timestamps:",
    f"{missing_timestamp_total:,}",
)
print(
    "Overall posted-between rate:",
    (
        f"{posted_between_collections_total / expected_new_insert_total:.2%}"
        if expected_new_insert_total > 0
        else "0.00%"
    ),
)

print("\nApp-level follow-up timestamp summary:")
display(
    timestamp_summary_display_df[
        [
            "app_name",
            "audited_new_inserts",
            "collection_interval_hours",
            "posted_between_collections_count",
            "older_reviews_surfaced_later_count",
            "timestamp_after_fetch_count",
            "missing_or_unusable_timestamp_count",
            "posted_between_collections_rate",
            "older_surfaced_rate",
            "new_insert_timestamp_min",
            "new_insert_timestamp_max",
            "posted_between_timestamp_min",
            "posted_between_timestamp_max",
        ]
    ]
)

print("\nTimestamp validation checks:")
display(timestamp_validation_df)

Run B follow-up timestamp validation passed.
----------------------------------------------------------------------------------
New inserts audited: 3,753
Posted between collections: 0
Older reviews surfaced later: 3,753
Timestamps after fetch: 0
Missing or unusable timestamps: 0
Overall posted-between rate: 0.00%

App-level follow-up timestamp summary:


,app_name,audited_new_inserts,collection_interval_hours,posted_between_collections_count,older_reviews_surfaced_later_count,timestamp_after_fetch_count,missing_or_unusable_timestamp_count,posted_between_collections_rate,older_surfaced_rate,new_insert_timestamp_min,new_insert_timestamp_max,posted_between_timestamp_min,posted_between_timestamp_max
0,YouTube,994,14.66,0,994,0,0,0.00%,100.00%,2026-07-13T01:59:02+00:00,2026-07-13T16:35:07+00:00,None,None
1,TikTok,406,14.66,0,406,0,0,0.00%,100.00%,2026-07-13T01:59:51+00:00,2026-07-13T16:34:23+00:00,None,None
2,Spotify,456,14.66,0,456,0,0,0.00%,100.00%,2026-07-13T01:56:51+00:00,2026-07-13T16:35:05+00:00,None,None
3,Instagram,1196,14.66,0,1196,0,0,0.00%,100.00%,2026-07-13T08:02:32+00:00,2026-07-13T16:36:13+00:00,None,None
4,Uber,309,14.66,0,309,0,0,0.00%,100.00%,2026-07-13T02:01:50+00:00,2026-07-13T16:34:04+00:00,None,None
5,DoorDash,25,14.66,0,25,0,0,0.00%,100.00%,2026-07-13T01:58:41+00:00,2026-07-13T16:32:29+00:00,None,None
6,Duolingo,3,14.66,0,3,0,0,0.00%,100.00%,2026-07-13T08:45:54+00:00,2026-07-13T16:07:35+00:00,None,None
7,Google Maps,151,14.66,0,151,0,0,0.00%,100.00%,2026-07-13T02:06:35+00:00,2026-07-13T16:34:26+00:00,None,None
8,Netflix,152,14.66,0,152,0,0,0.00%,100.00%,2026-07-13T01:58:20+00:00,2026-07-13T16:31:09+00:00,None,None
9,Reddit,61,14.66,0,61,0,0,0.00%,100.00%,2026-07-13T02:03:14+00:00,2026-07-13T16:32:15+00:00,None,None



Timestamp validation checks:


,validation_check,passed
0,audit_rows_match_followup_new_inserts,True
1,app_level_audit_counts_match_new_inserts,True
2,all_timestamp_classes_reconcile,True
3,all_10_apps_in_timestamp_summary,True
4,all_rows_have_current_run_id,True
5,all_rows_match_current_fetch_boundary,True
6,all_collection_intervals_positive,True


## 9. Diagnose source freshness lag and returned-window movement

The timestamp audit shows that all follow-up inserts were posted before the previous collection boundary.

To interpret this result correctly, this section compares:

- the latest review timestamp available at the first collection
- the first-collection fetch timestamp
- the latest review timestamp returned in the follow-up
- the follow-up fetch timestamp
- the movement of the returned review window between collections

This helps distinguish a consistent source-availability delay from isolated older reviews appearing because of ordering or returned-window changes.

The analysis does not change the timestamp classification. It provides additional context for the final cadence recommendation.

In [10]:
previous_window_df = (
    pre_run_app_snapshot_df[
        [
            "app_id",
            "latest_review_timestamp",
        ]
    ]
    .rename(
        columns={
            "latest_review_timestamp": (
                "previous_window_latest_review_at"
            )
        }
    )
)

current_window_df = (
    app_summary_df[
        [
            "app_id",
            "max_review_date",
        ]
    ]
    .rename(
        columns={
            "max_review_date": (
                "followup_window_latest_review_at"
            )
        }
    )
)

new_insert_max_df = (
    timestamp_summary_df[
        [
            "app_id",
            "new_insert_timestamp_max",
        ]
    ]
    .copy()
)

source_freshness_df = (
    app_config_df[
        [
            "app_name",
            "app_id",
        ]
    ]
    .merge(
        previous_window_df,
        on="app_id",
        how="left",
        validate="one_to_one",
    )
    .merge(
        app_collection_boundary_df[
            [
                "app_id",
                "previous_fetched_at",
                "current_fetched_at",
                "collection_interval_hours",
            ]
        ],
        on="app_id",
        how="left",
        validate="one_to_one",
    )
    .merge(
        current_window_df,
        on="app_id",
        how="left",
        validate="one_to_one",
    )
    .merge(
        new_insert_max_df,
        on="app_id",
        how="left",
        validate="one_to_one",
    )
)

timestamp_columns = [
    "previous_window_latest_review_at",
    "previous_fetched_at",
    "followup_window_latest_review_at",
    "current_fetched_at",
    "new_insert_timestamp_max",
]

for column in timestamp_columns:
    source_freshness_df[
        f"{column}_parsed"
    ] = pd.to_datetime(
        source_freshness_df[column],
        utc=True,
        errors="coerce",
    )

parsed_columns = [
    f"{column}_parsed"
    for column in timestamp_columns
]

if len(source_freshness_df) != 10:
    raise ValueError(
        "Expected source-freshness results for 10 apps."
    )

if source_freshness_df["app_id"].nunique() != 10:
    raise ValueError(
        "The source-freshness table does not "
        "contain 10 unique apps."
    )

if source_freshness_df[
    parsed_columns
].isna().any().any():
    raise ValueError(
        "At least one source-freshness timestamp "
        "could not be parsed."
    )

source_freshness_df[
    "previous_source_lag_hours"
] = (
    source_freshness_df[
        "previous_fetched_at_parsed"
    ]
    - source_freshness_df[
        "previous_window_latest_review_at_parsed"
    ]
).dt.total_seconds() / 3600

source_freshness_df[
    "followup_source_lag_hours"
] = (
    source_freshness_df[
        "current_fetched_at_parsed"
    ]
    - source_freshness_df[
        "followup_window_latest_review_at_parsed"
    ]
).dt.total_seconds() / 3600

source_freshness_df[
    "returned_window_advance_hours"
] = (
    source_freshness_df[
        "followup_window_latest_review_at_parsed"
    ]
    - source_freshness_df[
        "previous_window_latest_review_at_parsed"
    ]
).dt.total_seconds() / 3600

source_freshness_df[
    "window_advance_minus_interval_hours"
] = (
    source_freshness_df[
        "returned_window_advance_hours"
    ]
    - source_freshness_df[
        "collection_interval_hours"
    ]
)

source_freshness_df[
    "source_lag_change_hours"
] = (
    source_freshness_df[
        "followup_source_lag_hours"
    ]
    - source_freshness_df[
        "previous_source_lag_hours"
    ]
)

source_freshness_df[
    "followup_latest_is_new_insert"
] = (
    source_freshness_df[
        "followup_window_latest_review_at_parsed"
    ]
    == source_freshness_df[
        "new_insert_timestamp_max_parsed"
    ]
)

source_freshness_checks = {
    "all_10_apps_present": (
        len(source_freshness_df) == 10
    ),
    "all_timestamps_parsed": (
        not source_freshness_df[
            parsed_columns
        ].isna().any().any()
    ),
    "previous_latest_not_after_previous_fetch": (
        (
            source_freshness_df[
                "previous_window_latest_review_at_parsed"
            ]
            <= source_freshness_df[
                "previous_fetched_at_parsed"
            ]
        ).all()
    ),
    "followup_latest_not_after_followup_fetch": (
        (
            source_freshness_df[
                "followup_window_latest_review_at_parsed"
            ]
            <= source_freshness_df[
                "current_fetched_at_parsed"
            ]
        ).all()
    ),
    "all_source_lags_nonnegative": (
        source_freshness_df[
            "previous_source_lag_hours"
        ].ge(0).all()
        and source_freshness_df[
            "followup_source_lag_hours"
        ].ge(0).all()
    ),
    "returned_window_did_not_move_backward": (
        source_freshness_df[
            "returned_window_advance_hours"
        ].ge(0).all()
    ),
    "timestamp_audit_found_zero_posted_between": (
        int(
            timestamp_summary_df[
                "posted_between_collections_count"
            ].sum()
        )
        == 0
    ),
}

failed_source_freshness_checks = [
    check_name
    for check_name, passed
    in source_freshness_checks.items()
    if not passed
]

if failed_source_freshness_checks:
    raise ValueError(
        "Source-freshness validation failed: "
        f"{failed_source_freshness_checks}"
    )

previous_lag_median = float(
    source_freshness_df[
        "previous_source_lag_hours"
    ].median()
)

followup_lag_median = float(
    source_freshness_df[
        "followup_source_lag_hours"
    ].median()
)

followup_lag_min = float(
    source_freshness_df[
        "followup_source_lag_hours"
    ].min()
)

followup_lag_max = float(
    source_freshness_df[
        "followup_source_lag_hours"
    ].max()
)

window_advance_median = float(
    source_freshness_df[
        "returned_window_advance_hours"
    ].median()
)

apps_with_lag_over_20_hours = int(
    source_freshness_df[
        "followup_source_lag_hours"
    ].ge(20).sum()
)

apps_with_window_advance = int(
    source_freshness_df[
        "returned_window_advance_hours"
    ].gt(0).sum()
)

apps_latest_timestamp_is_new_insert = int(
    source_freshness_df[
        "followup_latest_is_new_insert"
    ].sum()
)

source_freshness_validation_df = pd.DataFrame(
    [
        {
            "validation_check": check_name,
            "passed": passed,
        }
        for check_name, passed
        in source_freshness_checks.items()
    ]
)

source_freshness_display_df = (
    source_freshness_df[
        [
            "app_name",
            "previous_window_latest_review_at",
            "previous_fetched_at",
            "previous_source_lag_hours",
            "followup_window_latest_review_at",
            "current_fetched_at",
            "followup_source_lag_hours",
            "collection_interval_hours",
            "returned_window_advance_hours",
            "window_advance_minus_interval_hours",
            "followup_latest_is_new_insert",
        ]
    ]
    .copy()
)

numeric_columns = [
    "previous_source_lag_hours",
    "followup_source_lag_hours",
    "collection_interval_hours",
    "returned_window_advance_hours",
    "window_advance_minus_interval_hours",
]

for column in numeric_columns:
    source_freshness_display_df[column] = (
        source_freshness_display_df[column]
        .round(2)
    )

print(
    "Source freshness and returned-window "
    "diagnostic passed."
)
print("-" * 82)
print(
    "Median previous source lag:",
    f"{previous_lag_median:.2f} hours",
)
print(
    "Median follow-up source lag:",
    f"{followup_lag_median:.2f} hours",
)
print(
    "Follow-up source lag range:",
    f"{followup_lag_min:.2f}–"
    f"{followup_lag_max:.2f} hours",
)
print(
    "Median returned-window advance:",
    f"{window_advance_median:.2f} hours",
)
print(
    "Apps with at least 20 hours of source lag:",
    apps_with_lag_over_20_hours,
)
print(
    "Apps whose returned window moved forward:",
    apps_with_window_advance,
)
print(
    "Apps whose latest returned timestamp "
    "was a newly inserted review:",
    apps_latest_timestamp_is_new_insert,
)

print("\nApp-level source freshness diagnostic:")
display(source_freshness_display_df)

print("\nSource-freshness validation checks:")
display(source_freshness_validation_df)

Source freshness and returned-window diagnostic passed.
----------------------------------------------------------------------------------
Median previous source lag: 24.11 hours
Median follow-up source lag: 24.04 hours
Follow-up source lag range: 24.01–24.49 hours
Median returned-window advance: 14.72 hours
Apps with at least 20 hours of source lag: 10
Apps whose returned window moved forward: 10
Apps whose latest returned timestamp was a newly inserted review: 10

App-level source freshness diagnostic:


,app_name,previous_window_latest_review_at,previous_fetched_at,previous_source_lag_hours,followup_window_latest_review_at,current_fetched_at,followup_source_lag_hours,collection_interval_hours,returned_window_advance_hours,window_advance_minus_interval_hours,followup_latest_is_new_insert
0,YouTube,2026-07-13T01:56:32+00:00,2026-07-14T01:56:37.779729+00:00,24.00,2026-07-13T16:35:07+00:00,2026-07-14T16:36:31.000320+00:00,24.02,14.66,14.64,-0.02,True
1,TikTok,2026-07-13T01:54:48+00:00,2026-07-14T01:56:42.049396+00:00,24.03,2026-07-13T16:34:23+00:00,2026-07-14T16:36:35.028756+00:00,24.04,14.66,14.66,-0.00,True
2,Spotify,2026-07-13T01:56:37+00:00,2026-07-14T01:56:45.466435+00:00,24.00,2026-07-13T16:35:05+00:00,2026-07-14T16:36:37.862502+00:00,24.03,14.66,14.64,-0.02,True
3,Instagram,2026-07-13T01:56:17+00:00,2026-07-14T01:56:48.271551+00:00,24.01,2026-07-13T16:36:13+00:00,2026-07-14T16:36:40.693143+00:00,24.01,14.66,14.67,0.00,True
4,Uber,2026-07-13T01:46:36+00:00,2026-07-14T01:56:51.086121+00:00,24.17,2026-07-13T16:34:04+00:00,2026-07-14T16:36:43.617047+00:00,24.04,14.66,14.79,0.13,True
5,DoorDash,2026-07-13T01:45:26+00:00,2026-07-14T01:56:54.049406+00:00,24.19,2026-07-13T16:32:29+00:00,2026-07-14T16:36:46.455557+00:00,24.07,14.66,14.78,0.12,True
6,Duolingo,2026-07-12T12:05:22+00:00,2026-07-14T01:56:57.030308+00:00,37.86,2026-07-13T16:07:35+00:00,2026-07-14T16:36:49.425230+00:00,24.49,14.66,28.04,13.37,True
7,Google Maps,2026-07-13T01:48:56+00:00,2026-07-14T01:56:59.944051+00:00,24.13,2026-07-13T16:34:26+00:00,2026-07-14T16:36:52.434501+00:00,24.04,14.66,14.76,0.09,True
8,Netflix,2026-07-13T01:26:29+00:00,2026-07-14T01:57:02.795277+00:00,24.51,2026-07-13T16:31:09+00:00,2026-07-14T16:36:55.322483+00:00,24.10,14.66,15.08,0.41,True
9,Reddit,2026-07-13T01:51:50+00:00,2026-07-14T01:57:05.791189+00:00,24.09,2026-07-13T16:32:15+00:00,2026-07-14T16:36:58.231573+00:00,24.08,14.66,14.67,0.01,True



Source-freshness validation checks:


,validation_check,passed
0,all_10_apps_present,True
1,all_timestamps_parsed,True
2,previous_latest_not_after_previous_fetch,True
3,followup_latest_not_after_followup_fetch,True
4,all_source_lags_nonnegative,True
5,returned_window_did_not_move_backward,True
6,timestamp_audit_found_zero_posted_between,True


## 10. Normalize runtime measurement for cadence comparison

The earlier Cadence Run A stored wall-clock collection runtime, including the fixed delay between app requests.

The two Run B collections separately recorded:

- app-processing runtime
- wall-clock collection runtime

To keep the run-level cadence comparison consistent, the database `runtime_seconds` field for both Run B collections is updated to use wall-clock collection runtime.

The processing-only runtime is preserved separately in the comparison outputs. This correction changes only the runtime measurement definition and does not change any review, duplicate, timestamp, or database-growth result.

In [11]:
first_collection_processing_seconds = float(
    checkpoint_metadata[
        "processing_runtime_seconds"
    ]
)

first_collection_wall_clock_seconds = float(
    checkpoint_metadata[
        "wall_clock_collection_seconds"
    ]
)

followup_processing_seconds = float(
    processing_runtime_seconds
)

followup_wall_clock_seconds = float(
    collection_runtime_seconds
)

runtime_values = [
    first_collection_processing_seconds,
    first_collection_wall_clock_seconds,
    followup_processing_seconds,
    followup_wall_clock_seconds,
]

if any(
    value <= 0
    for value in runtime_values
):
    raise ValueError(
        "At least one runtime value is not positive."
    )

runtime_records_before_df = pd.read_sql_query(
    """
    SELECT
        run_id,
        run_label,
        status,
        runtime_seconds
    FROM phase2_ingestion_runs
    WHERE run_id IN (?, ?)
    ORDER BY run_started_at
    """,
    conn,
    params=(
        PREVIOUS_RUN_ID,
        RUN_ID,
    ),
)

if len(runtime_records_before_df) != 2:
    raise ValueError(
        "Expected exactly two Run B records "
        "for runtime normalization."
    )

expected_runtime_labels = {
    "phase2_cadence_runB_first_collection",
    "phase2_cadence_runB_followup_collection",
}

if set(
    runtime_records_before_df["run_label"]
) != expected_runtime_labels:
    raise ValueError(
        "The runtime records do not match "
        "the two Run B collections."
    )

if not runtime_records_before_df[
    "status"
].eq("completed").all():
    raise ValueError(
        "Both Run B collections must be completed "
        "before runtime normalization."
    )

cur.execute(
    """
    UPDATE phase2_ingestion_runs
    SET runtime_seconds = ?
    WHERE run_id = ?
    """,
    (
        first_collection_wall_clock_seconds,
        PREVIOUS_RUN_ID,
    ),
)

cur.execute(
    """
    UPDATE phase2_ingestion_runs
    SET runtime_seconds = ?
    WHERE run_id = ?
    """,
    (
        followup_wall_clock_seconds,
        RUN_ID,
    ),
)

conn.commit()

runtime_records_after_df = pd.read_sql_query(
    """
    SELECT
        run_id,
        run_label,
        run_started_at,
        run_finished_at,
        runtime_seconds,
        status
    FROM phase2_ingestion_runs
    WHERE run_id IN (?, ?)
    ORDER BY run_started_at
    """,
    conn,
    params=(
        PREVIOUS_RUN_ID,
        RUN_ID,
    ),
)

first_runtime_after = float(
    runtime_records_after_df.loc[
        runtime_records_after_df["run_id"]
        == PREVIOUS_RUN_ID,
        "runtime_seconds",
    ].iloc[0]
)

followup_runtime_after = float(
    runtime_records_after_df.loc[
        runtime_records_after_df["run_id"]
        == RUN_ID,
        "runtime_seconds",
    ].iloc[0]
)

runtime_checks = {
    "two_runB_records_found": (
        len(runtime_records_after_df) == 2
    ),
    "both_runB_records_completed": (
        runtime_records_after_df[
            "status"
        ].eq("completed").all()
    ),
    "first_collection_wall_clock_saved": (
        abs(
            first_runtime_after
            - first_collection_wall_clock_seconds
        )
        < 0.000001
    ),
    "followup_wall_clock_saved": (
        abs(
            followup_runtime_after
            - followup_wall_clock_seconds
        )
        < 0.000001
    ),
    "processing_runtime_preserved_separately": (
        first_collection_processing_seconds > 0
        and followup_processing_seconds > 0
    ),
}

failed_runtime_checks = [
    check_name
    for check_name, passed
    in runtime_checks.items()
    if not passed
]

if failed_runtime_checks:
    raise ValueError(
        "Runtime normalization failed: "
        f"{failed_runtime_checks}"
    )

runtime_metric_df = pd.DataFrame(
    [
        {
            "run_id": PREVIOUS_RUN_ID,
            "run_label": (
                "phase2_cadence_runB_first_collection"
            ),
            "processing_runtime_seconds": (
                first_collection_processing_seconds
            ),
            "wall_clock_collection_seconds": (
                first_collection_wall_clock_seconds
            ),
            "database_runtime_seconds": (
                first_runtime_after
            ),
        },
        {
            "run_id": RUN_ID,
            "run_label": (
                "phase2_cadence_runB_followup_collection"
            ),
            "processing_runtime_seconds": (
                followup_processing_seconds
            ),
            "wall_clock_collection_seconds": (
                followup_wall_clock_seconds
            ),
            "database_runtime_seconds": (
                followup_runtime_after
            ),
        },
    ]
)

runtime_validation_df = pd.DataFrame(
    [
        {
            "validation_check": check_name,
            "passed": passed,
        }
        for check_name, passed
        in runtime_checks.items()
    ]
)

print("Run B runtime measurement normalized.")
print("-" * 76)
print(
    "First collection processing runtime:",
    f"{first_collection_processing_seconds:.2f} seconds",
)
print(
    "First collection wall-clock runtime:",
    f"{first_collection_wall_clock_seconds:.2f} seconds",
)
print(
    "Follow-up processing runtime:",
    f"{followup_processing_seconds:.2f} seconds",
)
print(
    "Follow-up wall-clock runtime:",
    f"{followup_wall_clock_seconds:.2f} seconds",
)

print("\nNormalized runtime records:")
display(runtime_metric_df)

print("\nRuntime validation checks:")
display(runtime_validation_df)

Run B runtime measurement normalized.
----------------------------------------------------------------------------
First collection processing runtime: 10.09 seconds
First collection wall-clock runtime: 30.76 seconds
Follow-up processing runtime: 9.46 seconds
Follow-up wall-clock runtime: 30.02 seconds

Normalized runtime records:


,run_id,run_label,processing_runtime_seconds,wall_clock_collection_seconds,database_runtime_seconds
0,phase2_cadence_runB_first_collection_20260714_...,phase2_cadence_runB_first_collection,10.093505,30.761982,30.761982
1,phase2_cadence_runB_followup_collection_202607...,phase2_cadence_runB_followup_collection,9.464587,30.016523,30.016523



Runtime validation checks:


,validation_check,passed
0,two_runB_records_found,True
1,both_runB_records_completed,True
2,first_collection_wall_clock_saved,True
3,followup_wall_clock_saved,True
4,processing_runtime_preserved_separately,True


## 11. Reconstruct the Cadence Run A timestamp audit using app-specific boundaries

To compare Cadence Run A and Run B on the same basis, this section reconstructs the earlier Run A timestamp analysis directly from the database.

For each app, the review timestamps inserted during Cadence Run A are compared with:

- that app's collection timestamp during the preceding Day 3 run
- that app's collection timestamp during Cadence Run A

Each Run A insert is classified as:

- **Posted between collections**
- **Older review surfaced later**
- **Timestamp after fetch**
- **Missing or unusable timestamp**

This uses the same app-specific logic applied to the Run B follow-up and avoids relying only on overall run start and finish times.

In [12]:
DAY3_RUN_LABEL = (
    "phase2_day3_controlled_repeated_run"
)

RUNA_RUN_LABEL = (
    "phase2_cadence_runA_twice_daily_test"
)

cadence_A_run_records_df = pd.read_sql_query(
    """
    SELECT
        run_id,
        run_label,
        frequency_label,
        target_reviews_per_app,
        app_count,
        run_started_at,
        run_finished_at,
        runtime_seconds,
        status,
        records_fetched_total,
        new_records_inserted_total,
        duplicates_skipped_total,
        review_rows_before,
        review_rows_after,
        review_rows_growth,
        db_size_before_mb,
        db_size_after_mb,
        db_size_growth_mb,
        errors_total
    FROM phase2_ingestion_runs
    WHERE run_label IN (?, ?)
    ORDER BY run_started_at
    """,
    conn,
    params=(
        DAY3_RUN_LABEL,
        RUNA_RUN_LABEL,
    ),
)

if len(cadence_A_run_records_df) != 2:
    raise ValueError(
        "Expected exactly the Day 3 and Cadence Run A records."
    )

if set(
    cadence_A_run_records_df["run_label"]
) != {
    DAY3_RUN_LABEL,
    RUNA_RUN_LABEL,
}:
    raise ValueError(
        "The expected Day 3 and Run A labels were not found."
    )

if not cadence_A_run_records_df[
    "status"
].eq("completed").all():
    raise ValueError(
        "Day 3 and Cadence Run A must both be completed."
    )

if not cadence_A_run_records_df[
    "target_reviews_per_app"
].eq(1200).all():
    raise ValueError(
        "Day 3 or Run A used a different review target."
    )

if not cadence_A_run_records_df[
    "app_count"
].eq(10).all():
    raise ValueError(
        "Day 3 or Run A used a different app count."
    )

DAY3_RUN_ID = str(
    cadence_A_run_records_df.loc[
        cadence_A_run_records_df["run_label"]
        == DAY3_RUN_LABEL,
        "run_id",
    ].iloc[0]
)

RUNA_RUN_ID = str(
    cadence_A_run_records_df.loc[
        cadence_A_run_records_df["run_label"]
        == RUNA_RUN_LABEL,
        "run_id",
    ].iloc[0]
)


def read_app_fetch_boundaries(
    run_id,
    timestamp_column_name,
    inserted_column_name,
):
    boundary_df = pd.read_sql_query(
        """
        SELECT
            a.app_name,
            a.app_id,
            COUNT(r.review_key) AS inserted_rows,
            COUNT(
                DISTINCT r.fetched_at
            ) AS distinct_fetch_timestamps,
            MIN(r.fetched_at) AS fetched_at_min,
            MAX(r.fetched_at) AS fetched_at_max
        FROM phase2_apps AS a
        LEFT JOIN phase2_reviews_raw AS r
            ON a.app_id = r.app_id
            AND r.run_id = ?
        GROUP BY
            a.app_name,
            a.app_id
        ORDER BY a.rowid
        """,
        conn,
        params=(run_id,),
    )

    if len(boundary_df) != 10:
        raise ValueError(
            f"Expected 10 app boundaries for run {run_id}."
        )

    if not boundary_df[
        "inserted_rows"
    ].gt(0).all():
        raise ValueError(
            f"At least one app has no inserted rows in run {run_id}."
        )

    if not boundary_df[
        "distinct_fetch_timestamps"
    ].eq(1).all():
        raise ValueError(
            f"At least one app has more than one fetch timestamp "
            f"in run {run_id}."
        )

    if not (
        boundary_df["fetched_at_min"]
        == boundary_df["fetched_at_max"]
    ).all():
        raise ValueError(
            f"Fetch timestamp minimum and maximum do not match "
            f"for run {run_id}."
        )

    boundary_df = boundary_df.rename(
        columns={
            "inserted_rows": inserted_column_name,
            "fetched_at_min": timestamp_column_name,
        }
    )

    return boundary_df[
        [
            "app_name",
            "app_id",
            inserted_column_name,
            timestamp_column_name,
        ]
    ].copy()


day3_app_boundary_df = read_app_fetch_boundaries(
    DAY3_RUN_ID,
    "previous_fetched_at",
    "day3_inserted_rows",
)

runA_app_boundary_df = read_app_fetch_boundaries(
    RUNA_RUN_ID,
    "current_fetched_at",
    "runA_inserted_rows",
)

runA_collection_boundary_df = (
    day3_app_boundary_df.merge(
        runA_app_boundary_df[
            [
                "app_id",
                "runA_inserted_rows",
                "current_fetched_at",
            ]
        ],
        on="app_id",
        how="inner",
        validate="one_to_one",
    )
)

if len(runA_collection_boundary_df) != 10:
    raise ValueError(
        "The reconstructed Run A boundary table "
        "does not contain 10 apps."
    )

runA_collection_boundary_df[
    "previous_fetched_at_parsed"
] = pd.to_datetime(
    runA_collection_boundary_df[
        "previous_fetched_at"
    ],
    utc=True,
    errors="coerce",
)

runA_collection_boundary_df[
    "current_fetched_at_parsed"
] = pd.to_datetime(
    runA_collection_boundary_df[
        "current_fetched_at"
    ],
    utc=True,
    errors="coerce",
)

if runA_collection_boundary_df[
    [
        "previous_fetched_at_parsed",
        "current_fetched_at_parsed",
    ]
].isna().any().any():
    raise ValueError(
        "At least one Run A collection boundary "
        "could not be parsed."
    )

runA_collection_boundary_df[
    "collection_interval_hours"
] = (
    runA_collection_boundary_df[
        "current_fetched_at_parsed"
    ]
    - runA_collection_boundary_df[
        "previous_fetched_at_parsed"
    ]
).dt.total_seconds() / 3600

if runA_collection_boundary_df[
    "collection_interval_hours"
].le(0).any():
    raise ValueError(
        "At least one reconstructed Run A interval "
        "is not positive."
    )

runA_app_summary_df = pd.read_sql_query(
    """
    SELECT
        app_name,
        app_id,
        target_reviews,
        records_fetched,
        unique_reviews_in_batch,
        duplicate_reviews_in_batch,
        new_records_inserted,
        duplicates_skipped,
        runtime_seconds,
        min_review_date,
        max_review_date,
        error_message
    FROM phase2_app_run_summary
    WHERE run_id = ?
    ORDER BY rowid
    """,
    conn,
    params=(RUNA_RUN_ID,),
)

if len(runA_app_summary_df) != 10:
    raise ValueError(
        "Expected 10 app summaries for Cadence Run A."
    )

if not runA_app_summary_df[
    "records_fetched"
].eq(1200).all():
    raise ValueError(
        "At least one Run A app did not return 1,200 reviews."
    )

if runA_app_summary_df[
    "error_message"
].fillna("").ne("").any():
    raise ValueError(
        "Cadence Run A contains an app-level error."
    )

runA_timestamp_audit_df = pd.read_sql_query(
    """
    SELECT
        review_key,
        app_name,
        app_id,
        review_id,
        review_created_at,
        fetched_at,
        score,
        app_version,
        run_id
    FROM phase2_reviews_raw
    WHERE run_id = ?
    ORDER BY
        app_name,
        review_created_at,
        review_id
    """,
    conn,
    params=(RUNA_RUN_ID,),
)

expected_runA_new_insert_total = int(
    runA_app_summary_df[
        "new_records_inserted"
    ].sum()
)

if len(runA_timestamp_audit_df) != (
    expected_runA_new_insert_total
):
    raise ValueError(
        "Run A timestamp-audit rows do not match "
        "the app-level new insert total."
    )

runA_timestamp_audit_df[
    "review_timestamp_parsed"
] = pd.to_datetime(
    runA_timestamp_audit_df[
        "review_created_at"
    ],
    utc=True,
    errors="coerce",
)

runA_timestamp_audit_df[
    "row_fetched_at_parsed"
] = pd.to_datetime(
    runA_timestamp_audit_df["fetched_at"],
    utc=True,
    errors="coerce",
)

runA_timestamp_audit_df = (
    runA_timestamp_audit_df.merge(
        runA_collection_boundary_df[
            [
                "app_id",
                "previous_fetched_at_parsed",
                "current_fetched_at_parsed",
                "collection_interval_hours",
            ]
        ],
        on="app_id",
        how="left",
        validate="many_to_one",
    )
)

if runA_timestamp_audit_df[
    [
        "previous_fetched_at_parsed",
        "current_fetched_at_parsed",
    ]
].isna().any().any():
    raise ValueError(
        "At least one Run A review is missing "
        "an app-specific boundary."
    )

runA_fetch_boundary_match = (
    runA_timestamp_audit_df[
        "row_fetched_at_parsed"
    ]
    == runA_timestamp_audit_df[
        "current_fetched_at_parsed"
    ]
)

if not runA_fetch_boundary_match.all():
    raise ValueError(
        "At least one Run A review fetched_at value "
        "does not match its app boundary."
    )


def classify_runA_timestamp(row):
    review_ts = row["review_timestamp_parsed"]
    previous_ts = row[
        "previous_fetched_at_parsed"
    ]
    current_ts = row[
        "current_fetched_at_parsed"
    ]

    if (
        pd.isna(review_ts)
        or pd.isna(previous_ts)
        or pd.isna(current_ts)
    ):
        return "missing_or_unusable_timestamp"

    if review_ts <= previous_ts:
        return "older_review_surfaced_later"

    if review_ts <= current_ts:
        return "posted_between_collections"

    return "timestamp_after_fetch"


runA_timestamp_audit_df[
    "timestamp_classification"
] = runA_timestamp_audit_df.apply(
    classify_runA_timestamp,
    axis=1,
)

classification_columns = {
    "posted_between_collections": (
        "posted_between_collections"
    ),
    "older_review_surfaced_later": (
        "older_review_surfaced_later"
    ),
    "timestamp_after_fetch": (
        "timestamp_after_fetch"
    ),
    "missing_or_unusable_timestamp": (
        "missing_or_unusable_timestamp"
    ),
}

for output_column, class_name in (
    classification_columns.items()
):
    runA_timestamp_audit_df[
        output_column
    ] = (
        runA_timestamp_audit_df[
            "timestamp_classification"
        ]
        == class_name
    ).astype(int)


def safe_timestamp_iso(value):
    if value is None or pd.isna(value):
        return None

    return pd.Timestamp(value).isoformat()


runA_expected_by_app = (
    runA_app_summary_df
    .set_index("app_id")[
        "new_records_inserted"
    ]
    .to_dict()
)

runA_boundary_by_app = (
    runA_collection_boundary_df
    .set_index("app_id")
    .to_dict("index")
)

runA_timestamp_summary_rows = []

for _, app_row in app_config_df.iterrows():
    app_name = str(app_row["app_name"])
    app_id = str(app_row["app_id"])

    app_audit_df = runA_timestamp_audit_df[
        runA_timestamp_audit_df[
            "app_id"
        ] == app_id
    ].copy()

    valid_timestamp_df = app_audit_df[
        app_audit_df[
            "review_timestamp_parsed"
        ].notna()
    ]

    posted_between_df = app_audit_df[
        app_audit_df[
            "timestamp_classification"
        ]
        == "posted_between_collections"
    ]

    boundary = runA_boundary_by_app[
        app_id
    ]

    runA_timestamp_summary_rows.append(
        {
            "app_name": app_name,
            "app_id": app_id,
            "expected_new_inserts": int(
                runA_expected_by_app.get(
                    app_id,
                    0,
                )
            ),
            "audited_new_inserts": int(
                len(app_audit_df)
            ),
            "previous_fetched_at": (
                safe_timestamp_iso(
                    boundary[
                        "previous_fetched_at_parsed"
                    ]
                )
            ),
            "current_fetched_at": (
                safe_timestamp_iso(
                    boundary[
                        "current_fetched_at_parsed"
                    ]
                )
            ),
            "collection_interval_hours": float(
                boundary[
                    "collection_interval_hours"
                ]
            ),
            "posted_between_collections_count": int(
                app_audit_df[
                    "posted_between_collections"
                ].sum()
            ),
            "older_reviews_surfaced_later_count": int(
                app_audit_df[
                    "older_review_surfaced_later"
                ].sum()
            ),
            "timestamp_after_fetch_count": int(
                app_audit_df[
                    "timestamp_after_fetch"
                ].sum()
            ),
            "missing_or_unusable_timestamp_count": int(
                app_audit_df[
                    "missing_or_unusable_timestamp"
                ].sum()
            ),
            "new_insert_timestamp_min": (
                safe_timestamp_iso(
                    valid_timestamp_df[
                        "review_timestamp_parsed"
                    ].min()
                )
                if len(valid_timestamp_df) > 0
                else None
            ),
            "new_insert_timestamp_max": (
                safe_timestamp_iso(
                    valid_timestamp_df[
                        "review_timestamp_parsed"
                    ].max()
                )
                if len(valid_timestamp_df) > 0
                else None
            ),
            "posted_between_timestamp_min": (
                safe_timestamp_iso(
                    posted_between_df[
                        "review_timestamp_parsed"
                    ].min()
                )
                if len(posted_between_df) > 0
                else None
            ),
            "posted_between_timestamp_max": (
                safe_timestamp_iso(
                    posted_between_df[
                        "review_timestamp_parsed"
                    ].max()
                )
                if len(posted_between_df) > 0
                else None
            ),
        }
    )

runA_timestamp_summary_df = pd.DataFrame(
    runA_timestamp_summary_rows
)

runA_timestamp_summary_df[
    "classification_total"
] = (
    runA_timestamp_summary_df[
        "posted_between_collections_count"
    ]
    + runA_timestamp_summary_df[
        "older_reviews_surfaced_later_count"
    ]
    + runA_timestamp_summary_df[
        "timestamp_after_fetch_count"
    ]
    + runA_timestamp_summary_df[
        "missing_or_unusable_timestamp_count"
    ]
)

runA_timestamp_summary_df[
    "posted_between_collections_rate"
] = (
    runA_timestamp_summary_df[
        "posted_between_collections_count"
    ]
    / runA_timestamp_summary_df[
        "audited_new_inserts"
    ].replace(0, pd.NA)
).fillna(0.0)

runA_timestamp_summary_df[
    "older_surfaced_rate"
] = (
    runA_timestamp_summary_df[
        "older_reviews_surfaced_later_count"
    ]
    / runA_timestamp_summary_df[
        "audited_new_inserts"
    ].replace(0, pd.NA)
).fillna(0.0)

runA_timestamp_checks = {
    "audit_rows_match_runA_new_inserts": (
        len(runA_timestamp_audit_df)
        == expected_runA_new_insert_total
    ),
    "app_audit_counts_match_new_inserts": (
        runA_timestamp_summary_df[
            "audited_new_inserts"
        ].equals(
            runA_timestamp_summary_df[
                "expected_new_inserts"
            ]
        )
    ),
    "all_timestamp_classes_reconcile": (
        runA_timestamp_summary_df[
            "classification_total"
        ].equals(
            runA_timestamp_summary_df[
                "audited_new_inserts"
            ]
        )
    ),
    "all_10_apps_present": (
        len(runA_timestamp_summary_df) == 10
    ),
    "all_rows_have_runA_id": (
        runA_timestamp_audit_df[
            "run_id"
        ].eq(RUNA_RUN_ID).all()
    ),
    "all_rows_match_runA_fetch_boundary": (
        runA_fetch_boundary_match.all()
    ),
    "all_runA_intervals_positive": (
        runA_timestamp_summary_df[
            "collection_interval_hours"
        ].gt(0).all()
    ),
}

failed_runA_timestamp_checks = [
    check_name
    for check_name, passed
    in runA_timestamp_checks.items()
    if not passed
]

if failed_runA_timestamp_checks:
    raise ValueError(
        "Cadence Run A timestamp reconstruction failed: "
        f"{failed_runA_timestamp_checks}"
    )

runA_posted_between_total = int(
    runA_timestamp_summary_df[
        "posted_between_collections_count"
    ].sum()
)

runA_older_surfaced_total = int(
    runA_timestamp_summary_df[
        "older_reviews_surfaced_later_count"
    ].sum()
)

runA_after_fetch_total = int(
    runA_timestamp_summary_df[
        "timestamp_after_fetch_count"
    ].sum()
)

runA_missing_timestamp_total = int(
    runA_timestamp_summary_df[
        "missing_or_unusable_timestamp_count"
    ].sum()
)

runA_timestamp_validation_df = pd.DataFrame(
    [
        {
            "validation_check": check_name,
            "passed": passed,
        }
        for check_name, passed
        in runA_timestamp_checks.items()
    ]
)

runA_timestamp_display_df = (
    runA_timestamp_summary_df.copy()
)

runA_timestamp_display_df[
    "collection_interval_hours"
] = runA_timestamp_display_df[
    "collection_interval_hours"
].map(
    lambda value: f"{value:.2f}"
)

runA_timestamp_display_df[
    "posted_between_collections_rate"
] = runA_timestamp_display_df[
    "posted_between_collections_rate"
].map(
    lambda value: f"{value:.2%}"
)

runA_timestamp_display_df[
    "older_surfaced_rate"
] = runA_timestamp_display_df[
    "older_surfaced_rate"
].map(
    lambda value: f"{value:.2%}"
)

print(
    "Cadence Run A app-specific timestamp "
    "reconstruction passed."
)
print("-" * 82)
print(
    "Run A new inserts audited:",
    f"{len(runA_timestamp_audit_df):,}",
)
print(
    "Posted between collections:",
    f"{runA_posted_between_total:,}",
)
print(
    "Older reviews surfaced later:",
    f"{runA_older_surfaced_total:,}",
)
print(
    "Timestamps after fetch:",
    f"{runA_after_fetch_total:,}",
)
print(
    "Missing or unusable timestamps:",
    f"{runA_missing_timestamp_total:,}",
)
print(
    "Run A interval range:",
    (
        f"{runA_timestamp_summary_df['collection_interval_hours'].min():.2f}"
        "–"
        f"{runA_timestamp_summary_df['collection_interval_hours'].max():.2f}"
        " hours"
    ),
)

print("\nCadence Run A app-level timestamp summary:")
display(
    runA_timestamp_display_df[
        [
            "app_name",
            "audited_new_inserts",
            "collection_interval_hours",
            "posted_between_collections_count",
            "older_reviews_surfaced_later_count",
            "timestamp_after_fetch_count",
            "missing_or_unusable_timestamp_count",
            "posted_between_collections_rate",
            "older_surfaced_rate",
            "new_insert_timestamp_min",
            "new_insert_timestamp_max",
            "posted_between_timestamp_min",
            "posted_between_timestamp_max",
        ]
    ]
)

print("\nCadence Run A timestamp validation checks:")
display(runA_timestamp_validation_df)

Cadence Run A app-specific timestamp reconstruction passed.
----------------------------------------------------------------------------------
Run A new inserts audited: 4,395
Posted between collections: 0
Older reviews surfaced later: 4,395
Timestamps after fetch: 0
Missing or unusable timestamps: 0
Run A interval range: 17.16–17.16 hours

Cadence Run A app-level timestamp summary:


,app_name,audited_new_inserts,collection_interval_hours,posted_between_collections_count,older_reviews_surfaced_later_count,timestamp_after_fetch_count,missing_or_unusable_timestamp_count,posted_between_collections_rate,older_surfaced_rate,new_insert_timestamp_min,new_insert_timestamp_max,posted_between_timestamp_min,posted_between_timestamp_max
0,YouTube,1199,17.16,0,1199,0,0,0.00%,100.00%,2026-07-08T10:17:11+00:00,2026-07-08T21:13:22+00:00,None,None
1,TikTok,622,17.16,0,622,0,0,0.00%,100.00%,2026-07-08T04:08:06+00:00,2026-07-08T21:14:15+00:00,None,None
2,Spotify,580,17.16,0,580,0,0,0.00%,100.00%,2026-07-08T04:06:13+00:00,2026-07-08T21:14:19+00:00,None,None
3,Instagram,1198,17.16,0,1198,0,0,0.00%,100.00%,2026-07-08T11:26:23+00:00,2026-07-08T21:12:32+00:00,None,None
4,Uber,317,17.16,0,317,0,0,0.00%,100.00%,2026-07-08T04:09:18+00:00,2026-07-08T21:12:10+00:00,None,None
5,DoorDash,73,17.16,0,73,0,0,0.00%,100.00%,2026-07-08T04:54:25+00:00,2026-07-08T21:03:08+00:00,None,None
6,Duolingo,6,17.16,0,6,0,0,0.00%,100.00%,2026-07-08T04:17:19+00:00,2026-07-08T15:57:17+00:00,None,None
7,Google Maps,184,17.16,0,184,0,0,0.00%,100.00%,2026-07-04T15:23:42+00:00,2026-07-08T21:14:19+00:00,None,None
8,Netflix,123,17.16,0,123,0,0,0.00%,100.00%,2026-07-08T04:08:00+00:00,2026-07-08T21:01:32+00:00,None,None
9,Reddit,93,17.16,0,93,0,0,0.00%,100.00%,2026-07-08T04:15:59+00:00,2026-07-08T20:43:50+00:00,None,None



Cadence Run A timestamp validation checks:


,validation_check,passed
0,audit_rows_match_runA_new_inserts,True
1,app_audit_counts_match_new_inserts,True
2,all_timestamp_classes_reconcile,True
3,all_10_apps_present,True
4,all_rows_have_runA_id,True
5,all_rows_match_runA_fetch_boundary,True
6,all_runA_intervals_positive,True


## 12. Compare Cadence Run A and Run B and assign app-level recommendations

The two controlled higher-frequency tests are compared using the same definitions and app-specific collection boundaries.

The comparison separates two different operational questions:

### Timestamp freshness

A twice-daily freshness benefit is demonstrated only when review timestamps confirm that the reviews were posted between the two collections.

### Returned-window coverage

Even when the source has a reporting delay, a rapidly changing 1,200-review returned window may create a risk that a once-daily collection misses reviews that move through the window between runs.

For this controlled comparison, the following project-specific rules are used:

- **Twice-daily coverage candidate:** at least 80% of the returned batch was new to the database in both cadence tests
- **Once-daily candidate:** at least 70% of the returned batch was duplicate in both cadence tests
- **Monitor before changing cadence:** results fall between those two patterns

These thresholds are operational rules for this experiment, not universal Google Play rules.

A coverage recommendation does not override the timestamp result. The report will state separately whether a posting-time freshness benefit was demonstrated.

In [13]:
HIGH_WINDOW_TURNOVER_THRESHOLD = 0.80
DUPLICATE_HEAVY_THRESHOLD = 0.70

# -------------------------------------------------------------------
# Read the final run-level records after runtime normalization.
# -------------------------------------------------------------------
cadence_run_records_final_df = pd.read_sql_query(
    """
    SELECT
        run_id,
        run_label,
        frequency_label,
        run_started_at,
        run_finished_at,
        runtime_seconds,
        status,
        records_fetched_total,
        new_records_inserted_total,
        duplicates_skipped_total,
        review_rows_before,
        review_rows_after,
        review_rows_growth,
        db_size_before_mb,
        db_size_after_mb,
        db_size_growth_mb,
        errors_total
    FROM phase2_ingestion_runs
    WHERE run_id IN (?, ?)
    ORDER BY run_started_at
    """,
    conn,
    params=(
        RUNA_RUN_ID,
        RUN_ID,
    ),
)

if len(cadence_run_records_final_df) != 2:
    raise ValueError(
        "Expected exactly two cadence outcome runs."
    )

if not cadence_run_records_final_df[
    "status"
].eq("completed").all():
    raise ValueError(
        "Both cadence outcome runs must be completed."
    )

run_record_by_id = (
    cadence_run_records_final_df
    .set_index("run_id")
    .to_dict("index")
)

runA_record = run_record_by_id[RUNA_RUN_ID]
runB_record = run_record_by_id[RUN_ID]

# -------------------------------------------------------------------
# Calculate comparable source-lag values for Cadence Run A.
# -------------------------------------------------------------------
runA_source_lag_df = (
    runA_collection_boundary_df[
        [
            "app_name",
            "app_id",
            "current_fetched_at",
            "current_fetched_at_parsed",
        ]
    ]
    .merge(
        runA_app_summary_df[
            [
                "app_id",
                "max_review_date",
            ]
        ],
        on="app_id",
        how="left",
        validate="one_to_one",
    )
)

runA_source_lag_df[
    "latest_returned_review_parsed"
] = pd.to_datetime(
    runA_source_lag_df["max_review_date"],
    utc=True,
    errors="coerce",
)

if runA_source_lag_df[
    "latest_returned_review_parsed"
].isna().any():
    raise ValueError(
        "At least one Run A latest review timestamp "
        "could not be parsed."
    )

runA_source_lag_df[
    "source_lag_hours"
] = (
    runA_source_lag_df[
        "current_fetched_at_parsed"
    ]
    - runA_source_lag_df[
        "latest_returned_review_parsed"
    ]
).dt.total_seconds() / 3600

if runA_source_lag_df[
    "source_lag_hours"
].lt(0).any():
    raise ValueError(
        "At least one Run A source lag is negative."
    )

runA_median_source_lag = float(
    runA_source_lag_df[
        "source_lag_hours"
    ].median()
)

runB_median_source_lag = float(
    source_freshness_df[
        "followup_source_lag_hours"
    ].median()
)

# -------------------------------------------------------------------
# Build the run-level cadence comparison.
# -------------------------------------------------------------------
runA_new_timestamp_min = safe_timestamp_iso(
    runA_timestamp_audit_df[
        "review_timestamp_parsed"
    ].min()
)

runA_new_timestamp_max = safe_timestamp_iso(
    runA_timestamp_audit_df[
        "review_timestamp_parsed"
    ].max()
)

runB_new_timestamp_min = safe_timestamp_iso(
    new_review_timestamp_audit_df[
        "review_timestamp_parsed"
    ].min()
)

runB_new_timestamp_max = safe_timestamp_iso(
    new_review_timestamp_audit_df[
        "review_timestamp_parsed"
    ].max()
)

cadence_run_comparison_df = pd.DataFrame(
    [
        {
            "cadence_test": "Cadence Run A",
            "run_id": RUNA_RUN_ID,
            "preceding_collection": (
                "Day 3 controlled repeated run"
            ),
            "collection_interval_mean_hours": float(
                runA_timestamp_summary_df[
                    "collection_interval_hours"
                ].mean()
            ),
            "collection_interval_min_hours": float(
                runA_timestamp_summary_df[
                    "collection_interval_hours"
                ].min()
            ),
            "collection_interval_max_hours": float(
                runA_timestamp_summary_df[
                    "collection_interval_hours"
                ].max()
            ),
            "records_fetched": int(
                runA_record[
                    "records_fetched_total"
                ]
            ),
            "new_database_inserts": int(
                runA_record[
                    "new_records_inserted_total"
                ]
            ),
            "duplicates_skipped": int(
                runA_record[
                    "duplicates_skipped_total"
                ]
            ),
            "new_insert_rate": (
                int(
                    runA_record[
                        "new_records_inserted_total"
                    ]
                )
                / int(
                    runA_record[
                        "records_fetched_total"
                    ]
                )
            ),
            "duplicate_rate": (
                int(
                    runA_record[
                        "duplicates_skipped_total"
                    ]
                )
                / int(
                    runA_record[
                        "records_fetched_total"
                    ]
                )
            ),
            "posted_between_collections": (
                runA_posted_between_total
            ),
            "older_reviews_surfaced_later": (
                runA_older_surfaced_total
            ),
            "posted_between_rate_of_new_inserts": (
                runA_posted_between_total
                / int(
                    runA_record[
                        "new_records_inserted_total"
                    ]
                )
            ),
            "new_insert_timestamp_min": (
                runA_new_timestamp_min
            ),
            "new_insert_timestamp_max": (
                runA_new_timestamp_max
            ),
            "median_source_lag_hours": (
                runA_median_source_lag
            ),
            "wall_clock_runtime_seconds": float(
                runA_record["runtime_seconds"]
            ),
            "review_row_growth": int(
                runA_record["review_rows_growth"]
            ),
            "database_growth_mb": float(
                runA_record["db_size_growth_mb"]
            ),
            "errors": int(
                runA_record["errors_total"]
            ),
        },
        {
            "cadence_test": "Cadence Run B",
            "run_id": RUN_ID,
            "preceding_collection": (
                "Run B first collection"
            ),
            "collection_interval_mean_hours": float(
                timestamp_summary_df[
                    "collection_interval_hours"
                ].mean()
            ),
            "collection_interval_min_hours": float(
                timestamp_summary_df[
                    "collection_interval_hours"
                ].min()
            ),
            "collection_interval_max_hours": float(
                timestamp_summary_df[
                    "collection_interval_hours"
                ].max()
            ),
            "records_fetched": int(
                runB_record[
                    "records_fetched_total"
                ]
            ),
            "new_database_inserts": int(
                runB_record[
                    "new_records_inserted_total"
                ]
            ),
            "duplicates_skipped": int(
                runB_record[
                    "duplicates_skipped_total"
                ]
            ),
            "new_insert_rate": (
                int(
                    runB_record[
                        "new_records_inserted_total"
                    ]
                )
                / int(
                    runB_record[
                        "records_fetched_total"
                    ]
                )
            ),
            "duplicate_rate": (
                int(
                    runB_record[
                        "duplicates_skipped_total"
                    ]
                )
                / int(
                    runB_record[
                        "records_fetched_total"
                    ]
                )
            ),
            "posted_between_collections": (
                posted_between_collections_total
            ),
            "older_reviews_surfaced_later": (
                older_surfaced_total
            ),
            "posted_between_rate_of_new_inserts": (
                posted_between_collections_total
                / int(
                    runB_record[
                        "new_records_inserted_total"
                    ]
                )
            ),
            "new_insert_timestamp_min": (
                runB_new_timestamp_min
            ),
            "new_insert_timestamp_max": (
                runB_new_timestamp_max
            ),
            "median_source_lag_hours": (
                runB_median_source_lag
            ),
            "wall_clock_runtime_seconds": float(
                runB_record["runtime_seconds"]
            ),
            "review_row_growth": int(
                runB_record["review_rows_growth"]
            ),
            "database_growth_mb": float(
                runB_record["db_size_growth_mb"]
            ),
            "errors": int(
                runB_record["errors_total"]
            ),
        },
    ]
)

# -------------------------------------------------------------------
# Build the app-level comparison.
# -------------------------------------------------------------------
runA_app_core_df = (
    runA_app_summary_df[
        [
            "app_name",
            "app_id",
            "records_fetched",
            "new_records_inserted",
            "duplicates_skipped",
        ]
    ]
    .rename(
        columns={
            "records_fetched": (
                "runA_records_fetched"
            ),
            "new_records_inserted": (
                "runA_new_inserts"
            ),
            "duplicates_skipped": (
                "runA_duplicates"
            ),
        }
    )
)

runA_timestamp_core_df = (
    runA_timestamp_summary_df[
        [
            "app_id",
            "collection_interval_hours",
            "posted_between_collections_count",
            "older_reviews_surfaced_later_count",
            "new_insert_timestamp_min",
            "new_insert_timestamp_max",
        ]
    ]
    .rename(
        columns={
            "collection_interval_hours": (
                "runA_interval_hours"
            ),
            "posted_between_collections_count": (
                "runA_posted_between"
            ),
            "older_reviews_surfaced_later_count": (
                "runA_older_surfaced"
            ),
            "new_insert_timestamp_min": (
                "runA_new_timestamp_min"
            ),
            "new_insert_timestamp_max": (
                "runA_new_timestamp_max"
            ),
        }
    )
)

runB_app_core_df = (
    app_summary_df[
        [
            "app_id",
            "records_fetched",
            "new_records_inserted",
            "duplicates_skipped",
        ]
    ]
    .rename(
        columns={
            "records_fetched": (
                "runB_records_fetched"
            ),
            "new_records_inserted": (
                "runB_new_inserts"
            ),
            "duplicates_skipped": (
                "runB_duplicates"
            ),
        }
    )
)

runB_timestamp_core_df = (
    timestamp_summary_df[
        [
            "app_id",
            "collection_interval_hours",
            "posted_between_collections_count",
            "older_reviews_surfaced_later_count",
            "new_insert_timestamp_min",
            "new_insert_timestamp_max",
        ]
    ]
    .rename(
        columns={
            "collection_interval_hours": (
                "runB_interval_hours"
            ),
            "posted_between_collections_count": (
                "runB_posted_between"
            ),
            "older_reviews_surfaced_later_count": (
                "runB_older_surfaced"
            ),
            "new_insert_timestamp_min": (
                "runB_new_timestamp_min"
            ),
            "new_insert_timestamp_max": (
                "runB_new_timestamp_max"
            ),
        }
    )
)

cadence_app_comparison_df = (
    app_config_df[
        [
            "app_name",
            "app_id",
        ]
    ]
    .merge(
        runA_app_core_df,
        on=[
            "app_name",
            "app_id",
        ],
        how="left",
        validate="one_to_one",
    )
    .merge(
        runA_timestamp_core_df,
        on="app_id",
        how="left",
        validate="one_to_one",
    )
    .merge(
        runB_app_core_df,
        on="app_id",
        how="left",
        validate="one_to_one",
    )
    .merge(
        runB_timestamp_core_df,
        on="app_id",
        how="left",
        validate="one_to_one",
    )
)

if len(cadence_app_comparison_df) != 10:
    raise ValueError(
        "Expected a cadence comparison for 10 apps."
    )

if cadence_app_comparison_df.isna().any().any():
    missing_columns = cadence_app_comparison_df.columns[
        cadence_app_comparison_df.isna().any()
    ].tolist()

    raise ValueError(
        "The app-level cadence comparison contains "
        f"missing values in: {missing_columns}"
    )

cadence_app_comparison_df[
    "runA_new_insert_rate"
] = (
    cadence_app_comparison_df[
        "runA_new_inserts"
    ]
    / cadence_app_comparison_df[
        "runA_records_fetched"
    ]
)

cadence_app_comparison_df[
    "runA_duplicate_rate"
] = (
    cadence_app_comparison_df[
        "runA_duplicates"
    ]
    / cadence_app_comparison_df[
        "runA_records_fetched"
    ]
)

cadence_app_comparison_df[
    "runB_new_insert_rate"
] = (
    cadence_app_comparison_df[
        "runB_new_inserts"
    ]
    / cadence_app_comparison_df[
        "runB_records_fetched"
    ]
)

cadence_app_comparison_df[
    "runB_duplicate_rate"
] = (
    cadence_app_comparison_df[
        "runB_duplicates"
    ]
    / cadence_app_comparison_df[
        "runB_records_fetched"
    ]
)

cadence_app_comparison_df[
    "total_posted_between_collections"
] = (
    cadence_app_comparison_df[
        "runA_posted_between"
    ]
    + cadence_app_comparison_df[
        "runB_posted_between"
    ]
)

cadence_app_comparison_df[
    "timestamp_freshness_benefit"
] = cadence_app_comparison_df[
    "total_posted_between_collections"
].map(
    lambda count: (
        "Demonstrated"
        if count > 0
        else "Not demonstrated"
    )
)


def assign_cadence_recommendation(row):
    high_turnover_in_both = (
        row["runA_new_insert_rate"]
        >= HIGH_WINDOW_TURNOVER_THRESHOLD
        and row["runB_new_insert_rate"]
        >= HIGH_WINDOW_TURNOVER_THRESHOLD
    )

    duplicate_heavy_in_both = (
        row["runA_duplicate_rate"]
        >= DUPLICATE_HEAVY_THRESHOLD
        and row["runB_duplicate_rate"]
        >= DUPLICATE_HEAVY_THRESHOLD
    )

    if high_turnover_in_both:
        return (
            "Twice-daily candidate for "
            "returned-window coverage"
        )

    if duplicate_heavy_in_both:
        return "Once-daily candidate"

    return "Monitor before changing cadence"


def build_recommendation_reason(row):
    if (
        row["runA_new_insert_rate"]
        >= HIGH_WINDOW_TURNOVER_THRESHOLD
        and row["runB_new_insert_rate"]
        >= HIGH_WINDOW_TURNOVER_THRESHOLD
    ):
        return (
            "High returned-window turnover in both tests. "
            "The fixed 1,200-review batch was close to "
            "saturation, creating a coverage risk under "
            "a longer interval. No posting-time freshness "
            "benefit was observed."
        )

    if (
        row["runA_duplicate_rate"]
        >= DUPLICATE_HEAVY_THRESHOLD
        and row["runB_duplicate_rate"]
        >= DUPLICATE_HEAVY_THRESHOLD
    ):
        return (
            "Duplicate-heavy in both tests, with no "
            "timestamp-verified reviews posted between "
            "collections."
        )

    return (
        "Moderate returned-window turnover, but no "
        "timestamp-verified freshness benefit. More "
        "evidence is needed before creating an "
        "app-specific twice-daily exception."
    )


cadence_app_comparison_df[
    "cadence_recommendation"
] = cadence_app_comparison_df.apply(
    assign_cadence_recommendation,
    axis=1,
)

cadence_app_comparison_df[
    "recommendation_reason"
] = cadence_app_comparison_df.apply(
    build_recommendation_reason,
    axis=1,
)

# -------------------------------------------------------------------
# Validate the final comparison.
# -------------------------------------------------------------------
comparison_checks = {
    "two_cadence_runs_present": (
        len(cadence_run_comparison_df) == 2
    ),
    "both_runs_fetched_12000": (
        cadence_run_comparison_df[
            "records_fetched"
        ].eq(12000).all()
    ),
    "runA_new_inserts_match_4395": (
        int(
            cadence_run_comparison_df.loc[
                cadence_run_comparison_df[
                    "cadence_test"
                ]
                == "Cadence Run A",
                "new_database_inserts",
            ].iloc[0]
        )
        == 4395
    ),
    "runB_new_inserts_match_3753": (
        int(
            cadence_run_comparison_df.loc[
                cadence_run_comparison_df[
                    "cadence_test"
                ]
                == "Cadence Run B",
                "new_database_inserts",
            ].iloc[0]
        )
        == 3753
    ),
    "zero_posted_between_in_both_tests": (
        cadence_run_comparison_df[
            "posted_between_collections"
        ].eq(0).all()
    ),
    "all_new_inserts_classified_as_older": (
        (
            cadence_run_comparison_df[
                "new_database_inserts"
            ]
            == cadence_run_comparison_df[
                "older_reviews_surfaced_later"
            ]
        ).all()
    ),
    "all_10_apps_compared": (
        len(cadence_app_comparison_df) == 10
    ),
    "all_apps_have_zero_true_posted_count": (
        cadence_app_comparison_df[
            "total_posted_between_collections"
        ].eq(0).all()
    ),
    "all_apps_received_one_recommendation": (
        cadence_app_comparison_df[
            "cadence_recommendation"
        ].notna().all()
    ),
    "both_tests_show_over_20_hour_median_source_lag": (
        runA_median_source_lag >= 20
        and runB_median_source_lag >= 20
    ),
    "both_cadence_runs_have_zero_errors": (
        cadence_run_comparison_df[
            "errors"
        ].eq(0).all()
    ),
}

failed_comparison_checks = [
    check_name
    for check_name, passed
    in comparison_checks.items()
    if not passed
]

if failed_comparison_checks:
    raise ValueError(
        "Final cadence comparison validation failed: "
        f"{failed_comparison_checks}"
    )

comparison_validation_df = pd.DataFrame(
    [
        {
            "validation_check": check_name,
            "passed": passed,
        }
        for check_name, passed
        in comparison_checks.items()
    ]
)

# -------------------------------------------------------------------
# Prepare readable display tables.
# -------------------------------------------------------------------
cadence_run_display_df = (
    cadence_run_comparison_df.copy()
)

for column in [
    "new_insert_rate",
    "duplicate_rate",
    "posted_between_rate_of_new_inserts",
]:
    cadence_run_display_df[column] = (
        cadence_run_display_df[column]
        .map(lambda value: f"{value:.2%}")
    )

for column in [
    "collection_interval_mean_hours",
    "collection_interval_min_hours",
    "collection_interval_max_hours",
    "median_source_lag_hours",
    "wall_clock_runtime_seconds",
    "database_growth_mb",
]:
    cadence_run_display_df[column] = (
        cadence_run_display_df[column]
        .round(2)
    )

cadence_app_display_df = (
    cadence_app_comparison_df.copy()
)

for column in [
    "runA_new_insert_rate",
    "runA_duplicate_rate",
    "runB_new_insert_rate",
    "runB_duplicate_rate",
]:
    cadence_app_display_df[column] = (
        cadence_app_display_df[column]
        .map(lambda value: f"{value:.2%}")
    )

for column in [
    "runA_interval_hours",
    "runB_interval_hours",
]:
    cadence_app_display_df[column] = (
        cadence_app_display_df[column]
        .round(2)
    )

twice_daily_candidates = (
    cadence_app_comparison_df.loc[
        cadence_app_comparison_df[
            "cadence_recommendation"
        ]
        == (
            "Twice-daily candidate for "
            "returned-window coverage"
        ),
        "app_name",
    ].tolist()
)

once_daily_candidates = (
    cadence_app_comparison_df.loc[
        cadence_app_comparison_df[
            "cadence_recommendation"
        ]
        == "Once-daily candidate",
        "app_name",
    ].tolist()
)

monitor_candidates = (
    cadence_app_comparison_df.loc[
        cadence_app_comparison_df[
            "cadence_recommendation"
        ]
        == "Monitor before changing cadence",
        "app_name",
    ].tolist()
)

print("Final cadence comparison validated.")
print("-" * 82)
print(
    "Timestamp-verified reviews posted "
    "between collections in Run A:",
    runA_posted_between_total,
)
print(
    "Timestamp-verified reviews posted "
    "between collections in Run B:",
    posted_between_collections_total,
)
print(
    "Cadence Run A median source lag:",
    f"{runA_median_source_lag:.2f} hours",
)
print(
    "Cadence Run B median source lag:",
    f"{runB_median_source_lag:.2f} hours",
)
print(
    "Twice-daily coverage candidates:",
    ", ".join(twice_daily_candidates)
    if twice_daily_candidates
    else "None",
)
print(
    "Once-daily candidates:",
    ", ".join(once_daily_candidates)
    if once_daily_candidates
    else "None",
)
print(
    "Monitor before changing cadence:",
    ", ".join(monitor_candidates)
    if monitor_candidates
    else "None",
)

print("\nRun-level cadence comparison:")
display(
    cadence_run_display_df[
        [
            "cadence_test",
            "collection_interval_mean_hours",
            "records_fetched",
            "new_database_inserts",
            "duplicates_skipped",
            "new_insert_rate",
            "duplicate_rate",
            "posted_between_collections",
            "older_reviews_surfaced_later",
            "posted_between_rate_of_new_inserts",
            "new_insert_timestamp_min",
            "new_insert_timestamp_max",
            "median_source_lag_hours",
            "wall_clock_runtime_seconds",
            "review_row_growth",
            "database_growth_mb",
            "errors",
        ]
    ]
)

print("\nApp-level cadence recommendation:")
display(
    cadence_app_display_df[
        [
            "app_name",
            "runA_new_inserts",
            "runA_duplicate_rate",
            "runA_interval_hours",
            "runA_posted_between",
            "runA_new_timestamp_min",
            "runA_new_timestamp_max",
            "runB_new_inserts",
            "runB_duplicate_rate",
            "runB_interval_hours",
            "runB_posted_between",
            "runB_new_timestamp_min",
            "runB_new_timestamp_max",
            "timestamp_freshness_benefit",
            "cadence_recommendation",
            "recommendation_reason",
        ]
    ]
)

print("\nFinal comparison validation checks:")
display(comparison_validation_df)

Final cadence comparison validated.
----------------------------------------------------------------------------------
Timestamp-verified reviews posted between collections in Run A: 0
Timestamp-verified reviews posted between collections in Run B: 0
Cadence Run A median source lag: 24.04 hours
Cadence Run B median source lag: 24.04 hours
Twice-daily coverage candidates: YouTube, Instagram
Once-daily candidates: Uber, DoorDash, Duolingo, Google Maps, Netflix, Reddit
Monitor before changing cadence: TikTok, Spotify

Run-level cadence comparison:


,cadence_test,collection_interval_mean_hours,records_fetched,new_database_inserts,duplicates_skipped,new_insert_rate,duplicate_rate,posted_between_collections,older_reviews_surfaced_later,posted_between_rate_of_new_inserts,new_insert_timestamp_min,new_insert_timestamp_max,median_source_lag_hours,wall_clock_runtime_seconds,review_row_growth,database_growth_mb,errors
0,Cadence Run A,17.16,12000,4395,7605,36.62%,63.38%,0,4395,0.00%,2026-07-04T15:23:42+00:00,2026-07-08T21:14:19+00:00,24.04,28.14,4395,11.76,0
1,Cadence Run B,14.66,12000,3753,8247,31.27%,68.73%,0,3753,0.00%,2026-07-13T01:56:51+00:00,2026-07-13T16:36:13+00:00,24.04,30.02,3753,10.59,0



App-level cadence recommendation:


,app_name,runA_new_inserts,runA_duplicate_rate,runA_interval_hours,runA_posted_between,runA_new_timestamp_min,runA_new_timestamp_max,runB_new_inserts,runB_duplicate_rate,runB_interval_hours,runB_posted_between,runB_new_timestamp_min,runB_new_timestamp_max,timestamp_freshness_benefit,cadence_recommendation,recommendation_reason
0,YouTube,1199,0.08%,17.16,0,2026-07-08T10:17:11+00:00,2026-07-08T21:13:22+00:00,994,17.17%,14.66,0,2026-07-13T01:59:02+00:00,2026-07-13T16:35:07+00:00,Not demonstrated,Twice-daily candidate for returned-window cove...,High returned-window turnover in both tests. T...
1,TikTok,622,48.17%,17.16,0,2026-07-08T04:08:06+00:00,2026-07-08T21:14:15+00:00,406,66.17%,14.66,0,2026-07-13T01:59:51+00:00,2026-07-13T16:34:23+00:00,Not demonstrated,Monitor before changing cadence,"Moderate returned-window turnover, but no time..."
2,Spotify,580,51.67%,17.16,0,2026-07-08T04:06:13+00:00,2026-07-08T21:14:19+00:00,456,62.00%,14.66,0,2026-07-13T01:56:51+00:00,2026-07-13T16:35:05+00:00,Not demonstrated,Monitor before changing cadence,"Moderate returned-window turnover, but no time..."
3,Instagram,1198,0.17%,17.16,0,2026-07-08T11:26:23+00:00,2026-07-08T21:12:32+00:00,1196,0.33%,14.66,0,2026-07-13T08:02:32+00:00,2026-07-13T16:36:13+00:00,Not demonstrated,Twice-daily candidate for returned-window cove...,High returned-window turnover in both tests. T...
4,Uber,317,73.58%,17.16,0,2026-07-08T04:09:18+00:00,2026-07-08T21:12:10+00:00,309,74.25%,14.66,0,2026-07-13T02:01:50+00:00,2026-07-13T16:34:04+00:00,Not demonstrated,Once-daily candidate,"Duplicate-heavy in both tests, with no timesta..."
5,DoorDash,73,93.92%,17.16,0,2026-07-08T04:54:25+00:00,2026-07-08T21:03:08+00:00,25,97.92%,14.66,0,2026-07-13T01:58:41+00:00,2026-07-13T16:32:29+00:00,Not demonstrated,Once-daily candidate,"Duplicate-heavy in both tests, with no timesta..."
6,Duolingo,6,99.50%,17.16,0,2026-07-08T04:17:19+00:00,2026-07-08T15:57:17+00:00,3,99.75%,14.66,0,2026-07-13T08:45:54+00:00,2026-07-13T16:07:35+00:00,Not demonstrated,Once-daily candidate,"Duplicate-heavy in both tests, with no timesta..."
7,Google Maps,184,84.67%,17.16,0,2026-07-04T15:23:42+00:00,2026-07-08T21:14:19+00:00,151,87.42%,14.66,0,2026-07-13T02:06:35+00:00,2026-07-13T16:34:26+00:00,Not demonstrated,Once-daily candidate,"Duplicate-heavy in both tests, with no timesta..."
8,Netflix,123,89.75%,17.16,0,2026-07-08T04:08:00+00:00,2026-07-08T21:01:32+00:00,152,87.33%,14.66,0,2026-07-13T01:58:20+00:00,2026-07-13T16:31:09+00:00,Not demonstrated,Once-daily candidate,"Duplicate-heavy in both tests, with no timesta..."
9,Reddit,93,92.25%,17.16,0,2026-07-08T04:15:59+00:00,2026-07-08T20:43:50+00:00,61,94.92%,14.66,0,2026-07-13T02:03:14+00:00,2026-07-13T16:32:15+00:00,Not demonstrated,Once-daily candidate,"Duplicate-heavy in both tests, with no timesta..."



Final comparison validation checks:


,validation_check,passed
0,two_cadence_runs_present,True
1,both_runs_fetched_12000,True
2,runA_new_inserts_match_4395,True
3,runB_new_inserts_match_3753,True
4,zero_posted_between_in_both_tests,True
5,all_new_inserts_classified_as_older,True
6,all_10_apps_compared,True
7,all_apps_have_zero_true_posted_count,True
8,all_apps_received_one_recommendation,True
9,both_tests_show_over_20_hour_median_source_lag,True


## 13. Final findings and cadence recommendation

Both controlled higher-frequency tests completed successfully with the same 10 apps, 1,200-review target, source settings, database continuation, and duplicate-prevention logic.

### Run-level results

- **Cadence Run A:** 17.16-hour interval, 4,395 new database inserts, 7,605 duplicates, 28.14-second runtime, 4,395-row database growth, and 11.76 MB of database growth
- **Cadence Run B:** 14.66-hour interval, 3,753 new database inserts, 8,247 duplicates, 30.02-second runtime, 3,753-row database growth, and 10.59 MB of database growth
- Both runs completed with zero collection errors

### Review timestamp result

The timestamp audit found zero reviews that were posted between collections in either cadence test.

All 8,148 database-new records across Run A and Run B had review timestamps at or before the preceding app-specific collection boundary. They were new to the database because they entered the returned review window later, not because they were newly posted between collections.

The newest returned review remained approximately 24.04 hours behind the collection time in both cadence tests. This was a consistent observation in these controlled runs, but it should not be treated as a universal Google Play rule.

### App-level cadence result

- **Twice-daily coverage candidates:** YouTube and Instagram  
  Both apps had at least 80% returned-window turnover in both cadence tests. Twice-daily collection may reduce the risk of reviews moving through the fixed 1,200-review window before the next collection. This is a coverage consideration, not a posting-time freshness benefit.

- **Once-daily candidates:** Uber, DoorDash, Duolingo, Google Maps, Netflix, and Reddit  
  These apps were at least 70% duplicate in both cadence tests and produced no timestamp-verified reviews posted between collections.

- **Monitor before changing cadence:** TikTok and Spotify  
  Their returned-window turnover was moderate and the current evidence does not support a clear app-specific twice-daily exception.

### Recommendation

A blanket twice-daily schedule is not supported for all 10 apps.

The controlled results support using once-daily collection as the default, while treating YouTube and Instagram as possible twice-daily exceptions when returned-window coverage is important. TikTok and Spotify should remain under observation, and the other six apps can remain on once-daily collection under the current 1,200-review setup.

In [14]:
FINAL_EXPORT_DIR = Path(
    "/content/phase2_cadence_runB_followup_final_package"
)

if FINAL_EXPORT_DIR.exists():
    shutil.rmtree(FINAL_EXPORT_DIR)

FINAL_EXPORT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

export_timestamp = datetime.now(
    timezone.utc
).strftime("%Y%m%d_%H%M%S_utc")

file_prefix = (
    "phase2_cadence_runB_followup"
)

conn.commit()

# -------------------------------------------------------------------
# Read the final database state after runtime normalization.
# -------------------------------------------------------------------
final_run_history_df = pd.read_sql_query(
    """
    SELECT
        run_id,
        run_label,
        frequency_label,
        target_reviews_per_app,
        app_count,
        run_started_at,
        run_finished_at,
        runtime_seconds,
        status,
        records_fetched_total,
        new_records_inserted_total,
        duplicates_skipped_total,
        errors_total,
        quality_flag_total,
        quality_flags_inserted,
        review_rows_before,
        review_rows_after,
        review_rows_growth,
        db_size_before_mb,
        db_size_after_mb,
        db_size_growth_mb
    FROM phase2_ingestion_runs
    ORDER BY run_started_at
    """,
    conn,
)

final_runB_records_df = pd.read_sql_query(
    """
    SELECT
        run_id,
        run_label,
        frequency_label,
        run_started_at,
        run_finished_at,
        runtime_seconds,
        status,
        records_fetched_total,
        new_records_inserted_total,
        duplicates_skipped_total,
        errors_total,
        quality_flag_total,
        quality_flags_inserted,
        review_rows_before,
        review_rows_after,
        review_rows_growth,
        db_size_before_mb,
        db_size_after_mb,
        db_size_growth_mb
    FROM phase2_ingestion_runs
    WHERE run_id IN (?, ?)
    ORDER BY run_started_at
    """,
    conn,
    params=(
        PREVIOUS_RUN_ID,
        RUN_ID,
    ),
)

final_raw_rows = int(
    pd.read_sql_query(
        """
        SELECT COUNT(*) AS n
        FROM phase2_reviews_raw
        """,
        conn,
    )["n"].iloc[0]
)

final_cleaned_rows = int(
    pd.read_sql_query(
        """
        SELECT COUNT(*) AS n
        FROM phase2_reviews_cleaned
        """,
        conn,
    )["n"].iloc[0]
)

final_quality_flag_rows = int(
    pd.read_sql_query(
        """
        SELECT COUNT(*) AS n
        FROM phase2_quality_flags
        """,
        conn,
    )["n"].iloc[0]
)

final_app_summary_rows = int(
    pd.read_sql_query(
        """
        SELECT COUNT(*) AS n
        FROM phase2_app_run_summary
        """,
        conn,
    )["n"].iloc[0]
)

final_duplicate_identity_groups = int(
    pd.read_sql_query(
        """
        SELECT COUNT(*) AS n
        FROM (
            SELECT
                source,
                app_id,
                review_id,
                COUNT(*) AS row_count
            FROM phase2_reviews_raw
            GROUP BY
                source,
                app_id,
                review_id
            HAVING COUNT(*) > 1
        )
        """,
        conn,
    )["n"].iloc[0]
)

final_raw_without_cleaned = int(
    pd.read_sql_query(
        """
        SELECT COUNT(*) AS n
        FROM phase2_reviews_raw AS r
        LEFT JOIN phase2_reviews_cleaned AS c
            ON r.review_key = c.review_key
        WHERE c.review_key IS NULL
        """,
        conn,
    )["n"].iloc[0]
)

final_cleaned_without_raw = int(
    pd.read_sql_query(
        """
        SELECT COUNT(*) AS n
        FROM phase2_reviews_cleaned AS c
        LEFT JOIN phase2_reviews_raw AS r
            ON c.review_key = r.review_key
        WHERE r.review_key IS NULL
        """,
        conn,
    )["n"].iloc[0]
)

final_orphan_quality_flags = int(
    pd.read_sql_query(
        """
        SELECT COUNT(*) AS n
        FROM phase2_quality_flags AS q
        LEFT JOIN phase2_reviews_raw AS r
            ON q.review_key = r.review_key
        WHERE r.review_key IS NULL
        """,
        conn,
    )["n"].iloc[0]
)

final_foreign_key_violations_df = pd.read_sql_query(
    "PRAGMA foreign_key_check",
    conn,
)

final_foreign_key_violation_count = int(
    len(final_foreign_key_violations_df)
)

final_database_size_mb = float(
    DB_PATH.stat().st_size / (1024 ** 2)
)

final_database_checks = {
    "six_completed_runs": (
        len(final_run_history_df) == 6
        and final_run_history_df[
            "status"
        ].eq("completed").all()
    ),
    "sixty_app_summary_rows": (
        final_app_summary_rows == 60
    ),
    "raw_rows_equal_34601": (
        final_raw_rows == 34601
    ),
    "cleaned_rows_equal_34601": (
        final_cleaned_rows == 34601
    ),
    "raw_and_cleaned_totals_match": (
        final_raw_rows == final_cleaned_rows
    ),
    "no_duplicate_review_identities": (
        final_duplicate_identity_groups == 0
    ),
    "no_raw_rows_without_cleaned": (
        final_raw_without_cleaned == 0
    ),
    "no_cleaned_rows_without_raw": (
        final_cleaned_without_raw == 0
    ),
    "no_orphan_quality_flags": (
        final_orphan_quality_flags == 0
    ),
    "no_foreign_key_violations": (
        final_foreign_key_violation_count == 0
    ),
    "two_final_runB_records_present": (
        len(final_runB_records_df) == 2
    ),
    "runB_followup_completed": (
        final_runB_records_df.loc[
            final_runB_records_df["run_id"] == RUN_ID,
            "status",
        ].iloc[0]
        == "completed"
    ),
    "runB_followup_growth_matches_3753": (
        int(
            final_runB_records_df.loc[
                final_runB_records_df["run_id"] == RUN_ID,
                "review_rows_growth",
            ].iloc[0]
        )
        == 3753
    ),
}

failed_final_database_checks = [
    check_name
    for check_name, passed
    in final_database_checks.items()
    if not passed
]

if failed_final_database_checks:
    raise ValueError(
        "Final database validation failed: "
        f"{failed_final_database_checks}"
    )

final_database_validation_df = pd.DataFrame(
    [
        {
            "validation_check": check_name,
            "passed": passed,
        }
        for check_name, passed
        in final_database_checks.items()
    ]
)

# -------------------------------------------------------------------
# Save all final CSV outputs.
# -------------------------------------------------------------------
csv_exports = {
    (
        f"{file_prefix}_app_summary.csv"
    ): app_summary_df,

    (
        f"{file_prefix}_timestamp_audit.csv"
    ): new_review_timestamp_audit_df,

    (
        f"{file_prefix}_timestamp_summary.csv"
    ): timestamp_summary_df,

    (
        f"{file_prefix}_source_freshness_diagnostic.csv"
    ): source_freshness_df,

    (
        "phase2_cadence_runA_"
        "timestamp_audit_reconstructed.csv"
    ): runA_timestamp_audit_df,

    (
        "phase2_cadence_runA_"
        "timestamp_summary_reconstructed.csv"
    ): runA_timestamp_summary_df,

    (
        "phase2_cadence_runA_runB_"
        "run_level_comparison.csv"
    ): cadence_run_comparison_df,

    (
        "phase2_cadence_runA_runB_"
        "app_level_recommendations.csv"
    ): cadence_app_comparison_df,

    (
        f"{file_prefix}_runtime_metrics.csv"
    ): runtime_metric_df,

    (
        f"{file_prefix}_final_runB_records.csv"
    ): final_runB_records_df,

    (
        "phase2_complete_run_history.csv"
    ): final_run_history_df,

    (
        f"{file_prefix}_database_validation.csv"
    ): validation_df,

    (
        f"{file_prefix}_timestamp_validation.csv"
    ): timestamp_validation_df,

    (
        f"{file_prefix}_source_freshness_validation.csv"
    ): source_freshness_validation_df,

    (
        f"{file_prefix}_runtime_validation.csv"
    ): runtime_validation_df,

    (
        "phase2_cadence_runA_"
        "timestamp_validation.csv"
    ): runA_timestamp_validation_df,

    (
        "phase2_cadence_runA_runB_"
        "comparison_validation.csv"
    ): comparison_validation_df,

    (
        f"{file_prefix}_final_database_validation.csv"
    ): final_database_validation_df,
}

for file_name, dataframe in csv_exports.items():
    dataframe.to_csv(
        FINAL_EXPORT_DIR / file_name,
        index=False,
    )

# -------------------------------------------------------------------
# Build the final Markdown report from the validated results.
# -------------------------------------------------------------------
run_table_lines = [
    (
        "| Test | Interval | New DB inserts | "
        "Duplicate rate | Posted between | "
        "Runtime | DB row growth | DB growth |"
    ),
    (
        "|---|---:|---:|---:|---:|---:|---:|---:|"
    ),
]

for _, row in cadence_run_comparison_df.iterrows():
    run_table_lines.append(
        (
            f"| {row['cadence_test']} "
            f"| {row['collection_interval_mean_hours']:.2f} h "
            f"| {int(row['new_database_inserts']):,} "
            f"| {row['duplicate_rate']:.2%} "
            f"| {int(row['posted_between_collections']):,} "
            f"| {row['wall_clock_runtime_seconds']:.2f} s "
            f"| {int(row['review_row_growth']):,} "
            f"| {row['database_growth_mb']:.2f} MB |"
        )
    )

app_table_lines = [
    (
        "| App | Run A new | Run A duplicate rate | "
        "Run B new | Run B duplicate rate | "
        "Posted between A / B | Recommendation |"
    ),
    (
        "|---|---:|---:|---:|---:|---:|---|"
    ),
]

for _, row in cadence_app_comparison_df.iterrows():
    app_table_lines.append(
        (
            f"| {row['app_name']} "
            f"| {int(row['runA_new_inserts']):,} "
            f"| {row['runA_duplicate_rate']:.2%} "
            f"| {int(row['runB_new_inserts']):,} "
            f"| {row['runB_duplicate_rate']:.2%} "
            f"| {int(row['runA_posted_between'])} / "
            f"{int(row['runB_posted_between'])} "
            f"| {row['cadence_recommendation']} |"
        )
    )

final_report_text = f"""# Google Play Phase 2 Cadence Test — Run B Follow-up

## Controlled setup

- Source: Google Play
- Apps: 10
- Target: 1,200 newest reviews per app
- Language and country: English / United States
- Duplicate identity: source + app_id + review_id
- Final database rows: {final_raw_rows:,}
- Completed Phase 2 runs: {len(final_run_history_df)}

## Run-level comparison

{chr(10).join(run_table_lines)}

## Timestamp finding

Cadence Run A inserted {int(runA_record['new_records_inserted_total']):,} new database records, and Cadence Run B inserted {int(runB_record['new_records_inserted_total']):,}.

Timestamp validation found zero reviews posted between collections in both tests. All {int(runA_record['new_records_inserted_total']) + int(runB_record['new_records_inserted_total']):,} new database inserts were already posted by the preceding app-specific collection boundary.

The median lag between collection time and the newest returned review timestamp was {runA_median_source_lag:.2f} hours in Cadence Run A and {runB_median_source_lag:.2f} hours in Cadence Run B.

## App-level result

{chr(10).join(app_table_lines)}

## Recommendation

A blanket twice-daily schedule is not supported for all 10 apps.

- Twice-daily coverage candidates: {", ".join(twice_daily_candidates)}
- Once-daily candidates: {", ".join(once_daily_candidates)}
- Monitor before changing cadence: {", ".join(monitor_candidates)}

YouTube and Instagram are twice-daily candidates only for returned-window coverage. No posting-time freshness benefit was demonstrated for either app.

The other six once-daily candidates were duplicate-heavy in both controlled cadence tests. TikTok and Spotify showed moderate turnover and should remain under observation.

These conclusions apply to the tested 1,200-review returned window and the controlled runs recorded in this database.
"""

final_report_path = (
    FINAL_EXPORT_DIR
    / "phase2_cadence_runA_runB_final_report.md"
)

final_report_path.write_text(
    final_report_text,
    encoding="utf-8",
)

# -------------------------------------------------------------------
# Create an internally consistent database backup.
# -------------------------------------------------------------------
backup_db_path = (
    FINAL_EXPORT_DIR
    / "google_play_reviews_after_runB_followup.sqlite"
)

if backup_db_path.exists():
    backup_db_path.unlink()

with sqlite3.connect(
    backup_db_path
) as backup_conn:
    conn.backup(backup_conn)

with sqlite3.connect(
    backup_db_path
) as verify_conn:
    backup_raw_rows = int(
        pd.read_sql_query(
            """
            SELECT COUNT(*) AS n
            FROM phase2_reviews_raw
            """,
            verify_conn,
        )["n"].iloc[0]
    )

    backup_cleaned_rows = int(
        pd.read_sql_query(
            """
            SELECT COUNT(*) AS n
            FROM phase2_reviews_cleaned
            """,
            verify_conn,
        )["n"].iloc[0]
    )

    backup_completed_runs = int(
        pd.read_sql_query(
            """
            SELECT COUNT(*) AS n
            FROM phase2_ingestion_runs
            WHERE status = 'completed'
            """,
            verify_conn,
        )["n"].iloc[0]
    )

    backup_app_summary_rows = int(
        pd.read_sql_query(
            """
            SELECT COUNT(*) AS n
            FROM phase2_app_run_summary
            """,
            verify_conn,
        )["n"].iloc[0]
    )

if backup_raw_rows != 34601:
    raise ValueError(
        "The backup database raw-row count is incorrect."
    )

if backup_cleaned_rows != 34601:
    raise ValueError(
        "The backup database cleaned-row count is incorrect."
    )

if backup_completed_runs != 6:
    raise ValueError(
        "The backup database does not contain "
        "six completed runs."
    )

if backup_app_summary_rows != 60:
    raise ValueError(
        "The backup database does not contain "
        "60 app-summary rows."
    )

database_sha256 = calculate_sha256(
    backup_db_path
)

# -------------------------------------------------------------------
# Save package metadata.
# -------------------------------------------------------------------
metadata = {
    "project": (
        "Google Play Phase 2 controlled cadence test"
    ),
    "source": SOURCE,
    "language": LANGUAGE,
    "country": COUNTRY,
    "app_count": 10,
    "target_reviews_per_app": 1200,
    "duplicate_identity": (
        "source + app_id + review_id"
    ),
    "runA_run_id": RUNA_RUN_ID,
    "runB_first_collection_run_id": (
        PREVIOUS_RUN_ID
    ),
    "runB_followup_run_id": RUN_ID,
    "runA_interval_hours": float(
        runA_timestamp_summary_df[
            "collection_interval_hours"
        ].mean()
    ),
    "runB_interval_hours": float(
        timestamp_summary_df[
            "collection_interval_hours"
        ].mean()
    ),
    "runA_new_database_inserts": int(
        runA_record[
            "new_records_inserted_total"
        ]
    ),
    "runB_new_database_inserts": int(
        runB_record[
            "new_records_inserted_total"
        ]
    ),
    "runA_posted_between_collections": (
        runA_posted_between_total
    ),
    "runB_posted_between_collections": (
        posted_between_collections_total
    ),
    "runA_median_source_lag_hours": (
        runA_median_source_lag
    ),
    "runB_median_source_lag_hours": (
        runB_median_source_lag
    ),
    "high_window_turnover_threshold": (
        HIGH_WINDOW_TURNOVER_THRESHOLD
    ),
    "duplicate_heavy_threshold": (
        DUPLICATE_HEAVY_THRESHOLD
    ),
    "twice_daily_coverage_candidates": (
        twice_daily_candidates
    ),
    "once_daily_candidates": (
        once_daily_candidates
    ),
    "monitor_candidates": (
        monitor_candidates
    ),
    "final_database_rows": final_raw_rows,
    "final_cleaned_rows": final_cleaned_rows,
    "final_quality_flag_rows": (
        final_quality_flag_rows
    ),
    "final_database_size_mb": (
        final_database_size_mb
    ),
    "completed_run_count": (
        len(final_run_history_df)
    ),
    "database_sha256": database_sha256,
    "exported_at": datetime.now(
        timezone.utc
    ).isoformat(),
}

metadata_path = (
    FINAL_EXPORT_DIR
    / f"{file_prefix}_metadata.json"
)

with open(
    metadata_path,
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        metadata,
        file,
        indent=2,
        ensure_ascii=False,
    )

# Compress the database inside the final package.
database_zip_path = (
    FINAL_EXPORT_DIR
    / "google_play_reviews_after_runB_followup.sqlite.zip"
)

with zipfile.ZipFile(
    database_zip_path,
    "w",
    compression=zipfile.ZIP_DEFLATED,
) as zf:
    zf.write(
        backup_db_path,
        arcname=backup_db_path.name,
    )

backup_db_path.unlink()

# -------------------------------------------------------------------
# Create the package manifest.
# -------------------------------------------------------------------
manifest_rows = []

for file_path in sorted(
    FINAL_EXPORT_DIR.iterdir()
):
    if file_path.is_file():
        manifest_rows.append(
            {
                "file_name": file_path.name,
                "size_bytes": (
                    file_path.stat().st_size
                ),
                "sha256": calculate_sha256(
                    file_path
                ),
            }
        )

manifest_df = pd.DataFrame(
    manifest_rows
)

manifest_df.to_csv(
    FINAL_EXPORT_DIR
    / "final_package_manifest.csv",
    index=False,
)

required_final_files = [
    (
        "phase2_cadence_runA_runB_"
        "final_report.md"
    ),
    (
        "phase2_cadence_runA_runB_"
        "run_level_comparison.csv"
    ),
    (
        "phase2_cadence_runA_runB_"
        "app_level_recommendations.csv"
    ),
    (
        f"{file_prefix}_timestamp_audit.csv"
    ),
    (
        f"{file_prefix}_source_freshness_diagnostic.csv"
    ),
    (
        "google_play_reviews_after_"
        "runB_followup.sqlite.zip"
    ),
    (
        f"{file_prefix}_metadata.json"
    ),
    "final_package_manifest.csv",
]

missing_final_files = [
    file_name
    for file_name in required_final_files
    if not (
        FINAL_EXPORT_DIR / file_name
    ).exists()
]

if missing_final_files:
    raise FileNotFoundError(
        "The final package is missing required files: "
        f"{missing_final_files}"
    )

package_base_path = (
    BASE_DIR
    / (
        f"{file_prefix}_github_upload_files_"
        f"{export_timestamp}"
    )
)

package_zip_path = Path(
    shutil.make_archive(
        str(package_base_path),
        "zip",
        root_dir=FINAL_EXPORT_DIR,
    )
)

if not package_zip_path.exists():
    raise FileNotFoundError(
        "The final package ZIP was not created."
    )

print(
    "Run B follow-up final package created successfully."
)
print("-" * 78)
print(
    "Final raw review rows:",
    f"{final_raw_rows:,}",
)
print(
    "Final cleaned review rows:",
    f"{final_cleaned_rows:,}",
)
print(
    "Completed Phase 2 runs:",
    len(final_run_history_df),
)
print(
    "App-summary rows:",
    final_app_summary_rows,
)
print(
    "Database size:",
    f"{final_database_size_mb:.2f} MB",
)
print(
    "Database SHA-256:",
    database_sha256,
)
print(
    "Package:",
    package_zip_path.name,
)
print(
    "Package size:",
    f"{package_zip_path.stat().st_size / (1024 ** 2):.2f} MB",
)

print("\nFinal database validation:")
display(final_database_validation_df)

print("\nFinal package files:")
display(
    pd.DataFrame(
        [
            {
                "file_name": path.name,
                "size_mb": (
                    path.stat().st_size
                    / (1024 ** 2)
                ),
            }
            for path in sorted(
                FINAL_EXPORT_DIR.iterdir()
            )
            if path.is_file()
        ]
    )
)

files.download(str(package_zip_path))

Run B follow-up final package created successfully.
------------------------------------------------------------------------------
Final raw review rows: 34,601
Final cleaned review rows: 34,601
Completed Phase 2 runs: 6
App-summary rows: 60
Database size: 84.68 MB
Database SHA-256: a26f6db1b16b35d7763fb8f6e9ce4048ecd3b947749811cc5c34c3d099f3cd69
Package: phase2_cadence_runB_followup_github_upload_files_20260714_165418_utc.zip
Package size: 21.48 MB

Final database validation:


,validation_check,passed
0,six_completed_runs,True
1,sixty_app_summary_rows,True
2,raw_rows_equal_34601,True
3,cleaned_rows_equal_34601,True
4,raw_and_cleaned_totals_match,True
5,no_duplicate_review_identities,True
6,no_raw_rows_without_cleaned,True
7,no_cleaned_rows_without_raw,True
8,no_orphan_quality_flags,True
9,no_foreign_key_violations,True



Final package files:


,file_name,size_mb
0,final_package_manifest.csv,0.002444
1,google_play_reviews_after_runB_followup.sqlite...,20.859097
2,phase2_cadence_runA_runB_app_level_recommendat...,0.004760
3,phase2_cadence_runA_runB_comparison_validation...,0.000411
4,phase2_cadence_runA_runB_final_report.md,0.002649
5,phase2_cadence_runA_runB_run_level_comparison.csv,0.001002
6,phase2_cadence_runA_timestamp_audit_reconstruc...,1.823096
7,phase2_cadence_runA_timestamp_summary_reconstr...,0.002276
8,phase2_cadence_runA_timestamp_validation.csv,0.000253
9,phase2_cadence_runB_followup_app_summary.csv,0.003635


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## 14. Complete the final report and rebuild the GitHub package

The original export already contains the validated database, detailed CSV outputs, timestamp audits, and cadence recommendations.

This final step expands the Markdown report so the main report directly includes:

- the role and results of the Run B first collection
- run-level new-insert timestamp ranges
- app-level new insert counts
- app-level duplicate rates
- app-level processing runtime
- app-level new-insert timestamp ranges
- timestamp-verified reviews posted between collections
- run-level database growth
- the final app-specific cadence recommendation

The database and analytical results are not changed. Only the report presentation, package metadata, manifest, and final ZIP are rebuilt.

In [15]:
from IPython.display import Markdown, display

if globals().get(
    "FINAL_REPORT_REBUILD_COMPLETED",
    False,
):
    raise ValueError(
        "The completed-report package has already "
        "been rebuilt in this runtime."
    )

required_objects = [
    "FINAL_EXPORT_DIR",
    "EXTRACT_DIR",
    "checkpoint_metadata",
    "cadence_run_comparison_df",
    "cadence_app_comparison_df",
    "runA_app_summary_df",
    "runA_timestamp_summary_df",
    "app_summary_df",
    "timestamp_summary_df",
    "runA_record",
    "runB_record",
    "runA_median_source_lag",
    "runB_median_source_lag",
    "runA_posted_between_total",
    "posted_between_collections_total",
    "twice_daily_candidates",
    "once_daily_candidates",
    "monitor_candidates",
    "final_raw_rows",
    "final_run_history_df",
    "final_database_size_mb",
    "calculate_sha256",
]

missing_objects = [
    name
    for name in required_objects
    if name not in globals()
]

if missing_objects:
    raise NameError(
        "Required notebook objects are missing: "
        f"{missing_objects}. Do not rerun the "
        "collection cells."
    )

if not FINAL_EXPORT_DIR.exists():
    raise FileNotFoundError(
        "The final export directory does not exist."
    )

# -------------------------------------------------------------------
# Read the saved Run B first-collection baseline outputs.
# -------------------------------------------------------------------
first_collection_timestamp_path = (
    EXTRACT_DIR
    / (
        "phase2_cadence_runB_first_collection_"
        "timestamp_summary.csv"
    )
)

first_collection_app_summary_path = (
    EXTRACT_DIR
    / (
        "phase2_cadence_runB_first_collection_"
        "app_summary.csv"
    )
)

if not first_collection_timestamp_path.exists():
    raise FileNotFoundError(
        "The Run B first-collection timestamp "
        "summary was not found."
    )

if not first_collection_app_summary_path.exists():
    raise FileNotFoundError(
        "The Run B first-collection app summary "
        "was not found."
    )

first_collection_timestamp_df = pd.read_csv(
    first_collection_timestamp_path
)

first_collection_app_summary_df = pd.read_csv(
    first_collection_app_summary_path
)

if len(first_collection_timestamp_df) != 10:
    raise ValueError(
        "Expected 10 rows in the Run B first-collection "
        "timestamp summary."
    )

if len(first_collection_app_summary_df) != 10:
    raise ValueError(
        "Expected 10 rows in the Run B first-collection "
        "app summary."
    )

first_collection_timestamp_min = pd.to_datetime(
    first_collection_timestamp_df[
        "new_insert_timestamp_min"
    ],
    utc=True,
    errors="coerce",
).min()

first_collection_timestamp_max = pd.to_datetime(
    first_collection_timestamp_df[
        "new_insert_timestamp_max"
    ],
    utc=True,
    errors="coerce",
).max()

if (
    pd.isna(first_collection_timestamp_min)
    or pd.isna(first_collection_timestamp_max)
):
    raise ValueError(
        "The Run B first-collection timestamp range "
        "could not be calculated."
    )

first_collection_timestamp_min_iso = (
    first_collection_timestamp_min.isoformat()
)

first_collection_timestamp_max_iso = (
    first_collection_timestamp_max.isoformat()
)

first_collection_fetched = int(
    checkpoint_metadata[
        "records_fetched_total"
    ]
)

first_collection_new_inserts = int(
    checkpoint_metadata[
        "new_records_inserted_total"
    ]
)

first_collection_duplicates = int(
    checkpoint_metadata[
        "duplicates_skipped_total"
    ]
)

first_collection_duplicate_rate = (
    first_collection_duplicates
    / first_collection_fetched
)

first_collection_new_insert_rate = (
    first_collection_new_inserts
    / first_collection_fetched
)

first_collection_gap_hours = float(
    checkpoint_metadata[
        "gap_before_collection_hours"
    ]
)

first_collection_wall_clock_seconds = float(
    checkpoint_metadata[
        "wall_clock_collection_seconds"
    ]
)

first_collection_db_growth_mb = float(
    checkpoint_metadata[
        "database_size_growth_mb"
    ]
)

first_collection_posted_after_prior_boundary = int(
    checkpoint_metadata[
        "posted_between_runs_total"
    ]
)

first_collection_older_surfaced = int(
    checkpoint_metadata[
        "older_reviews_surfaced_later_total"
    ]
)

# -------------------------------------------------------------------
# Add app-level runtime to the Run A and Run B comparison tables.
# -------------------------------------------------------------------
runA_runtime_df = (
    runA_app_summary_df[
        [
            "app_id",
            "runtime_seconds",
        ]
    ]
    .rename(
        columns={
            "runtime_seconds": (
                "runA_app_runtime_seconds"
            )
        }
    )
)

runB_runtime_df = (
    app_summary_df[
        [
            "app_id",
            "runtime_seconds",
        ]
    ]
    .rename(
        columns={
            "runtime_seconds": (
                "runB_app_runtime_seconds"
            )
        }
    )
)

complete_app_report_df = (
    cadence_app_comparison_df.merge(
        runA_runtime_df,
        on="app_id",
        how="left",
        validate="one_to_one",
    )
    .merge(
        runB_runtime_df,
        on="app_id",
        how="left",
        validate="one_to_one",
    )
)

if len(complete_app_report_df) != 10:
    raise ValueError(
        "Expected complete report results for 10 apps."
    )

required_complete_columns = [
    "runA_app_runtime_seconds",
    "runB_app_runtime_seconds",
    "runA_new_timestamp_min",
    "runA_new_timestamp_max",
    "runB_new_timestamp_min",
    "runB_new_timestamp_max",
]

if complete_app_report_df[
    required_complete_columns
].isna().any().any():
    missing_columns = complete_app_report_df.columns[
        complete_app_report_df.isna().any()
    ].tolist()

    raise ValueError(
        "The complete app report contains missing "
        f"values in: {missing_columns}"
    )


def format_timestamp_range(
    minimum_value,
    maximum_value,
):
    return (
        f"{minimum_value} to "
        f"{maximum_value}"
    )


# -------------------------------------------------------------------
# Build the Run B first-collection baseline table.
# -------------------------------------------------------------------
baseline_table_lines = [
    (
        "| Gap from Run A | Reviews fetched | "
        "New DB inserts | Duplicate rate | "
        "Posted after prior run boundary | "
        "Older reviews surfaced | "
        "New-insert timestamp range | "
        "Wall-clock runtime | DB row growth | "
        "DB growth |"
    ),
    (
        "|---:|---:|---:|---:|---:|---:|"
        "---|---:|---:|---:|"
    ),
    (
        f"| {first_collection_gap_hours:.2f} h "
        f"| {first_collection_fetched:,} "
        f"| {first_collection_new_inserts:,} "
        f"| {first_collection_duplicate_rate:.2%} "
        f"| {first_collection_posted_after_prior_boundary:,} "
        f"| {first_collection_older_surfaced:,} "
        f"| {format_timestamp_range(first_collection_timestamp_min_iso, first_collection_timestamp_max_iso)} "
        f"| {first_collection_wall_clock_seconds:.2f} s "
        f"| {first_collection_new_inserts:,} "
        f"| {first_collection_db_growth_mb:.2f} MB |"
    ),
]

# -------------------------------------------------------------------
# Build the complete run-level cadence table.
# -------------------------------------------------------------------
complete_run_table_lines = [
    (
        "| Test | Interval | Reviews fetched | "
        "New DB inserts | Duplicate rate | "
        "Posted between collections | "
        "Older reviews surfaced | "
        "New-insert timestamp range | "
        "Median source lag | "
        "Wall-clock runtime | "
        "DB row growth | DB growth | Errors |"
    ),
    (
        "|---|---:|---:|---:|---:|---:|---:|"
        "---|---:|---:|---:|---:|---:|"
    ),
]

for _, row in cadence_run_comparison_df.iterrows():
    complete_run_table_lines.append(
        (
            f"| {row['cadence_test']} "
            f"| {row['collection_interval_mean_hours']:.2f} h "
            f"| {int(row['records_fetched']):,} "
            f"| {int(row['new_database_inserts']):,} "
            f"| {row['duplicate_rate']:.2%} "
            f"| {int(row['posted_between_collections']):,} "
            f"| {int(row['older_reviews_surfaced_later']):,} "
            f"| {format_timestamp_range(row['new_insert_timestamp_min'], row['new_insert_timestamp_max'])} "
            f"| {row['median_source_lag_hours']:.2f} h "
            f"| {row['wall_clock_runtime_seconds']:.2f} s "
            f"| {int(row['review_row_growth']):,} "
            f"| {row['database_growth_mb']:.2f} MB "
            f"| {int(row['errors'])} |"
        )
    )

# -------------------------------------------------------------------
# Build the complete app-level Run A table.
# -------------------------------------------------------------------
runA_app_table_lines = [
    (
        "| App | New DB inserts | Duplicate rate | "
        "App processing runtime | "
        "New-insert timestamp range | "
        "Posted between collections |"
    ),
    (
        "|---|---:|---:|---:|---|---:|"
    ),
]

for _, row in complete_app_report_df.iterrows():
    runA_app_table_lines.append(
        (
            f"| {row['app_name']} "
            f"| {int(row['runA_new_inserts']):,} "
            f"| {row['runA_duplicate_rate']:.2%} "
            f"| {row['runA_app_runtime_seconds']:.2f} s "
            f"| {format_timestamp_range(row['runA_new_timestamp_min'], row['runA_new_timestamp_max'])} "
            f"| {int(row['runA_posted_between'])} |"
        )
    )

# -------------------------------------------------------------------
# Build the complete app-level Run B table.
# -------------------------------------------------------------------
runB_app_table_lines = [
    (
        "| App | New DB inserts | Duplicate rate | "
        "App processing runtime | "
        "New-insert timestamp range | "
        "Posted between collections |"
    ),
    (
        "|---|---:|---:|---:|---|---:|"
    ),
]

for _, row in complete_app_report_df.iterrows():
    runB_app_table_lines.append(
        (
            f"| {row['app_name']} "
            f"| {int(row['runB_new_inserts']):,} "
            f"| {row['runB_duplicate_rate']:.2%} "
            f"| {row['runB_app_runtime_seconds']:.2f} s "
            f"| {format_timestamp_range(row['runB_new_timestamp_min'], row['runB_new_timestamp_max'])} "
            f"| {int(row['runB_posted_between'])} |"
        )
    )

# -------------------------------------------------------------------
# Build the app-level recommendation table.
# -------------------------------------------------------------------
recommendation_table_lines = [
    (
        "| App | Run A new insert rate | "
        "Run B new insert rate | "
        "Timestamp freshness benefit | "
        "Recommendation | Reason |"
    ),
    (
        "|---|---:|---:|---|---|---|"
    ),
]

for _, row in complete_app_report_df.iterrows():
    recommendation_table_lines.append(
        (
            f"| {row['app_name']} "
            f"| {row['runA_new_insert_rate']:.2%} "
            f"| {row['runB_new_insert_rate']:.2%} "
            f"| {row['timestamp_freshness_benefit']} "
            f"| {row['cadence_recommendation']} "
            f"| {row['recommendation_reason']} |"
        )
    )

total_cadence_new_inserts = int(
    runA_record[
        "new_records_inserted_total"
    ]
) + int(
    runB_record[
        "new_records_inserted_total"
    ]
)

# -------------------------------------------------------------------
# Write the complete final report.
# -------------------------------------------------------------------
complete_final_report_text = f"""# Google Play Phase 2 Cadence Test — Final Run A and Run B Report

## Controlled setup

- Source: Google Play
- Apps: 10
- Target: 1,200 newest reviews per app
- Language and country: English / United States
- Duplicate identity: `source + app_id + review_id`
- Final raw review rows: {final_raw_rows:,}
- Final cleaned review rows: {final_raw_rows:,}
- Completed Phase 2 runs: {len(final_run_history_df)}
- Final database size: {final_database_size_mb:.2f} MB

## Run B first-collection baseline

The Run B first collection established the database baseline for the later follow-up collection.

It occurred {first_collection_gap_hours:.2f} hours after Cadence Run A. Because this gap was much longer than the controlled higher-frequency intervals, the first collection is reported as a baseline and is not treated as a separate twice-daily outcome.

{chr(10).join(baseline_table_lines)}

The first collection returned {first_collection_new_inserts:,} database-new records. Of those, {first_collection_posted_after_prior_boundary:,} had review timestamps after the prior Run A completion boundary and {first_collection_older_surfaced:,} were older reviews that surfaced later.

## Controlled cadence run-level comparison

{chr(10).join(complete_run_table_lines)}

Database growth is reported at the run level. SQLite file-page growth cannot be assigned reliably to individual apps.

## Cadence Run A app-level results

Cadence Run A followed the Day 3 collection after an average interval of {cadence_run_comparison_df.loc[cadence_run_comparison_df['cadence_test'] == 'Cadence Run A', 'collection_interval_mean_hours'].iloc[0]:.2f} hours.

{chr(10).join(runA_app_table_lines)}

## Cadence Run B follow-up app-level results

Cadence Run B followed the Run B first collection after an average interval of {cadence_run_comparison_df.loc[cadence_run_comparison_df['cadence_test'] == 'Cadence Run B', 'collection_interval_mean_hours'].iloc[0]:.2f} hours.

{chr(10).join(runB_app_table_lines)}

## Timestamp and source-freshness finding

Cadence Run A inserted {int(runA_record['new_records_inserted_total']):,} database-new records and Cadence Run B inserted {int(runB_record['new_records_inserted_total']):,}, for a combined total of {total_cadence_new_inserts:,}.

Timestamp validation found zero reviews posted between collections in both controlled cadence tests.

All {total_cadence_new_inserts:,} database-new records had review timestamps at or before the preceding app-specific collection boundary. They were new to the database because they entered the returned review window later, not because they were newly posted between collections.

The median difference between collection time and the newest returned review timestamp was {runA_median_source_lag:.2f} hours in Cadence Run A and {runB_median_source_lag:.2f} hours in Cadence Run B.

This approximately 24-hour returned-window lag was consistent in the two controlled tests, but it should not be interpreted as a universal Google Play rule.

## App-level cadence recommendation

The operational rules used in this experiment were:

- Twice-daily coverage candidate: at least 80% new-to-database returned-window turnover in both cadence tests
- Once-daily candidate: at least 70% duplicate in both cadence tests
- Monitor: results between those two patterns

These are project-specific decision rules for the tested 1,200-review window.

{chr(10).join(recommendation_table_lines)}

## Final recommendation

A blanket twice-daily schedule is not supported for all 10 apps.

- **Twice-daily coverage candidates:** {", ".join(twice_daily_candidates)}
- **Once-daily candidates:** {", ".join(once_daily_candidates)}
- **Monitor before changing cadence:** {", ".join(monitor_candidates)}

YouTube and Instagram are twice-daily candidates only for returned-window coverage. Their fixed 1,200-review windows showed high turnover in both controlled cadence tests, so a longer interval may create a coverage risk.

No posting-time freshness benefit was demonstrated for YouTube, Instagram, or any other tested app.

Uber, DoorDash, Duolingo, Google Maps, Netflix, and Reddit were duplicate-heavy in both cadence tests and can remain on once-daily collection under the current setup.

TikTok and Spotify showed moderate returned-window turnover and should remain under observation before an app-specific cadence exception is introduced.

## Measurement notes

- Run-level runtime is wall-clock collection time, including the fixed delay between app requests.
- App-level runtime is the processing time recorded for each app request.
- New database inserts indicate review identities not previously stored in the database.
- New database inserts are not automatically treated as newly posted reviews.
- Review-posting freshness is determined from app-specific review and collection timestamps.
- Database size growth is reported only at run level.
- The conclusions apply to the tested 10 apps, 1,200-review returned window, and recorded controlled runs.
"""

final_report_path = (
    FINAL_EXPORT_DIR
    / "phase2_cadence_runA_runB_final_report.md"
)

final_report_path.write_text(
    complete_final_report_text,
    encoding="utf-8",
)

# -------------------------------------------------------------------
# Update package metadata to document the completed report.
# -------------------------------------------------------------------
metadata_path = (
    FINAL_EXPORT_DIR
    / "phase2_cadence_runB_followup_metadata.json"
)

if not metadata_path.exists():
    raise FileNotFoundError(
        "The final package metadata file was not found."
    )

with open(
    metadata_path,
    "r",
    encoding="utf-8",
) as file:
    updated_metadata = json.load(file)

report_completed_at = datetime.now(
    timezone.utc
).isoformat()

updated_metadata.update(
    {
        "final_report_version": (
            "complete_v2"
        ),
        "final_report_completed_at": (
            report_completed_at
        ),
        "final_report_includes": [
            "Run B first-collection baseline",
            "run-level timestamp ranges",
            "app-level new insert counts",
            "app-level duplicate rates",
            "app-level processing runtime",
            "app-level timestamp ranges",
            "timestamp-verified posted-between counts",
            "run-level database growth",
            "app-level cadence recommendations",
        ],
    }
)

with open(
    metadata_path,
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        updated_metadata,
        file,
        indent=2,
        ensure_ascii=False,
    )

# -------------------------------------------------------------------
# Validate that the completed report contains every required section.
# -------------------------------------------------------------------
report_requirements = {
    "contains_controlled_setup": (
        "## Controlled setup"
        in complete_final_report_text
    ),
    "contains_first_collection_baseline": (
        "## Run B first-collection baseline"
        in complete_final_report_text
    ),
    "contains_run_level_comparison": (
        "## Controlled cadence run-level comparison"
        in complete_final_report_text
    ),
    "contains_runA_app_results": (
        "## Cadence Run A app-level results"
        in complete_final_report_text
    ),
    "contains_runB_app_results": (
        "## Cadence Run B follow-up app-level results"
        in complete_final_report_text
    ),
    "contains_app_runtime": (
        "App processing runtime"
        in complete_final_report_text
    ),
    "contains_timestamp_ranges": (
        "New-insert timestamp range"
        in complete_final_report_text
    ),
    "contains_database_growth_note": (
        "Database growth is reported at the run level"
        in complete_final_report_text
    ),
    "contains_first_collection_8638": (
        "8,638"
        in complete_final_report_text
    ),
    "contains_runA_4395": (
        "4,395"
        in complete_final_report_text
    ),
    "contains_runB_3753": (
        "3,753"
        in complete_final_report_text
    ),
    "contains_zero_posted_finding": (
        "zero reviews posted between collections"
        in complete_final_report_text
    ),
    "contains_all_10_apps": all(
        app_name in complete_final_report_text
        for app_name in app_config_df[
            "app_name"
        ].tolist()
    ),
    "contains_final_recommendation": (
        "## Final recommendation"
        in complete_final_report_text
    ),
    "contains_measurement_notes": (
        "## Measurement notes"
        in complete_final_report_text
    ),
}

failed_report_requirements = [
    check_name
    for check_name, passed
    in report_requirements.items()
    if not passed
]

if failed_report_requirements:
    raise ValueError(
        "Completed report validation failed: "
        f"{failed_report_requirements}"
    )

report_validation_df = pd.DataFrame(
    [
        {
            "validation_check": check_name,
            "passed": passed,
        }
        for check_name, passed
        in report_requirements.items()
    ]
)

# Save the report validation as an additional package file.
report_validation_path = (
    FINAL_EXPORT_DIR
    / (
        "phase2_cadence_runA_runB_"
        "final_report_validation.csv"
    )
)

report_validation_df.to_csv(
    report_validation_path,
    index=False,
)

# -------------------------------------------------------------------
# Rebuild the manifest after the report and metadata updates.
# -------------------------------------------------------------------
manifest_path = (
    FINAL_EXPORT_DIR
    / "final_package_manifest.csv"
)

if manifest_path.exists():
    manifest_path.unlink()

manifest_rows = []

for file_path in sorted(
    FINAL_EXPORT_DIR.iterdir()
):
    if (
        file_path.is_file()
        and file_path.name
        != "final_package_manifest.csv"
    ):
        manifest_rows.append(
            {
                "file_name": file_path.name,
                "size_bytes": (
                    file_path.stat().st_size
                ),
                "sha256": calculate_sha256(
                    file_path
                ),
            }
        )

updated_manifest_df = pd.DataFrame(
    manifest_rows
)

updated_manifest_df.to_csv(
    manifest_path,
    index=False,
)

# Validate every manifest record.
manifest_failures = []

for _, row in updated_manifest_df.iterrows():
    file_path = (
        FINAL_EXPORT_DIR
        / str(row["file_name"])
    )

    if not file_path.exists():
        manifest_failures.append(
            f"{file_path.name}: missing"
        )
        continue

    actual_size = int(
        file_path.stat().st_size
    )

    actual_hash = calculate_sha256(
        file_path
    )

    if actual_size != int(
        row["size_bytes"]
    ):
        manifest_failures.append(
            f"{file_path.name}: size mismatch"
        )

    if actual_hash != str(
        row["sha256"]
    ):
        manifest_failures.append(
            f"{file_path.name}: SHA-256 mismatch"
        )

if manifest_failures:
    raise ValueError(
        "Updated manifest validation failed: "
        f"{manifest_failures}"
    )

# -------------------------------------------------------------------
# Rebuild and validate the final GitHub ZIP.
# -------------------------------------------------------------------
rebuild_timestamp = datetime.now(
    timezone.utc
).strftime("%Y%m%d_%H%M%S_utc")

updated_package_base_path = (
    BASE_DIR
    / (
        "phase2_cadence_runB_followup_"
        "github_upload_files_complete_report_"
        f"{rebuild_timestamp}"
    )
)

updated_package_zip_path = Path(
    shutil.make_archive(
        str(updated_package_base_path),
        "zip",
        root_dir=FINAL_EXPORT_DIR,
    )
)

if not updated_package_zip_path.exists():
    raise FileNotFoundError(
        "The updated final ZIP was not created."
    )

expected_package_files = {
    path.name
    for path in FINAL_EXPORT_DIR.iterdir()
    if path.is_file()
}

with zipfile.ZipFile(
    updated_package_zip_path,
    "r",
) as zf:
    actual_package_files = {
        Path(name).name
        for name in zf.namelist()
        if not name.endswith("/")
    }

if actual_package_files != expected_package_files:
    missing_in_zip = sorted(
        expected_package_files
        - actual_package_files
    )

    unexpected_in_zip = sorted(
        actual_package_files
        - expected_package_files
    )

    raise ValueError(
        "Updated ZIP contents do not match the "
        "final export folder. "
        f"Missing: {missing_in_zip}; "
        f"Unexpected: {unexpected_in_zip}"
    )

FINAL_REPORT_REBUILD_COMPLETED = True

print(
    "Completed final report and updated GitHub "
    "package created successfully."
)
print("-" * 82)
print(
    "Updated report:",
    final_report_path.name,
)
print(
    "Updated report size:",
    f"{final_report_path.stat().st_size / 1024:.2f} KB",
)
print(
    "Report validation checks:",
    len(report_validation_df),
)
print(
    "Report validation failures:",
    len(failed_report_requirements),
)
print(
    "Manifest records:",
    len(updated_manifest_df),
)
print(
    "Files in updated ZIP:",
    len(actual_package_files),
)
print(
    "Updated package:",
    updated_package_zip_path.name,
)
print(
    "Updated package size:",
    (
        f"{updated_package_zip_path.stat().st_size / (1024 ** 2):.2f} MB"
    ),
)

print("\nCompleted report validation:")
display(report_validation_df)

print("\nUpdated package files:")
display(
    pd.DataFrame(
        [
            {
                "file_name": path.name,
                "size_mb": (
                    path.stat().st_size
                    / (1024 ** 2)
                ),
            }
            for path in sorted(
                FINAL_EXPORT_DIR.iterdir()
            )
            if path.is_file()
        ]
    )
)

print("\nCompleted final report preview:")
display(
    Markdown(
        complete_final_report_text
    )
)

files.download(
    str(updated_package_zip_path)
)

Completed final report and updated GitHub package created successfully.
----------------------------------------------------------------------------------
Updated report: phase2_cadence_runA_runB_final_report.md
Updated report size: 9.43 KB
Report validation checks: 15
Report validation failures: 0
Manifest records: 22
Files in updated ZIP: 23
Updated package: phase2_cadence_runB_followup_github_upload_files_complete_report_20260714_170311_utc.zip
Updated package size: 21.48 MB

Completed report validation:


,validation_check,passed
0,contains_controlled_setup,True
1,contains_first_collection_baseline,True
2,contains_run_level_comparison,True
3,contains_runA_app_results,True
4,contains_runB_app_results,True
5,contains_app_runtime,True
6,contains_timestamp_ranges,True
7,contains_database_growth_note,True
8,contains_first_collection_8638,True
9,contains_runA_4395,True



Updated package files:


,file_name,size_mb
0,final_package_manifest.csv,0.002561
1,google_play_reviews_after_runB_followup.sqlite...,20.859097
2,phase2_cadence_runA_runB_app_level_recommendat...,0.004760
3,phase2_cadence_runA_runB_comparison_validation...,0.000411
4,phase2_cadence_runA_runB_final_report.md,0.009212
5,phase2_cadence_runA_runB_final_report_validati...,0.000472
6,phase2_cadence_runA_runB_run_level_comparison.csv,0.001002
7,phase2_cadence_runA_timestamp_audit_reconstruc...,1.823096
8,phase2_cadence_runA_timestamp_summary_reconstr...,0.002276
9,phase2_cadence_runA_timestamp_validation.csv,0.000253



Completed final report preview:


# Google Play Phase 2 Cadence Test — Final Run A and Run B Report

## Controlled setup

- Source: Google Play
- Apps: 10
- Target: 1,200 newest reviews per app
- Language and country: English / United States
- Duplicate identity: `source + app_id + review_id`
- Final raw review rows: 34,601
- Final cleaned review rows: 34,601
- Completed Phase 2 runs: 6
- Final database size: 84.68 MB

## Run B first-collection baseline

The Run B first collection established the database baseline for the later follow-up collection.

It occurred 100.65 hours after Cadence Run A. Because this gap was much longer than the controlled higher-frequency intervals, the first collection is reported as a baseline and is not treated as a separate twice-daily outcome.

| Gap from Run A | Reviews fetched | New DB inserts | Duplicate rate | Posted after prior run boundary | Older reviews surfaced | New-insert timestamp range | Wall-clock runtime | DB row growth | DB growth |
|---:|---:|---:|---:|---:|---:|---|---:|---:|---:|
| 100.65 h | 12,000 | 8,638 | 28.02% | 8,015 | 623 | 2026-07-08T21:16:36+00:00 to 2026-07-13T01:56:37+00:00 | 30.76 s | 8,638 | 18.56 MB |

The first collection returned 8,638 database-new records. Of those, 8,015 had review timestamps after the prior Run A completion boundary and 623 were older reviews that surfaced later.

## Controlled cadence run-level comparison

| Test | Interval | Reviews fetched | New DB inserts | Duplicate rate | Posted between collections | Older reviews surfaced | New-insert timestamp range | Median source lag | Wall-clock runtime | DB row growth | DB growth | Errors |
|---|---:|---:|---:|---:|---:|---:|---|---:|---:|---:|---:|---:|
| Cadence Run A | 17.16 h | 12,000 | 4,395 | 63.38% | 0 | 4,395 | 2026-07-04T15:23:42+00:00 to 2026-07-08T21:14:19+00:00 | 24.04 h | 28.14 s | 4,395 | 11.76 MB | 0 |
| Cadence Run B | 14.66 h | 12,000 | 3,753 | 68.73% | 0 | 3,753 | 2026-07-13T01:56:51+00:00 to 2026-07-13T16:36:13+00:00 | 24.04 h | 30.02 s | 3,753 | 10.59 MB | 0 |

Database growth is reported at the run level. SQLite file-page growth cannot be assigned reliably to individual apps.

## Cadence Run A app-level results

Cadence Run A followed the Day 3 collection after an average interval of 17.16 hours.

| App | New DB inserts | Duplicate rate | App processing runtime | New-insert timestamp range | Posted between collections |
|---|---:|---:|---:|---|---:|
| YouTube | 1,199 | 0.08% | 1.22 s | 2026-07-08T10:17:11+00:00 to 2026-07-08T21:13:22+00:00 | 0 |
| TikTok | 622 | 48.17% | 0.68 s | 2026-07-08T04:08:06+00:00 to 2026-07-08T21:14:15+00:00 | 0 |
| Spotify | 580 | 51.67% | 0.74 s | 2026-07-08T04:06:13+00:00 to 2026-07-08T21:14:19+00:00 | 0 |
| Instagram | 1,198 | 0.17% | 1.02 s | 2026-07-08T11:26:23+00:00 to 2026-07-08T21:12:32+00:00 | 0 |
| Uber | 317 | 73.58% | 0.98 s | 2026-07-08T04:09:18+00:00 to 2026-07-08T21:12:10+00:00 | 0 |
| DoorDash | 73 | 93.92% | 0.57 s | 2026-07-08T04:54:25+00:00 to 2026-07-08T21:03:08+00:00 | 0 |
| Duolingo | 6 | 99.50% | 0.78 s | 2026-07-08T04:17:19+00:00 to 2026-07-08T15:57:17+00:00 | 0 |
| Google Maps | 184 | 84.67% | 0.65 s | 2026-07-04T15:23:42+00:00 to 2026-07-08T21:14:19+00:00 | 0 |
| Netflix | 123 | 89.75% | 0.72 s | 2026-07-08T04:08:00+00:00 to 2026-07-08T21:01:32+00:00 | 0 |
| Reddit | 93 | 92.25% | 0.68 s | 2026-07-08T04:15:59+00:00 to 2026-07-08T20:43:50+00:00 | 0 |

## Cadence Run B follow-up app-level results

Cadence Run B followed the Run B first collection after an average interval of 14.66 hours.

| App | New DB inserts | Duplicate rate | App processing runtime | New-insert timestamp range | Posted between collections |
|---|---:|---:|---:|---|---:|
| YouTube | 994 | 17.17% | 1.93 s | 2026-07-13T01:59:02+00:00 to 2026-07-13T16:35:07+00:00 | 0 |
| TikTok | 406 | 66.17% | 0.79 s | 2026-07-13T01:59:51+00:00 to 2026-07-13T16:34:23+00:00 | 0 |
| Spotify | 456 | 62.00% | 0.77 s | 2026-07-13T01:56:51+00:00 to 2026-07-13T16:35:05+00:00 | 0 |
| Instagram | 1,196 | 0.33% | 0.85 s | 2026-07-13T08:02:32+00:00 to 2026-07-13T16:36:13+00:00 | 0 |
| Uber | 309 | 74.25% | 0.78 s | 2026-07-13T02:01:50+00:00 to 2026-07-13T16:34:04+00:00 | 0 |
| DoorDash | 25 | 97.92% | 0.93 s | 2026-07-13T01:58:41+00:00 to 2026-07-13T16:32:29+00:00 | 0 |
| Duolingo | 3 | 99.75% | 0.97 s | 2026-07-13T08:45:54+00:00 to 2026-07-13T16:07:35+00:00 | 0 |
| Google Maps | 151 | 87.42% | 0.84 s | 2026-07-13T02:06:35+00:00 to 2026-07-13T16:34:26+00:00 | 0 |
| Netflix | 152 | 87.33% | 0.86 s | 2026-07-13T01:58:20+00:00 to 2026-07-13T16:31:09+00:00 | 0 |
| Reddit | 61 | 94.92% | 0.74 s | 2026-07-13T02:03:14+00:00 to 2026-07-13T16:32:15+00:00 | 0 |

## Timestamp and source-freshness finding

Cadence Run A inserted 4,395 database-new records and Cadence Run B inserted 3,753, for a combined total of 8,148.

Timestamp validation found zero reviews posted between collections in both controlled cadence tests.

All 8,148 database-new records had review timestamps at or before the preceding app-specific collection boundary. They were new to the database because they entered the returned review window later, not because they were newly posted between collections.

The median difference between collection time and the newest returned review timestamp was 24.04 hours in Cadence Run A and 24.04 hours in Cadence Run B.

This approximately 24-hour returned-window lag was consistent in the two controlled tests, but it should not be interpreted as a universal Google Play rule.

## App-level cadence recommendation

The operational rules used in this experiment were:

- Twice-daily coverage candidate: at least 80% new-to-database returned-window turnover in both cadence tests
- Once-daily candidate: at least 70% duplicate in both cadence tests
- Monitor: results between those two patterns

These are project-specific decision rules for the tested 1,200-review window.

| App | Run A new insert rate | Run B new insert rate | Timestamp freshness benefit | Recommendation | Reason |
|---|---:|---:|---|---|---|
| YouTube | 99.92% | 82.83% | Not demonstrated | Twice-daily candidate for returned-window coverage | High returned-window turnover in both tests. The fixed 1,200-review batch was close to saturation, creating a coverage risk under a longer interval. No posting-time freshness benefit was observed. |
| TikTok | 51.83% | 33.83% | Not demonstrated | Monitor before changing cadence | Moderate returned-window turnover, but no timestamp-verified freshness benefit. More evidence is needed before creating an app-specific twice-daily exception. |
| Spotify | 48.33% | 38.00% | Not demonstrated | Monitor before changing cadence | Moderate returned-window turnover, but no timestamp-verified freshness benefit. More evidence is needed before creating an app-specific twice-daily exception. |
| Instagram | 99.83% | 99.67% | Not demonstrated | Twice-daily candidate for returned-window coverage | High returned-window turnover in both tests. The fixed 1,200-review batch was close to saturation, creating a coverage risk under a longer interval. No posting-time freshness benefit was observed. |
| Uber | 26.42% | 25.75% | Not demonstrated | Once-daily candidate | Duplicate-heavy in both tests, with no timestamp-verified reviews posted between collections. |
| DoorDash | 6.08% | 2.08% | Not demonstrated | Once-daily candidate | Duplicate-heavy in both tests, with no timestamp-verified reviews posted between collections. |
| Duolingo | 0.50% | 0.25% | Not demonstrated | Once-daily candidate | Duplicate-heavy in both tests, with no timestamp-verified reviews posted between collections. |
| Google Maps | 15.33% | 12.58% | Not demonstrated | Once-daily candidate | Duplicate-heavy in both tests, with no timestamp-verified reviews posted between collections. |
| Netflix | 10.25% | 12.67% | Not demonstrated | Once-daily candidate | Duplicate-heavy in both tests, with no timestamp-verified reviews posted between collections. |
| Reddit | 7.75% | 5.08% | Not demonstrated | Once-daily candidate | Duplicate-heavy in both tests, with no timestamp-verified reviews posted between collections. |

## Final recommendation

A blanket twice-daily schedule is not supported for all 10 apps.

- **Twice-daily coverage candidates:** YouTube, Instagram
- **Once-daily candidates:** Uber, DoorDash, Duolingo, Google Maps, Netflix, Reddit
- **Monitor before changing cadence:** TikTok, Spotify

YouTube and Instagram are twice-daily candidates only for returned-window coverage. Their fixed 1,200-review windows showed high turnover in both controlled cadence tests, so a longer interval may create a coverage risk.

No posting-time freshness benefit was demonstrated for YouTube, Instagram, or any other tested app.

Uber, DoorDash, Duolingo, Google Maps, Netflix, and Reddit were duplicate-heavy in both cadence tests and can remain on once-daily collection under the current setup.

TikTok and Spotify showed moderate returned-window turnover and should remain under observation before an app-specific cadence exception is introduced.

## Measurement notes

- Run-level runtime is wall-clock collection time, including the fixed delay between app requests.
- App-level runtime is the processing time recorded for each app request.
- New database inserts indicate review identities not previously stored in the database.
- New database inserts are not automatically treated as newly posted reviews.
- Review-posting freshness is determined from app-specific review and collection timestamps.
- Database size growth is reported only at run level.
- The conclusions apply to the tested 10 apps, 1,200-review returned window, and recorded controlled runs.


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>